# 01. Cloudless DET Feedback GA Search Tutorial

이 노트북은 기존 01의 cell 기능/process를 유지하되, 실제 local 모델 실행 경로를 `00_local_qwen_worker_preflight.ipynb`에서 검증된 canonical model-suite 경로로 교체한 버전입니다.

최종 실행 경로:

```text
utils/ga_search/model_suite_benchmark.py
utils/ga_search/local_model_registry.py
utils/ga_search/local_worker.py
utils/ga_search/candidate_generation.py
utils/ga_search/evaluation.py
```

유지하는 process:

1. canonical import / compile / render smoke
2. single-row strict DET evaluation
3. single-row retry / diagnostic
4. optional worker real row smoke
5. category-level benchmark
6. manual prompt patch application and visibility check
7. prompt logs / raw responses / service context inspection

중요:

- 기본 평가 경로에서 `mock`을 사용하지 않습니다.
- `mock`은 GT를 Generated로 복사할 수 있으므로, 이 노트북의 모델 성능 확인에는 사용하지 않습니다.
- 먼저 `qwen25_coder_7b`로 row-level smoke를 확인하고, 이후 14B를 실행합니다.
- `max_new_tokens=64`는 JSON truncation으로 `invalid_json`이 날 수 있으므로 7B 기본값은 `512`, 14B 기본값은 `1024`입니다.

추가 overnight process:

8. full local DET smoke run
9. full local DET all-row run without category limit
10. monitor detached full run
11. aggregate Excel-ready CSV files for next-day advisor/prompt editing
12. rerun failed rows after prompt modification

이 notebook의 `full local DET`는 local Qwen이 생성한 JOICode를 strict DET로 평가하고, 다음 날 Excel에서 실패 row를 검토해 advisor prompt와 JOICode generation prompt를 수정하기 위한 입력 artifact를 만드는 단계입니다.


추가 후처리 process:

15. strict DET `row_evaluation.csv` 로드
16. failure taxonomy/advisor routing table 정의
17. row-level diagnostic mapping 함수 정의
18. row advisor feedback record 생성
19. `advisor_rich_feedback.json`저장
20. cluster별 localized prompt patch 확장/구현/검증


In [11]:
# ============================================================
# Setup 1: imports and server preset
# ============================================================

import os
import sys
import json
import shlex
import time
import html
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)
pd.set_option("display.max_colwidth", 300)

SERVER_PRESET = os.environ.get("SERVER_PRESET", "a6000")  # "a6000" or "a100"

if SERVER_PRESET == "a6000":
    os.environ.setdefault("JOILANG_BASE_DIR", "/home/mgjeong/Desktop/llm/JOILang-Server")
    os.environ.setdefault("JOI_PY", "/home/mgjeong/miniconda3/envs/joi/bin/python")
    os.environ.setdefault("LOCAL_MODEL_BASE", "/home/mgjeong/Desktop/llm/local_models")
elif SERVER_PRESET == "a100":
    os.environ.setdefault("JOILANG_BASE_DIR", "/root/llm/JOILang-Server")
    # A100/root server keeps the existing root Python unless explicitly overridden.
    os.environ.setdefault("JOI_PY", "/root/miniconda3/bin/python")
    os.environ.setdefault("LOCAL_MODEL_BASE", "/root/llm/local_models")
else:
    raise ValueError(f"Unknown SERVER_PRESET={SERVER_PRESET!r}")

In [12]:
# ============================================================
# Setup 2: paths, canonical helpers, and viewers
# ============================================================

BASE_DIR = Path(os.environ["JOILANG_BASE_DIR"]).expanduser().resolve()
JOI_PY = Path(os.environ["JOI_PY"]).expanduser().resolve()
LOCAL_MODEL_BASE = Path(os.environ["LOCAL_MODEL_BASE"]).expanduser().resolve()

assert BASE_DIR.exists(), f"BASE_DIR not found: {BASE_DIR}"
assert JOI_PY.exists(), f"JOI_PY not found: {JOI_PY}"

GA_CLI = BASE_DIR / "utils" / "ga_search" / "cli.py"
DATASET = BASE_DIR / "datasets" / "JOICommands-280.csv"
SERVICE_SCHEMA = BASE_DIR / "datasets" / "service_list_ver2.0.1.json"

MODEL = os.environ.get("JOI_GA_MODEL", "gpt_mg.version0_13")
ROW_NO = int(os.environ.get("ROW_NO", "251"))

MODEL_KEY_7B = os.environ.get("MODEL_KEY_7B", "qwen25_coder_7b")
MODEL_KEY_14B = os.environ.get("MODEL_KEY_14B", "qwen25_coder_14b")

CUDA_VISIBLE_DEVICES = os.environ.get("CUDA_VISIBLE_DEVICES_FOR_NOTEBOOK", "0")
LOCAL_DEVICE = os.environ.get("LOCAL_DEVICE", "cuda:0")

MAX_NEW_TOKENS_7B = int(os.environ.get("MAX_NEW_TOKENS_7B", "512"))
MAX_NEW_TOKENS_14B = int(os.environ.get("MAX_NEW_TOKENS_14B", "1024"))

CATEGORY_TO_RUN = int(os.environ.get("CATEGORY_TO_RUN", "5"))
LIMIT_PER_CATEGORY_RAW = os.environ.get("LIMIT_PER_CATEGORY", "all").strip().lower()
if LIMIT_PER_CATEGORY_RAW in {"", "all", "none", "null", "0", "-1"}:
    LIMIT_PER_CATEGORY = None
else:
    LIMIT_PER_CATEGORY = int(LIMIT_PER_CATEGORY_RAW)

RUN_TAG = os.environ.get("RUN_TAG", datetime.now().strftime("%Y%m%d_%H%M%S"))
NB_ROOT = BASE_DIR / "artifacts" / "ga_search_tutorial_runs" / f"cloudless_model_suite_{RUN_TAG}"
NB_ROOT.mkdir(parents=True, exist_ok=True)

print("SERVER_PRESET:", SERVER_PRESET)
print("BASE_DIR:", BASE_DIR)
print("JOI_PY:", JOI_PY)
print("LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)
print("NB_ROOT:", NB_ROOT)
print("MODEL:", MODEL)
print("ROW_NO:", ROW_NO)
print("MODEL_KEY_7B:", MODEL_KEY_7B)
print("MODEL_KEY_14B:", MODEL_KEY_14B)
print("CUDA_VISIBLE_DEVICES:", CUDA_VISIBLE_DEVICES)
print("LOCAL_DEVICE:", LOCAL_DEVICE)

# Runtime state placeholders.
# These prevent NameError when later inspection/export cells are run independently or after partial execution.
qwen7b_root = None
qwen7b_long_root = None
qwen14b_root = None
category_7b_root = None
category_14b_root = None
patch_apply_dir = None
patched_7b_root = None
full_7b_root = None
full_14b_root = None
category_7b_records = []
category_14b_records = []

def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def out_dir(label):
    p = NB_ROOT / str(label)
    p.mkdir(parents=True, exist_ok=True)
    return p

def run_cmd(cmd, log_path=None, check=False, timeout_sec=None, env_extra=None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["CUDA_VISIBLE_DEVICES"] = str(CUDA_VISIBLE_DEVICES)
    # Avoid stale /usr/local/cuda-* LD paths interfering with PyTorch wheel CUDA runtime.
    env["LD_LIBRARY_PATH"] = ""
    if env_extra:
        env.update({str(k): str(v) for k, v in env_extra.items()})

    cmd = [str(x) for x in cmd]
    print("\n[CMD]")
    print(" ".join(shlex.quote(x) for x in cmd))
    if log_path:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("[LOG]", log_path)

    started = time.time()
    proc = subprocess.run(
        cmd,
        cwd=str(BASE_DIR),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout_sec,
    )
    elapsed = time.time() - started
    output = proc.stdout or ""

    if log_path:
        log_path.write_text(output, encoding="utf-8", errors="replace")

    print(output[-8000:])
    print(f"[RC] {proc.returncode}  [elapsed] {elapsed:.2f}s")

    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed rc={proc.returncode}: {' '.join(cmd)}")

    return proc.returncode, output

def load_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

def read_csv_or_empty(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    return pd.read_csv(path)

def show_file(path, max_chars=5000):
    path = Path(path)
    print(path, "exists=", path.exists(), "size=", path.stat().st_size if path.exists() else 0)
    if path.exists():
        text = path.read_text(encoding="utf-8", errors="replace")
        print(text[:max_chars])
        if len(text) > max_chars:
            print(f"\n... truncated {len(text) - max_chars} chars")

def suite_model_dir(root, model_key):
    return Path(root) / str(model_key)

def suite_summary(root):
    p = Path(root) / "suite_summary.json"
    return load_json(p) if p.exists() else {}

def model_summary(root, model_key):
    p = suite_model_dir(root, model_key) / "model_summary.json"
    return load_json(p) if p.exists() else {}

def candidates_df(root, model_key=None):
    root = Path(root)
    if model_key:
        p = suite_model_dir(root, model_key) / "candidates" / "generation_000.csv"
    else:
        p = root / "candidates" / "generation_000.csv"
        if not p.exists():
            matches = list(root.glob("*/candidates/generation_000.csv"))
            p = matches[0] if matches else p
    return read_csv_or_empty(p)

def eval_df(root, model_key=None):
    root = Path(root)
    if model_key:
        p = suite_model_dir(root, model_key) / "eval" / "row_evaluation.csv"
    else:
        p = root / "eval" / "row_evaluation.csv"
        if not p.exists():
            matches = list(root.glob("*/eval/row_evaluation.csv"))
            p = matches[0] if matches else p
    return read_csv_or_empty(p)

def cli_base(command):
    return [str(JOI_PY), "-m", "utils.ga_search.cli", command]

def run_render(label="render_smoke", user_input="Turn on the light.", dry_run=True):
    od = out_dir(label)
    cmd = cli_base("render") + [
        "--model", MODEL,
        "--user-input", user_input,
        "--search-mode", "auto",
    ]
    if dry_run:
        cmd.append("--dry-run")
    rc, output = run_cmd(cmd, log_path=od / f"{label}.log", check=True, timeout_sec=300)
    return od, rc

def model_suite_cmd(output_dir, model_key, row_no=None, max_new_tokens=512, timeout_sec=900, extra_args=None):
    cmd = [
        str(JOI_PY), "-m", "utils.ga_search.model_suite_benchmark",
        "--model", MODEL,
        "--model-key", str(model_key),
        "--llm-mode", "worker",
        "--local-model-base-dir", str(LOCAL_MODEL_BASE),
        "--worker-python", str(JOI_PY),
        "--local-device", str(LOCAL_DEVICE),
        "--local-files-only", "true",
        "--local-trust-remote-code", "true",
        "--local-max-new-tokens", str(max_new_tokens),
        "--timeout-sec", str(timeout_sec),
        "--output-dir", str(output_dir),
    ]
    if row_no is not None:
        cmd += ["--row-no", str(row_no)]
    if extra_args:
        cmd += list(map(str, extra_args))
    return cmd

def run_model_suite_row(label, model_key, row_no, max_new_tokens, timeout_sec=1200, extra_args=None, check=False):
    root = out_dir(label)
    cmd = model_suite_cmd(
        root,
        model_key=model_key,
        row_no=row_no,
        max_new_tokens=max_new_tokens,
        timeout_sec=timeout_sec,
        extra_args=extra_args,
    )
    rc, output = run_cmd(cmd, log_path=root / "run.log", check=check, timeout_sec=timeout_sec + 300)
    return root, rc

def inspect_run(root, model_key, max_rows=20):
    root = Path(root)
    print("\nRUN_ROOT:", root)
    print("suite_summary:")
    print(json.dumps(suite_summary(root), ensure_ascii=False, indent=2)[:4000])
    print("\nmodel_summary:")
    print(json.dumps(model_summary(root, model_key), ensure_ascii=False, indent=2)[:4000])

    cand = candidates_df(root, model_key)
    ev = eval_df(root, model_key)

    print("\nCANDIDATES:")
    display(cand.head(max_rows))
    print("\nEVALUATION:")
    display(ev.head(max_rows))

    if not cand.empty:
        first = cand.iloc[0]
        for col in ["generation_error_type", "candidate_error", "prompt_log_paths", "raw_response_path", "service_context_source", "service_list_retrieval_scores", "repair_actions"]:
            if col in cand.columns:
                print("\n" + "=" * 100)
                print(col)
                print(first.get(col, ""))

        raw_path = first.get("raw_response_path", "")
        if isinstance(raw_path, str) and raw_path.strip():
            print("\n" + "=" * 100)
            print("RAW RESPONSE")
            show_file(raw_path, max_chars=6000)
    return cand, ev

def row_summary(root, model_key=None, max_rows=120):
    ev = eval_df(root, model_key)
    if ev.empty:
        print("No eval rows")
        return ev
    cols = [c for c in [
        "row_no", "category", "genome_id", "candidate_index", "det_score", "det_pass", "gt_exact",
        "gt_similarity", "schedule_match", "service_recall", "service_precision", "receiver_recall",
        "numeric_grounding", "enum_grounding", "dataflow_score", "failure_reasons", "generated_code", "gt_code"
    ] if c in ev.columns]
    display(ev[cols].head(max_rows))
    return ev

def gt_pretty(raw):
    if raw is None:
        return ""
    if not isinstance(raw, str):
        raw = json.dumps(raw, ensure_ascii=False)
    raw = raw.strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict):
            script = obj.get("script", obj.get("code", ""))
            meta = dict(obj)
            meta.pop("script", None)
            meta.pop("code", None)
            return "[JSON meta]\n" + json.dumps(meta, ensure_ascii=False, indent=2) + "\n\n[script/code]\n" + str(script).replace("\\n", "\n")
        return json.dumps(obj, ensure_ascii=False, indent=2)
    except Exception:
        return raw.replace("\\n", "\n").replace("\\t", "    ")

def _pre(title, text):
    return f"""
    <div>
      <div style="font-weight:700;background:#f7f7f7;padding:6px;border:1px solid #ddd;border-bottom:none;">{html.escape(str(title))}</div>
      <pre style="margin:0;white-space:pre;overflow:auto;max-height:520px;min-height:160px;tab-size:4;font-family:Consolas,'Courier New',monospace;font-size:13px;line-height:1.45;background:#fbfbfb;padding:10px;border:1px solid #ddd;">{html.escape(gt_pretty(text))}</pre>
    </div>
    """

def show_gt_vs_generated(root, row_no=None, model_key=None, max_rows=20):
    cand = candidates_df(root, model_key)
    ev = eval_df(root, model_key)
    if cand.empty:
        print("No candidates found:", root)
        return pd.DataFrame()

    if row_no is not None and "row_no" in cand.columns:
        cand = cand[cand["row_no"].astype(str) == str(row_no)]

    if not ev.empty:
        join_cols = [c for c in ["row_no", "genome_id", "candidate_index"] if c in cand.columns and c in ev.columns]
        merged = cand.merge(ev, on=join_cols, how="left", suffixes=("", "_eval")) if join_cols else cand
    else:
        merged = cand

    cards = []
    for _, r in merged.head(max_rows).iterrows():
        gt = r.get("gt", r.get("gt_json", ""))
        generated = r.get("generated_json", "") or r.get("candidates", "") or r.get("generated_code", "")
        title = f"row={r.get('row_no')} cat={r.get('category')} genome={r.get('genome_id')} det={r.get('det_score', '')} pass={r.get('det_pass', '')}"
        cards.append(f"""
        <div style="border:1px solid #ccc;border-radius:8px;padding:12px;margin:12px 0;">
          <div style="font-weight:700;margin-bottom:8px;">{html.escape(str(title))}</div>
          <div style="display:grid;grid-template-columns:minmax(0,1fr) minmax(0,1fr);gap:12px;">
            {_pre("GT", gt)}
            {_pre("Generated", generated)}
          </div>
          <div style="margin-top:8px;font-size:13px;"><b>failure_reasons:</b> {html.escape(str(r.get('failure_reasons', '')))}</div>
          <div style="margin-top:4px;font-size:13px;"><b>prompt_log_paths:</b> {html.escape(str(r.get('prompt_log_paths', '')))}</div>
          <div style="margin-top:4px;font-size:13px;"><b>raw_response_path:</b> {html.escape(str(r.get('raw_response_path', '')))}</div>
        </div>
        """)
    display(HTML("\n".join(cards)))
    return merged

def select_rows_by_category(category, limit_per_category=5):
    ds = pd.read_csv(DATASET)
    cat_col = "category" if "category" in ds.columns else "cat"
    row_col = "row_no" if "row_no" in ds.columns else None
    hit = ds[ds[cat_col].astype(str) == str(category)].copy()
    if row_col:
        rows = hit[row_col].dropna().astype(int).tolist()
    else:
        rows = [int(i) + 1 for i in hit.index.tolist()]
    if limit_per_category is None:
        return rows

    limit_per_category = int(limit_per_category)
    if limit_per_category <= 0:
        return rows

    return rows[:limit_per_category]

def run_category_rows(label, model_key, category, limit_per_category, max_new_tokens, timeout_sec=1200, extra_args=None):
    root = out_dir(label)
    rows = select_rows_by_category(category, limit_per_category)
    print("CATEGORY ROWS:", rows)
    records = []
    for row_no in rows:
        sub_root, rc = run_model_suite_row(
            label=f"{label}_row{row_no:03d}_{ts()}",
            model_key=model_key,
            row_no=row_no,
            max_new_tokens=max_new_tokens,
            timeout_sec=timeout_sec,
            extra_args=extra_args,
            check=False,
        )
        records.append({"row_no": row_no, "rc": rc, "run_root": str(sub_root)})
    idx = pd.DataFrame(records)
    idx_path = root / "category_run_index.csv"
    idx.to_csv(idx_path, index=False)
    print("category index:", idx_path)
    display(idx)
    return root, records

def show_category_results(records, model_key):
    cand_frames = []
    eval_frames = []
    for r in records:
        rr = Path(r["run_root"])
        cdf = candidates_df(rr, model_key)
        edf = eval_df(rr, model_key)
        if not cdf.empty:
            cdf["run_root"] = str(rr)
            cand_frames.append(cdf)
        if not edf.empty:
            edf["run_root"] = str(rr)
            eval_frames.append(edf)
    cand = pd.concat(cand_frames, ignore_index=True) if cand_frames else pd.DataFrame()
    ev = pd.concat(eval_frames, ignore_index=True) if eval_frames else pd.DataFrame()
    print("CATEGORY CANDIDATES:")
    display(cand.head(80))
    print("CATEGORY EVAL:")
    display(ev.head(80))
    return cand, ev

def show_artifact_table(paths):
    display(pd.DataFrame([
        {
            "path": str(p),
            "exists": Path(p).exists(),
            "size": Path(p).stat().st_size if Path(p).exists() else 0,
        }
        for p in paths
    ]))

def run_patch_apply(label, patches_path, genome_json=None):
    od = out_dir(label)

    cmd = [
        str(JOI_PY),
        "-m",
        "utils.ga_search.prompt_patch_apply",
        "--prompt-patches",
        str(patches_path),
        "--out-dir",
        str(od),
    ]

    if genome_json:
        cmd += ["--base-genome", str(genome_json)]

    rc, out = run_cmd(
        cmd,
        log_path=od / f"{label}.log",
        check=True,
        timeout_sec=300,
    )
    return od

SERVER_PRESET: a6000
BASE_DIR: /home/mgjeong/Desktop/llm/JOILang-Server
JOI_PY: /home/mgjeong/miniconda3/envs/joi/bin/python3.10
LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models
NB_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806
MODEL: gpt_mg.version0_13
ROW_NO: 251
MODEL_KEY_7B: qwen25_coder_7b
MODEL_KEY_14B: qwen25_coder_14b
CUDA_VISIBLE_DEVICES: 0
LOCAL_DEVICE: cuda:0


## 1. Canonical import / compile / render smoke

In [3]:
import_check = r"""
import importlib.util
for name in [
    "utils.ga_search.cli",
    "utils.ga_search.model_resolver",
    "utils.ga_search.render_adapter",
    "utils.ga_search.candidate_generation",
    "utils.ga_search.evaluation",
    "utils.ga_search.ga_engine",
    "utils.ga_search.local_model_registry",
    "utils.ga_search.local_worker",
    "utils.ga_search.model_suite_benchmark",
    "utils.det_evaluator",
]:
    spec = importlib.util.find_spec(name)
    print(name, "=>", spec.origin if spec else None)
    assert spec is not None
    forbidden = "version0_15" + "_update20260413"
    assert forbidden not in str(spec.origin)
"""
run_cmd([str(JOI_PY), "-c", import_check], log_path=out_dir("import_origin_check") / "import_origin_check.log", check=True, timeout_sec=120)
run_cmd([str(JOI_PY), "-m", "compileall", "utils/ga_search", "utils/det_evaluator.py"], log_path=out_dir("compileall_utils") / "compileall.log", check=True, timeout_sec=300)
_ = run_render(label="render_smoke", user_input="Turn on the light.", dry_run=True)


[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -c '
import importlib.util
for name in [
    "utils.ga_search.cli",
    "utils.ga_search.model_resolver",
    "utils.ga_search.render_adapter",
    "utils.ga_search.candidate_generation",
    "utils.ga_search.evaluation",
    "utils.ga_search.ga_engine",
    "utils.ga_search.local_model_registry",
    "utils.ga_search.local_worker",
    "utils.ga_search.model_suite_benchmark",
    "utils.det_evaluator",
]:
    spec = importlib.util.find_spec(name)
    print(name, "=>", spec.origin if spec else None)
    assert spec is not None
    forbidden = "version0_15" + "_update20260413"
    assert forbidden not in str(spec.origin)
'
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/import_origin_check/import_origin_check.log
utils.ga_search.cli => /home/mgjeong/Desktop/llm/JOILang-Server/utils/ga_search/cli.py
utils.ga_search.model_resolver => /home/mgjeong/Desktop/llm/JOIL

## 2. Single-row strict DET eval: real local Qwen 7B worker

기존 01의 single-row strict DET 확인 기능을 유지하되, `mock` 대신 00에서 검증한 `utils.ga_search.model_suite_benchmark` worker 경로를 사용합니다.

In [4]:
ROW_NO = int(os.environ.get("ROW_NO", str(ROW_NO)))

print("ACTIVE ROW_NO:", ROW_NO)
print("ACTIVE MODEL_KEY_7B:", MODEL_KEY_7B)
print("ACTIVE MAX_NEW_TOKENS_7B:", MAX_NEW_TOKENS_7B)
print("ACTIVE JOI_PY:", JOI_PY)
print("ACTIVE LOCAL_MODEL_BASE:", LOCAL_MODEL_BASE)

qwen7b_root, qwen7b_rc = run_model_suite_row(
    label=f"qwen7b_row{ROW_NO}_{ts()}",
    model_key=MODEL_KEY_7B,
    row_no=ROW_NO,
    max_new_tokens=MAX_NEW_TOKENS_7B,
    timeout_sec=1200,
    check=False,
)
print("qwen7b_rc:", qwen7b_rc)

qwen7b_cand, qwen7b_eval = inspect_run(qwen7b_root, MODEL_KEY_7B)
_ = show_gt_vs_generated(qwen7b_root, row_no=ROW_NO, model_key=MODEL_KEY_7B, max_rows=10)

qwen7b_ms = model_summary(qwen7b_root, MODEL_KEY_7B) or {}
qwen7b_generation_error_rate = float(qwen7b_ms.get("generation_error_rate", 1.0) or 1.0)
if qwen7b_generation_error_rate != 0.0:
    print("\n[WARN] Qwen 7B generation_error_rate != 0. Inspect raw response above.")
else:
    print("\n[OK] Real local Qwen 7B generation completed without generation error.")

ACTIVE ROW_NO: 251
ACTIVE MODEL_KEY_7B: qwen25_coder_7b
ACTIVE MAX_NEW_TOKENS_7B: 512
ACTIVE JOI_PY: /home/mgjeong/miniconda3/envs/joi/bin/python3.10
ACTIVE LOCAL_MODEL_BASE: /home/mgjeong/Desktop/llm/local_models

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.model_suite_benchmark --model gpt_mg.version0_13 --model-key qwen25_coder_7b --llm-mode worker --local-model-base-dir /home/mgjeong/Desktop/llm/local_models --worker-python /home/mgjeong/miniconda3/envs/joi/bin/python3.10 --local-device cuda:0 --local-files-only true --local-trust-remote-code true --local-max-new-tokens 512 --timeout-sec 1200 --output-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809 --row-no 251
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/run.log
{
  "output_root": "/home/mgjeong/Deskto

,row_no,category,command_eng,command_kor,gt,genome_id,generation,candidate_index,candidate_strategy,prompt_render_mode,prompt_assets_dir,prompt_log_paths,raw_response_path,candidates,generated_code,generated_json,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,service_context_source,service_context_mode,service_list_retrieval_scores,connected_devices_used,service_schema_path,repair_applied,repair_actions,backend
0,251,8,"At midnight, close the door and check the light every hour until 6 AM; if the brightness is greater than 30, lower it to 10.","자정이 되면 문을 닫고, 오전 6시까지 한 시간마다 조명을 체크해서 밝기가 30보다 크면 10으로 낮춰줘.","{""name"": """", \n""cron"": ""0 0 * * *"", \n""period"": 3600000, \n""script"": ""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(...",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_2...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(1...","active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10...",NaN,0,53156,138,0,29.498713,24.1082,provided_schema,schema_fallback,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",NaN,datasets/service_list_ver2.0.1.json,False,[],worker



EVALUATION:


,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend
0,251,8,suite_base,0,52.0533,False,False,0.360111,True,True,True,False,False,False,0.25,0.25,0.5,1.0,0.5,1.0,"[""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","{""gt_services"":[""clock_hour"",""door_close"",""light_movetobrightness"",""lightsensor_brightness""],""generated_services"":[""clock_hour"",""doorControl_close"",""lightLevel_light"",""light_moveToRGB""],""gt_receivers"":[""#Door"",""#Light"",""(#Clock"",""(#Light""],""generated_receivers"":[""#DoorLock"",""#Light"",""#LightSenso...",\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n},"active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"":"""",""cron"":""0 0 * * *"",""period"":3600000,""script"":""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n}""}","{""name"":""MidnightDoorAndLightControl"",""cron"":""0 0 * * *"",""period"":3600000,""code"":""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 1...",0,worker_direct,NaN,0,53156,138,0,29.498713,24.1082,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_2...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker



generation_error_type
nan

prompt_log_paths
["/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md", "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.json"]

raw_response_path
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json

service_context_source
provided_schema

service_list_retrieval_scores
{"status": "retrieval_disabled", "reason": "canonical ga_search has not enabled service retrieval yet"}

repair_actions
[]

RAW RESPONSE
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260


[WARN] Qwen 7B generation_error_rate != 0. Inspect raw response above.


## 3. Single-row strict DET eval: retry / diagnostic

기존 01의 non-mock diagnostic 기능을 유지하되, local endpoint unavailable test가 아니라 00에서 검증한 worker 경로의 longer-output retry로 바꿉니다.

In [5]:
RUN_QWEN7B_LONG_RETRY = False
MAX_NEW_TOKENS_7B_LONG = int(os.environ.get("MAX_NEW_TOKENS_7B_LONG", "1024"))

if RUN_QWEN7B_LONG_RETRY:
    qwen7b_long_root, qwen7b_long_rc = run_model_suite_row(
        label=f"qwen7b_row{ROW_NO}_long_{ts()}",
        model_key=MODEL_KEY_7B,
        row_no=ROW_NO,
        max_new_tokens=MAX_NEW_TOKENS_7B_LONG,
        timeout_sec=5000,
        check=False,
    )
    print("qwen7b_long_rc:", qwen7b_long_rc)
    inspect_run(qwen7b_long_root, MODEL_KEY_7B)
    _ = show_gt_vs_generated(qwen7b_long_root, row_no=ROW_NO, model_key=MODEL_KEY_7B, max_rows=10)
else:
    print("Long retry skipped. Set RUN_QWEN7B_LONG_RETRY=True only if 512-token run returns invalid_json or truncated JSON.")

Long retry skipped. Set RUN_QWEN7B_LONG_RETRY=True only if 512-token run returns invalid_json or truncated JSON.


## 4. Optional worker real row smoke

기존 01의 optional worker real smoke 기능을 유지하되, `utils.ga_search.cli search` skeleton이 아니라 model-suite worker row benchmark로 14B를 실행합니다.

In [6]:
RUN_QWEN14B_AFTER_7B_OK = os.environ.get("RUN_QWEN14B_AFTER_7B_OK", "false").lower() == "true"

if RUN_QWEN14B_AFTER_7B_OK and qwen7b_generation_error_rate == 0.0:
    qwen14b_root, qwen14b_rc = run_model_suite_row(
        label=f"qwen14b_row{ROW_NO}_{ts()}",
        model_key=MODEL_KEY_14B,
        row_no=ROW_NO,
        max_new_tokens=MAX_NEW_TOKENS_14B,
        timeout_sec=5000,
        check=False,
    )
    print("qwen14b_rc:", qwen14b_rc)
    inspect_run(qwen14b_root, MODEL_KEY_14B)
    _ = show_gt_vs_generated(qwen14b_root, row_no=ROW_NO, model_key=MODEL_KEY_14B, max_rows=10)
elif RUN_QWEN14B_AFTER_7B_OK:
    print("Qwen 14B skipped because Qwen 7B had generation errors.")
else:
    print("Qwen 14B skipped. Set RUN_QWEN14B_AFTER_7B_OK=true after Qwen 7B succeeds.")

Qwen 14B skipped. Set RUN_QWEN14B_AFTER_7B_OK=true after Qwen 7B succeeds.


## 5. Category search: real local Qwen worker by repeated row-level benchmark

기존 01의 category-level 확인 기능을 유지하되, mock search 대신 category row들을 선택해서 model-suite worker row benchmark를 반복합니다.

In [7]:
RUN_CATEGORY_7B = os.environ.get("RUN_CATEGORY_7B", "true").lower() == "true"
RUN_CATEGORY_14B = os.environ.get("RUN_CATEGORY_14B", "false").lower() == "true"

category_7b_records = []
category_14b_records = []

if RUN_CATEGORY_7B:
    category_7b_root, category_7b_records = run_category_rows(
        label=f"category{CATEGORY_TO_RUN}_qwen7b_{ts()}",
        model_key=MODEL_KEY_7B,
        category=CATEGORY_TO_RUN,
        limit_per_category=LIMIT_PER_CATEGORY,
        max_new_tokens=MAX_NEW_TOKENS_7B,
        timeout_sec=5000,
    )
    category_7b_candidates, category_7b_eval = show_category_results(category_7b_records, MODEL_KEY_7B)
else:
    print("Category 7B run skipped. Set RUN_CATEGORY_7B=true to run.")

if RUN_CATEGORY_14B:
    category_14b_root, category_14b_records = run_category_rows(
        label=f"category{CATEGORY_TO_RUN}_qwen14b_{ts()}",
        model_key=MODEL_KEY_14B,
        category=CATEGORY_TO_RUN,
        limit_per_category=LIMIT_PER_CATEGORY,
        max_new_tokens=MAX_NEW_TOKENS_14B,
        timeout_sec=2400,
    )
    category_14b_candidates, category_14b_eval = show_category_results(category_14b_records, MODEL_KEY_14B)
else:
    print("Category 14B run skipped. Set RUN_CATEGORY_14B=true after 7B category run is stable.")

CATEGORY ROWS: [121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150]

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.model_suite_benchmark --model gpt_mg.version0_13 --model-key qwen25_coder_7b --llm-mode worker --local-model-base-dir /home/mgjeong/Desktop/llm/local_models --worker-python /home/mgjeong/miniconda3/envs/joi/bin/python3.10 --local-device cuda:0 --local-files-only true --local-trust-remote-code true --local-max-new-tokens 512 --timeout-sec 5000 --output-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839 --row-no 121
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839/run.log
{
  "output_root": "/home/mgjeong/De

,row_no,rc,run_root
0,121,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839
1,122,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row122_20260624_015908
2,123,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row123_20260624_015938
3,124,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row124_20260624_020006
4,125,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row125_20260624_020033
5,126,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row126_20260624_020102
6,127,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row127_20260624_020130
7,128,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row128_20260624_020158
8,129,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row129_20260624_020226
9,130,0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row130_20260624_020255


CATEGORY CANDIDATES:


,row_no,category,command_eng,command_kor,gt,genome_id,generation,candidate_index,candidate_strategy,prompt_render_mode,prompt_assets_dir,prompt_log_paths,raw_response_path,candidates,generated_code,generated_json,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,service_context_source,service_context_mode,service_list_retrieval_scores,connected_devices_used,service_schema_path,repair_applied,repair_actions,backend,run_root
0,121,5,Switch the TV to channel 7 and switch to channel 11 after 1 hour.,TV 채널을 7번으로 바꾸고 1시간 뒤에 11번으로 바꿔줘.,"{""name"": """", \n""cron"": """", \n""period"": 0, \n""script"": ""\n(#Television).television_setchannel(7)\n\ndelay(1 HOUR)\n\n(#Television).television_setchannel(11)""}",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839/qwen25_coder_7b/prompts/row_121_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_ru...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839/qwen25_coder_7b/raw_responses/row_121_cand_0_gen_000.json,"[{""name"": ""TVChannelChange"", ""cron"": """", ""period"": -1, ""code"": ""if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n}""}]",if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n},"{""name"": ""TVChannelChange"", ""cron"": """", ""period"": -1, ""code"": ""if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n}""}",NaN,0,53141,104,0,28.573905,24.1057,provided_schema,schema_fallback,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",NaN,datasets/service_list_ver2.0.1.json,False,[],worker,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839
1,122,5,Set the air conditioner to cool mode and switch to auto mode after 30 minutes.,에어컨을 냉방 모드로 설정하고 30분 뒤에 자동 모드로 설정해줘.,"{""name"": """", \n""cron"": """", \n""period"": 0, \n""script"": ""\n(#AirConditioner).airconditioner_setairconditionermode(\""cool\"")\n\ndelay(30 MIN)\n\n(#AirConditioner).airconditioner_setairconditionermode(\""auto\"")""}",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row122_20260624_015908/qwen25_coder_7b/prompts/row_122_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_ru...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row122_20260624_015908/qwen25_coder_7b/raw_responses/row_122_cand_0_gen_000.json,"[{""name"": ""AirConditionerCoolAndAuto"", ""cron"": """", ""period"": 1800000, ""code"": ""if ((#AirConditioner).airConditionerMode_airConditionerMode == \""auto\"") {\n (#AirConditioner).airConditionerMode_setAirConditionerMode(\""cool\"")\n} else {\n (#AirConditioner).airConditionerMode_setAirConditionerMod...","if ((#AirConditioner).airConditionerMode_airConditionerMode == ""auto"") 

CATEGORY EVAL:


,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend,run_root
0,121,5,suite_base,0,51.5377,False,False,0.467925,False,True,False,False,False,True,0.000000,0.000000,1.000000,1.0,1.000000,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service""]","{""gt_services"":[""television_setchannel""],""generated_services"":[""tvChannel_setTvChannel"",""tvChannel_tvChannel""],""gt_receivers"":[""#Television""],""generated_receivers"":[""#Television"",""(#Television""],""gt_numeric_literals"":[""7"",""1"",""11""],""generated_numeric_literals"":[""7"",""7"",""1"",""11"",""11""],""gt_string_...",\n(#Television).television_setchannel(7)\n\ndelay(1 HOUR)\n\n(#Television).television_setchannel(11),if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n},"{""name"":"""",""cron"":"""",""period"":0,""script"":""\n(#Television).television_setchannel(7)\n\ndelay(1 HOUR)\n\n(#Television).television_setchannel(11)""}","{""name"":""TVChannelChange"",""cron"":"""",""period"":-1,""code"":""if ((#Television).tvChannel_tvChannel != 7) {\n (#Television).tvChannel_setTvChannel(7)\n}\ndelay(1 HOUR)\nif ((#Television).tvChannel_tvChannel != 11) {\n (#Television).tvChannel_setTvChannel(11)\n}""}",0,worker_direct,NaN,0,53141,104,0,28.573905,24.1057,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839/qwen25_coder_7b/raw_responses/row_121_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839/qwen25_coder_7b/prompts/row_121_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_ru...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839_row121_20260624_015839
1,122,5,suite_base,0,30.2873,False,False,0.092910,False,True,False,False,False,True,0.000000,0.000000,1.000000,1.0,0.000000,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""numeric_grounding""]","{""gt_services"":[""airconditioner_setairconditionermode""],""generated_services"":[""airConditionerMode_airConditionerMode"",""airConditionerMode_setAirConditionerMode""],""gt_receivers"":[""#AirConditioner""],""generated_receivers"":[""#AirConditioner"",""(#AirConditioner""],""gt_numeric_literals"":[""30""],""generate...","\n(#AirConditioner).airconditioner_setairconditionermode(""cool"")\n\ndelay(30 MIN)\n\n(#AirConditioner).airconditioner_setairconditionermode(""auto"")","if ((#AirConditioner).airConditionerMode_airConditionerMode == ""auto"") {\n (#AirConditioner).airConditionerMode_setAirConditionerMode(""cool"")\n} else {\n (#AirConditioner).airConditionerMode_setAirConditionerMode(""cool"")\n (#AirConditioner).airConditionerMode_setAirConditionerMode(""auto"")\n}","{""name"":"""",""c

Category 14B run skipped. Set RUN_CATEGORY_14B=true after 7B category run is stable.


## 6. Manual prompt patch application and visibility check

기존 01의 manual prompt patch 생성/적용/visibility check 기능을 유지합니다. Patch 적용 후 model-suite가 `--genome-json`을 지원하면 optional patched rerun을 수행합니다.

In [13]:
patch_dir = out_dir(f"manual_patch_{ts()}")
patch_path = patch_dir / "prompt_patches.json"
patch_payload = {
    "advisor_meta": {
        "source": "tutorial_manual_patch",
        "official_metric": "strict_det",
        "target_runtime": "utils.ga_search.model_suite_benchmark",
    },
    "prompt_patches": [
        {
            "patch_id": "manual_det_grounding_rule_001",
            "target_block_family": "DET_Helper",
            "target_block_id": "06",
            "operation": "append_micro_rule",
            "priority": 90,
            "patch_text": "Before final JSON, verify schedule, receiver tags, service coverage, numeric/enum grounding, and temporal order against the command.",
            "evidence_rows": [str(ROW_NO)],
            "evidence_failure_reasons": ["gt_mismatch"],
            "mutation_intent": "strict_det_repair"
        }
    ],
}
patch_path.write_text(json.dumps(patch_payload, ensure_ascii=False, indent=2), encoding="utf-8")
show_file(patch_path)

patch_apply_dir = run_patch_apply(f"patch_apply_{ts()}", patch_path)
show_artifact_table([
    patch_apply_dir / "patched_genome.json",
    patch_apply_dir / "patch_application_report.json",
    patch_apply_dir / "patch_diff.md",
    patch_apply_dir / "patched_prompt_preview.md",
])
show_file(patch_apply_dir / "patch_application_report.json")
show_file(patch_apply_dir / "patch_diff.md", max_chars=3000)

RUN_PATCHED_RERUN = os.environ.get("RUN_PATCHED_RERUN", "false").lower() == "true"
patched_7b_root = None
patched_7b_rc = None

if RUN_PATCHED_RERUN:
    patched_genome = patch_apply_dir / "patched_genome.json"
    if not patched_genome.exists():
        raise FileNotFoundError(patched_genome)
    patched_7b_root, patched_7b_rc = run_model_suite_row(
        label=f"qwen7b_row{ROW_NO}_patched_{ts()}",
        model_key=MODEL_KEY_7B,
        row_no=ROW_NO,
        max_new_tokens=MAX_NEW_TOKENS_7B,
        timeout_sec=1200,
        extra_args=["--genome-json", str(patched_genome)],
        check=False,
    )
    print("patched_7b_rc:", patched_7b_rc)
    inspect_run(patched_7b_root, MODEL_KEY_7B)
    _ = compare_eval_runs(qwen7b_root, patched_7b_root, MODEL_KEY_7B)
else:
    print("Patched rerun skipped. Set RUN_PATCHED_RERUN=true if the current model_suite_benchmark supports --genome-json.")

/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/manual_patch_20260624_021815/prompt_patches.json exists= True size= 707
{
  "advisor_meta": {
    "source": "tutorial_manual_patch",
    "official_metric": "strict_det",
    "target_runtime": "utils.ga_search.model_suite_benchmark"
  },
  "prompt_patches": [
    {
      "patch_id": "manual_det_grounding_rule_001",
      "target_block_family": "DET_Helper",
      "target_block_id": "06",
      "operation": "append_micro_rule",
      "priority": 90,
      "patch_text": "Before final JSON, verify schedule, receiver tags, service coverage, numeric/enum grounding, and temporal order against the command.",
      "evidence_rows": [
        "251"
      ],
      "evidence_failure_reasons": [
        "gt_mismatch"
      ],
      "mutation_intent": "strict_det_repair"
    }
  ]
}

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.prompt_patch_apply --prompt-patc

,path,exists,size
0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_20260624_021815/patched_genome.json,True,438
1,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_20260624_021815/patch_application_report.json,True,1197
2,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_20260624_021815/patch_diff.md,True,160
3,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_20260624_021815/patched_prompt_preview.md,True,193


/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_20260624_021815/patch_application_report.json exists= True size= 1197
{
  "created_at": "2026-06-23T17:18:15.843157+00:00",
  "prompt_patches_path": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/manual_patch_20260624_021815/prompt_patches.json",
  "base_genome_path": "fallback",
  "patched_genome_path": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_20260624_021815/patched_genome.json",
  "accepted_proposal_count": 1,
  "advisor_child_scheduled_count": 1,
  "advisor_backed_diff_count": 1,
  "patch_count": 1,
  "visible_patch_ids": [
    "manual_det_grounding_rule_001"
  ],
  "patch_visibility_ok": true,
  "applications": [
    {
      "patch_id": "manual_det_grounding_rule_001",
      "operation": "append_micro_rule",


## 7. Inspect prompt logs / raw responses / service context

기존 01의 artifact inspection 기능을 유지하되, patched mock search가 아니라 실제 model-suite worker 결과를 확인합니다.

In [9]:
inspect_root = None
inspect_model_key = MODEL_KEY_7B

patched_7b_root = globals().get("patched_7b_root", None)
qwen7b_root = globals().get("qwen7b_root", None)
category_7b_records = globals().get("category_7b_records", [])

if patched_7b_root is not None:
    inspect_root = patched_7b_root
elif qwen7b_root is not None:
    inspect_root = qwen7b_root
elif category_7b_records:
    inspect_root = Path(category_7b_records[0]["run_root"])

print("INSPECT_ROOT:", inspect_root)
if inspect_root is None:
    print("No model-suite run available to inspect.")
else:
    cdf = candidates_df(inspect_root, inspect_model_key)
    display(cdf.head(20))
    for col in ["generation_error_type", "prompt_log_paths", "raw_response_path", "service_context_source", "service_list_retrieval_scores", "repair_actions"]:
        if col in cdf.columns:
            print("\nCOLUMN:", col)
            print(cdf[col].head(5).to_string(index=False))

    if not cdf.empty:
        first = cdf.iloc[0]
        prompt_paths_raw = first.get("prompt_log_paths", "")
        try:
            prompt_paths = json.loads(prompt_paths_raw) if isinstance(prompt_paths_raw, str) else prompt_paths_raw
        except Exception:
            prompt_paths = []
        if prompt_paths:
            print("\n" + "=" * 100)
            print("FIRST PROMPT LOG")
            show_file(prompt_paths[0], max_chars=6000)

        raw_path = first.get("raw_response_path", "")
        if isinstance(raw_path, str) and raw_path.strip():
            print("\n" + "=" * 100)
            print("RAW RESPONSE")
            show_file(raw_path, max_chars=6000)

print("\nFINAL RUN INDEX")
run_index = []

for name in ["qwen7b_root", "qwen7b_long_root", "qwen14b_root", "category_7b_root", "category_14b_root", "patch_apply_dir", "patched_7b_root", "full_7b_root", "full_14b_root"]:
    value = globals().get(name, None)
    if value is not None:
        run_index.append({"name": name, "path": str(value), "exists": Path(value).exists()})
        
display(pd.DataFrame(run_index))
print("NB_ROOT:", NB_ROOT)

INSPECT_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809


,row_no,category,command_eng,command_kor,gt,genome_id,generation,candidate_index,candidate_strategy,prompt_render_mode,prompt_assets_dir,prompt_log_paths,raw_response_path,candidates,generated_code,generated_json,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,service_context_source,service_context_mode,service_list_retrieval_scores,connected_devices_used,service_schema_path,repair_applied,repair_actions,backend
0,251,8,"At midnight, close the door and check the light every hour until 6 AM; if the brightness is greater than 30, lower it to 10.","자정이 되면 문을 닫고, 오전 6시까지 한 시간마다 조명을 체크해서 밝기가 30보다 크면 10으로 낮춰줘.","{""name"": """", \n""cron"": ""0 0 * * *"", \n""period"": 3600000, \n""script"": ""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(...",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_2...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(1...","active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10...",NaN,0,53156,138,0,29.498713,24.1082,provided_schema,schema_fallback,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",NaN,datasets/service_list_ver2.0.1.json,False,[],worker



COLUMN: generation_error_type
NaN

COLUMN: prompt_log_paths
["/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md", "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20...

COLUMN: raw_response_path
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json

COLUMN: service_context_source
provided_schema

COLUMN: service_list_retrieval_scores
{"status": "retrieval_disabled", "reason": "canonical ga_search has not enabled service retrieval yet"}

COLUMN: repair_actions
[]

FIRST PROMPT LOG
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809/qwen25_coder_7b/prompts/row_251

,name,path,exists
0,qwen7b_root,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/qwen7b_row251_20260624_015809,True
1,category_7b_root,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806/category5_qwen7b_20260624_015839,True


NB_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_015806


## 8. Full local DET smoke and full-run helpers

여기부터는 내일 Excel 후처리를 위해 local DET를 전체 row에 대해 돌리는 단계입니다.

기본 원칙:

- `mock` 사용 안 함
- `utils.ga_search.model_suite_benchmark` 사용
- smoke는 작은 row subset 또는 single row로 확인
- full은 `--row-no` 없이 실행해서 전체 dataset을 처리
- full 결과는 `eval/row_evaluation.csv`, `candidates/generation_000.csv`, `model_summary.json`, `suite_summary.json`을 Excel-ready CSV로 집계

In [14]:
# ============================================================
# Full local DET helpers
# ============================================================

def run_model_suite_full(label, model_key, max_new_tokens, timeout_sec=28800, extra_args=None, check=False):
    """
    Run model_suite_benchmark without --row-no.
    This is intended to evaluate the full dataset with local worker generation.
    """
    root = out_dir(label)
    cmd = model_suite_cmd(
        root,
        model_key=model_key,
        row_no=None,
        max_new_tokens=max_new_tokens,
        timeout_sec=timeout_sec,
        extra_args=extra_args,
    )
    rc, output = run_cmd(cmd, log_path=root / "run.log", check=check, timeout_sec=timeout_sec + 600)
    return root, rc

def write_full_det_launch_script(label, model_key, max_new_tokens, timeout_sec=28800, extra_args=None):
    """
    Write a detached shell script for overnight full local DET.
    Use this when you want the run to continue after closing the browser.
    """
    root = out_dir(label)
    cmd = model_suite_cmd(
        root,
        model_key=model_key,
        row_no=None,
        max_new_tokens=max_new_tokens,
        timeout_sec=timeout_sec,
        extra_args=extra_args,
    )
    log_path = root / "nohup.log"
    script_path = root / "launch_full_det.sh"

    quoted_cmd = " ".join(shlex.quote(str(x)) for x in cmd)
    script = f"""#!/usr/bin/env bash
set -euo pipefail
cd {shlex.quote(str(BASE_DIR))}
export PYTHONUNBUFFERED=1
export CUDA_VISIBLE_DEVICES={shlex.quote(str(CUDA_VISIBLE_DEVICES))}
export LD_LIBRARY_PATH=
echo "[START] $(date)"
echo "[CMD] {quoted_cmd}"
{quoted_cmd} 2>&1 | tee {shlex.quote(str(log_path))}
echo "[END] $(date)"
"""
    script_path.write_text(script, encoding="utf-8")
    script_path.chmod(0o755)
    print("script_path:", script_path)
    print("log_path:", log_path)
    print("run_root:", root)
    print("\nTo launch manually:")
    print(f"nohup bash {shlex.quote(str(script_path))} > {shlex.quote(str(root / 'nohup.stdout'))} 2>&1 &")
    return root, script_path, log_path

def launch_detached_script(script_path, stdout_path=None):
    script_path = Path(script_path)
    stdout_path = Path(stdout_path or (script_path.parent / "nohup.stdout"))
    cmd = ["nohup", "bash", str(script_path)]
    print("[LAUNCH]")
    print(" ".join(shlex.quote(x) for x in cmd), f"> {shlex.quote(str(stdout_path))} 2>&1 &")
    with open(stdout_path, "w", encoding="utf-8") as out:
        proc = subprocess.Popen(cmd, cwd=str(BASE_DIR), stdout=out, stderr=subprocess.STDOUT)
    print("pid:", proc.pid)
    print("stdout:", stdout_path)
    return proc.pid, stdout_path

def aggregate_model_suite_outputs(run_root, model_key):
    """
    Read model-suite outputs and export Excel-friendly CSV files.
    """
    run_root = Path(run_root)
    export_dir = run_root / "excel_exports"
    export_dir.mkdir(parents=True, exist_ok=True)

    cand = candidates_df(run_root, model_key)
    ev = eval_df(run_root, model_key)
    ms = model_summary(run_root, model_key)
    ss = suite_summary(run_root)

    cand_path = export_dir / f"{model_key}_candidates.csv"
    eval_path = export_dir / f"{model_key}_row_evaluation.csv"
    fail_path = export_dir / f"{model_key}_failures_for_advisor.csv"
    summary_path = export_dir / f"{model_key}_summary.json"

    if not cand.empty:
        cand.to_csv(cand_path, index=False, encoding="utf-8-sig")
    if not ev.empty:
        ev.to_csv(eval_path, index=False, encoding="utf-8-sig")

        d = ev.copy()
        if "det_score" in d.columns:
            d["det_score_num"] = pd.to_numeric(d["det_score"], errors="coerce").fillna(0)
        else:
            d["det_score_num"] = 0
        if "det_pass" in d.columns:
            det_pass = d["det_pass"].astype(str).str.lower().eq("true")
        else:
            det_pass = d["det_score_num"] >= 70

        failures = d[(~det_pass) | (d["det_score_num"] < 70)].copy()
        preferred = [c for c in [
            "row_no", "category", "det_score", "det_pass", "gt_exact", "gt_similarity",
            "schedule_match", "service_recall", "service_precision", "receiver_recall",
            "numeric_grounding", "enum_grounding", "dataflow_score",
            "failure_reasons", "gt_code", "generated_code", "gt_json", "generated_json"
        ] if c in failures.columns]
        if preferred:
            failures = failures[preferred]
        failures.to_csv(fail_path, index=False, encoding="utf-8-sig")
    else:
        failures = pd.DataFrame()

    summary_payload = {
        "run_root": str(run_root),
        "model_key": model_key,
        "suite_summary": ss,
        "model_summary": ms,
        "candidate_rows": int(len(cand)) if not cand.empty else 0,
        "eval_rows": int(len(ev)) if not ev.empty else 0,
        "failure_rows": int(len(failures)) if not failures.empty else 0,
        "export_dir": str(export_dir),
    }
    summary_path.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding="utf-8")

    print(json.dumps(summary_payload, ensure_ascii=False, indent=2))
    show_artifact_table([cand_path, eval_path, fail_path, summary_path])
    return cand, ev, failures, export_dir

def monitor_run_folder(run_root, model_key):
    run_root = Path(run_root)
    paths = [
        run_root / "run.log",
        run_root / "nohup.log",
        run_root / "nohup.stdout",
        run_root / "suite_summary.json",
        suite_model_dir(run_root, model_key) / "model_summary.json",
        suite_model_dir(run_root, model_key) / "candidates" / "generation_000.csv",
        suite_model_dir(run_root, model_key) / "eval" / "row_evaluation.csv",
    ]
    show_artifact_table(paths)
    for p in [run_root / "nohup.log", run_root / "run.log", run_root / "nohup.stdout"]:
        if p.exists():
            print("\n" + "=" * 100)
            print("TAIL:", p)
            text = p.read_text(encoding="utf-8", errors="replace")
            print(text[-6000:])
            break

## 9. Full local DET smoke

전체 280개를 돌리기 전에 single-row smoke 결과가 정상인지 다시 확인합니다.

- `generation_error_rate=0.0`
- `prompt_tokens/completion_tokens > 0`
- `raw_response_path` 존재
- `row_evaluation.csv` 생성

이 cell은 기존 single-row 결과를 재사용하거나, 필요하면 새 smoke를 돌립니다.

In [15]:
RUN_FULL_DET_SMOKE = os.environ.get("RUN_FULL_DET_SMOKE", "true").lower() == "true"

if RUN_FULL_DET_SMOKE:
    smoke_row = int(os.environ.get("FULL_DET_SMOKE_ROW", str(ROW_NO)))
    smoke_root, smoke_rc = run_model_suite_row(
        label=f"full_det_smoke_{MODEL_KEY_7B}_row{smoke_row}_{ts()}",
        model_key=MODEL_KEY_7B,
        row_no=smoke_row,
        max_new_tokens=MAX_NEW_TOKENS_7B,
        timeout_sec=1200,
        check=False,
    )
    print("smoke_rc:", smoke_rc)
    inspect_run(smoke_root, MODEL_KEY_7B)
    _ = show_gt_vs_generated(smoke_root, row_no=smoke_row, model_key=MODEL_KEY_7B, max_rows=10)
else:
    print("Full DET smoke skipped.")


[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.model_suite_benchmark --model gpt_mg.version0_13 --model-key qwen25_coder_7b --llm-mode worker --local-model-base-dir /home/mgjeong/Desktop/llm/local_models --worker-python /home/mgjeong/miniconda3/envs/joi/bin/python3.10 --local-device cuda:0 --local-files-only true --local-trust-remote-code true --local-max-new-tokens 512 --timeout-sec 1200 --output-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829 --row-no 251
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/run.log
{
  "output_root": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829",
  "preflight": [


,row_no,category,command_eng,command_kor,gt,genome_id,generation,candidate_index,candidate_strategy,prompt_render_mode,prompt_assets_dir,prompt_log_paths,raw_response_path,candidates,generated_code,generated_json,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,service_context_source,service_context_mode,service_list_retrieval_scores,connected_devices_used,service_schema_path,repair_applied,repair_actions,backend
0,251,8,"At midnight, close the door and check the light every hour until 6 AM; if the brightness is greater than 30, lower it to 10.","자정이 되면 문을 닫고, 오전 6시까지 한 시간마다 조명을 체크해서 밝기가 30보다 크면 10으로 낮춰줘.","{""name"": """", \n""cron"": ""0 0 * * *"", \n""period"": 3600000, \n""script"": ""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(...",suite_base,0,0,worker_direct,monolith,/home/mgjeong/Desktop/llm/JOILang-Server/gpt_mg/version0_13,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs...",/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(1...","active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"": ""MidnightDoorAndLightControl"", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""code"": ""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10...",NaN,0,53156,138,0,29.713297,24.1082,provided_schema,schema_fallback,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",NaN,datasets/service_list_ver2.0.1.json,False,[],worker



EVALUATION:


,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend
0,251,8,suite_base,0,52.0533,False,False,0.360111,True,True,True,False,False,False,0.25,0.25,0.5,1.0,0.5,1.0,"[""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","{""gt_services"":[""clock_hour"",""door_close"",""light_movetobrightness"",""lightsensor_brightness""],""generated_services"":[""clock_hour"",""doorControl_close"",""lightLevel_light"",""light_moveToRGB""],""gt_receivers"":[""#Door"",""#Light"",""(#Clock"",""(#Light""],""generated_receivers"":[""#DoorLock"",""#Light"",""#LightSenso...",\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n},"active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 10)\n }\n} while ((#Clock).clock_hour < 6)","{""name"":"""",""cron"":""0 0 * * *"",""period"":3600000,""script"":""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n}""}","{""name"":""MidnightDoorAndLightControl"",""cron"":""0 0 * * *"",""period"":3600000,""code"":""active := false\nif (active == false) {\n (#DoorLock).doorControl_close()\n active = true\n}\ndo {\n brightness = (#LightSensor).lightLevel_light\n if (brightness > 30) {\n (#Light).light_moveToRGB(10, 10, 1...",0,worker_direct,NaN,0,53156,138,0,29.713297,24.1082,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker



generation_error_type
nan

prompt_log_paths
["/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md", "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.json"]

raw_response_path
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_smoke_qwen25_coder_7b_row251_20260624_021829/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json

service_context_source
provided_schema

service_list_retrieval_scores
{"status": "retrieval_disabled", "reason": "canonical ga_search has not enabled service retrieval yet"}

repair_actions
[]

RAW RESPONSE
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search

## 10. Full local DET all rows: Qwen 7B

이 cell이 오늘 밤 돌려놓을 핵심 cell입니다.

- `RUN_FULL_7B_NOW=true`이면 notebook 안에서 바로 실행합니다.
- `DETACH_FULL_7B=true`이면 `nohup`으로 detached 실행합니다.
- 기본은 detached 실행을 권장합니다.
- 전체 dataset을 돌리므로 `--row-no`를 넣지 않습니다.

In [16]:
RUN_FULL_7B_NOW = os.environ.get("RUN_FULL_7B_NOW", "false").lower() == "true"
DETACH_FULL_7B = os.environ.get("DETACH_FULL_7B", "true").lower() == "true"
RUN_FULL_7B_NOW = True
DETACH_FULL_7B = True

FULL_7B_TIMEOUT_SEC = int(os.environ.get("FULL_7B_TIMEOUT_SEC", "43200"))  # 12 hours

full_7b_root = None
full_7b_rc = None
full_7b_script = None
full_7b_log = None

if RUN_FULL_7B_NOW:
    label = f"full_det_{MODEL_KEY_7B}_allrows_{ts()}"
    if DETACH_FULL_7B:
        full_7b_root, full_7b_script, full_7b_log = write_full_det_launch_script(
            label=label,
            model_key=MODEL_KEY_7B,
            max_new_tokens=MAX_NEW_TOKENS_7B,
            timeout_sec=FULL_7B_TIMEOUT_SEC,
        )
        full_7b_pid, full_7b_stdout = launch_detached_script(full_7b_script)
        print("FULL_7B_PID:", full_7b_pid)
        print("FULL_7B_ROOT:", full_7b_root)
        print("FULL_7B_LOG:", full_7b_log)
    else:
        full_7b_root, full_7b_rc = run_model_suite_full(
            label=label,
            model_key=MODEL_KEY_7B,
            max_new_tokens=MAX_NEW_TOKENS_7B,
            timeout_sec=FULL_7B_TIMEOUT_SEC,
            check=False,
        )
        print("full_7b_rc:", full_7b_rc)
        aggregate_model_suite_outputs(full_7b_root, MODEL_KEY_7B)
else:
    print("Full 7B run not launched.")
    print("To launch overnight from notebook, run:")
    print("  RUN_FULL_7B_NOW = True")
    print("  DETACH_FULL_7B = True")
    print("then rerun this cell.")

script_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/launch_full_det.sh
log_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/nohup.log
run_root: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859

To launch manually:
nohup bash /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/launch_full_det.sh > /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/nohup.stdout 2>&1 &
[LAUNCH]
nohup bash /home/mgjeong/Desktop/llm/JOILang-S

## 11. Monitor full local DET run

`full_7b_root`가 현재 kernel에 있으면 바로 모니터링합니다.  
kernel을 다시 열었으면 아래 `MANUAL_FULL_7B_ROOT`에 run_root 경로를 넣고 실행하세요.

In [17]:
MANUAL_FULL_7B_ROOT = os.environ.get("MANUAL_FULL_7B_ROOT", "").strip()

monitor_root = None
if "full_7b_root" in globals() and full_7b_root is not None:
    monitor_root = full_7b_root
elif MANUAL_FULL_7B_ROOT:
    monitor_root = Path(MANUAL_FULL_7B_ROOT)

if monitor_root is None:
    print("No full run root to monitor yet.")
    print("Set MANUAL_FULL_7B_ROOT=/path/to/full_det_run_root if the run was launched in another session.")
else:
    monitor_run_folder(monitor_root, MODEL_KEY_7B)

,path,exists,size
0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/run.log,False,0
1,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/nohup.log,True,0
2,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/nohup.stdout,True,636
3,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/suite_summary.json,False,0
4,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/model_summary.json,False,0
5,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/candidates/generation_000.csv,False,0
6,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/eval/row_evaluation.csv,False,0



TAIL: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/nohup.log



## 12. Export full local DET results for Excel / advisor prompt work

내일 할 작업을 위해 다음 파일을 만듭니다.

- `*_row_evaluation.csv`
- `*_candidates.csv`
- `*_failures_for_advisor.csv`
- `*_summary.json`

`*_failures_for_advisor.csv`를 Excel로 열어서 row별 failure reason, GT, generated code를 보고 advisor prompt와 JOICode generation prompt를 수정하면 됩니다.

In [18]:
EXPORT_FULL_7B_ROOT = os.environ.get("EXPORT_FULL_7B_ROOT", "").strip()

export_root = None
if "full_7b_root" in globals() and full_7b_root is not None:
    export_root = full_7b_root
elif EXPORT_FULL_7B_ROOT:
    export_root = Path(EXPORT_FULL_7B_ROOT)

if export_root is None:
    print("No full 7B root available.")
    print("Set EXPORT_FULL_7B_ROOT=/path/to/full_det_run_root after the overnight run finishes.")
else:
    full_7b_candidates, full_7b_eval, full_7b_failures, full_7b_export_dir = aggregate_model_suite_outputs(export_root, MODEL_KEY_7B)
    print("Excel export dir:", full_7b_export_dir)
    display(full_7b_failures.head(80))

{
  "run_root": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859",
  "model_key": "qwen25_coder_7b",
  "suite_summary": {},
  "model_summary": {},
  "candidate_rows": 0,
  "eval_rows": 0,
  "failure_rows": 0,
  "export_dir": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/excel_exports"
}


,path,exists,size
0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/excel_exports/qwen25_coder_7b_candidates.csv,False,0
1,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/excel_exports/qwen25_coder_7b_row_evaluation.csv,False,0
2,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/excel_exports/qwen25_coder_7b_failures_for_advisor.csv,False,0
3,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/excel_exports/qwen25_coder_7b_summary.json,True,518


Excel export dir: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/excel_exports


""


## 13. Optional full local DET all rows: Qwen 14B

7B full result를 먼저 확인한 뒤 14B를 돌리세요.  
14B는 시간이 오래 걸리므로 기본값은 실행하지 않습니다.

In [19]:
RUN_FULL_14B_NOW = os.environ.get("RUN_FULL_14B_NOW", "false").lower() == "true"
DETACH_FULL_14B = os.environ.get("DETACH_FULL_14B", "true").lower() == "true"
FULL_14B_TIMEOUT_SEC = int(os.environ.get("FULL_14B_TIMEOUT_SEC", "86400"))  # 24 hours

full_14b_root = None

if RUN_FULL_14B_NOW:
    label = f"full_det_{MODEL_KEY_14B}_allrows_{ts()}"
    if DETACH_FULL_14B:
        full_14b_root, full_14b_script, full_14b_log = write_full_det_launch_script(
            label=label,
            model_key=MODEL_KEY_14B,
            max_new_tokens=MAX_NEW_TOKENS_14B,
            timeout_sec=FULL_14B_TIMEOUT_SEC,
        )
        full_14b_pid, full_14b_stdout = launch_detached_script(full_14b_script)
        print("FULL_14B_PID:", full_14b_pid)
        print("FULL_14B_ROOT:", full_14b_root)
        print("FULL_14B_LOG:", full_14b_log)
    else:
        full_14b_root, full_14b_rc = run_model_suite_full(
            label=label,
            model_key=MODEL_KEY_14B,
            max_new_tokens=MAX_NEW_TOKENS_14B,
            timeout_sec=FULL_14B_TIMEOUT_SEC,
            check=False,
        )
        print("full_14b_rc:", full_14b_rc)
        aggregate_model_suite_outputs(full_14b_root, MODEL_KEY_14B)
else:
    print("Full 14B run skipped. Set RUN_FULL_14B_NOW=true only after 7B full run is stable.")

Full 14B run skipped. Set RUN_FULL_14B_NOW=true only after 7B full run is stable.


## 14. Rerun selected failed rows after prompt/advisor edits

내일 Excel 검토 후, 수정한 prompt/genome을 적용해 실패 row만 다시 돌릴 때 사용하는 cell입니다.

`FAILED_ROWS_TO_RERUN`에 row 번호를 넣고, `PATCHED_GENOME_JSON`이 있으면 같이 넘깁니다.

In [20]:
FAILED_ROWS_TO_RERUN_RAW = os.environ.get("FAILED_ROWS_TO_RERUN", "").strip()
PATCHED_GENOME_JSON = os.environ.get("PATCHED_GENOME_JSON", "").strip()
RERUN_FAILED_NOW = os.environ.get("RERUN_FAILED_NOW", "false").lower() == "true"

rerun_records = []

if RERUN_FAILED_NOW:
    if not FAILED_ROWS_TO_RERUN_RAW:
        raise ValueError("Set FAILED_ROWS_TO_RERUN, e.g. '121,122,125'")
    failed_rows = [int(x.strip()) for x in FAILED_ROWS_TO_RERUN_RAW.split(",") if x.strip()]
    extra = []
    if PATCHED_GENOME_JSON:
        extra += ["--genome-json", PATCHED_GENOME_JSON]

    for row_no in failed_rows:
        rr, rc = run_model_suite_row(
            label=f"rerun_failed_row{row_no}_{MODEL_KEY_7B}_{ts()}",
            model_key=MODEL_KEY_7B,
            row_no=row_no,
            max_new_tokens=MAX_NEW_TOKENS_7B,
            timeout_sec=1200,
            extra_args=extra,
            check=False,
        )
        rerun_records.append({"row_no": row_no, "rc": rc, "run_root": str(rr)})
    display(pd.DataFrame(rerun_records))
else:
    print("Failed-row rerun skipped.")
    print("After Excel/advisor prompt edits, set:")
    print("  FAILED_ROWS_TO_RERUN=121,122,125")
    print("  PATCHED_GENOME_JSON=/path/to/patched_genome.json  # optional")
    print("  RERUN_FAILED_NOW=true")

Failed-row rerun skipped.
After Excel/advisor prompt edits, set:
  FAILED_ROWS_TO_RERUN=121,122,125
  PATCHED_GENOME_JSON=/path/to/patched_genome.json  # optional
  RERUN_FAILED_NOW=true


## 15. Load strict DET row evaluation for row-advisor mapping

이 cell은 full/local DET 실행 결과의 `row_evaluation.csv`를 읽습니다.  
우선순위는 다음입니다.

1. `ROW_EVALUATION_CSV` 환경변수
2. `EXPORT_FULL_7B_ROOT` 환경변수
3. 현재 notebook 변수 `full_7b_root`
4. 현재 notebook 변수 `qwen7b_root`
5. `NB_ROOT` 아래 가장 최근 `row_evaluation.csv`

이후 `row_eval_df`를 만들고, 필요하면 `JOICommands-280.csv`의 command column을 병합합니다.

In [21]:
# ============================================================
# Cell 15. Load strict DET row_evaluation.csv
# ============================================================

from collections import Counter, defaultdict
import re

def _existing_path_or_none(x):
    if x is None:
        return None
    p = Path(str(x)).expanduser()
    return p if p.exists() else None

def _candidate_row_eval_paths_from_root(root, model_keys=None):
    root = Path(root).expanduser()
    model_keys = model_keys or []
    paths = []
    for mk in model_keys:
        paths.append(root / str(mk) / "eval" / "row_evaluation.csv")
    paths.extend([
        root / "eval" / "row_evaluation.csv",
        root / "row_evaluation.csv",
    ])
    paths.extend(sorted(root.glob("*/eval/row_evaluation.csv")))
    paths.extend(sorted(root.rglob("row_evaluation.csv"), key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)[:10])
    # preserve order, remove duplicates
    out = []
    seen = set()
    for p in paths:
        key = str(p)
        if key not in seen:
            out.append(p)
            seen.add(key)
    return out

def pick_row_evaluation_csv():
    explicit = os.environ.get("ROW_EVALUATION_CSV", "").strip()
    if explicit:
        p = Path(explicit).expanduser()
        if not p.exists():
            raise FileNotFoundError(f"ROW_EVALUATION_CSV not found: {p}")
        return p

    model_keys = [
        globals().get("MODEL_KEY_7B", os.environ.get("MODEL_KEY_7B", "qwen25_coder_7b")),
        globals().get("MODEL_KEY_14B", os.environ.get("MODEL_KEY_14B", "qwen25_coder_14b")),
    ]

    candidate_roots = []
    for env_name in ["EXPORT_FULL_7B_ROOT", "MANUAL_FULL_7B_ROOT", "EXPORT_FULL_14B_ROOT"]:
        value = os.environ.get(env_name, "").strip()
        if value:
            candidate_roots.append(Path(value).expanduser())

    for var_name in ["full_7b_root", "full_14b_root", "qwen7b_root", "qwen14b_root", "smoke_root", "monitor_root", "export_root"]:
        value = globals().get(var_name, None)
        if value is not None:
            candidate_roots.append(Path(value).expanduser())

    if "NB_ROOT" in globals():
        candidate_roots.append(Path(NB_ROOT))

    for root in candidate_roots:
        if not root.exists():
            continue
        for p in _candidate_row_eval_paths_from_root(root, model_keys=model_keys):
            if p.exists() and p.stat().st_size > 0:
                return p

    raise FileNotFoundError(
        "Cannot find row_evaluation.csv. Set ROW_EVALUATION_CSV=/path/to/row_evaluation.csv"
    )

def merge_dataset_commands(row_df):
    if "DATASET" not in globals():
        return row_df
    dataset_path = Path(DATASET)
    if not dataset_path.exists():
        return row_df
    ds = pd.read_csv(dataset_path)
    if "row_no" not in row_df.columns:
        return row_df

    if "row_no" not in ds.columns:
        ds = ds.copy()
        ds["row_no"] = range(1, len(ds) + 1)

    keep_cols = [c for c in ["row_no", "category", "command_eng", "command_kor", "gt"] if c in ds.columns]
    ds_small = ds[keep_cols].drop_duplicates("row_no")
    merged = row_df.merge(ds_small, on="row_no", how="left", suffixes=("", "_dataset"))

    # Prefer evaluation category/gt when present, fill only missing.
    for c in ["category", "gt"]:
        dc = c + "_dataset"
        if dc in merged.columns:
            if c in merged.columns:
                merged[c] = merged[c].where(merged[c].notna(), merged[dc])
                merged = merged.drop(columns=[dc])
            else:
                merged = merged.rename(columns={dc: c})
    return merged

ROW_EVALUATION_CSV = pick_row_evaluation_csv()
ROW_ADVISOR_OUT_DIR = ROW_EVALUATION_CSV.parent.parent / "row_advisor_mapping"
ROW_ADVISOR_OUT_DIR.mkdir(parents=True, exist_ok=True)

row_eval_df = pd.read_csv(ROW_EVALUATION_CSV)
row_eval_df = merge_dataset_commands(row_eval_df)

print("ROW_EVALUATION_CSV:", ROW_EVALUATION_CSV)
print("ROW_ADVISOR_OUT_DIR:", ROW_ADVISOR_OUT_DIR)
print("row_eval_df rows:", len(row_eval_df))
print("row_eval_df columns:", len(row_eval_df.columns))
display(row_eval_df.head(5))

ROW_EVALUATION_CSV: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/eval/row_evaluation.csv
ROW_ADVISOR_OUT_DIR: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping
row_eval_df rows: 280
row_eval_df columns: 47


,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend,command_eng,command_kor,gt
0,1,1,suite_base,0,65.1471,False,False,0.921569,False,True,False,False,False,True,0.0,0.0,1.0,1.0,1.0,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service""]","{""gt_services"":[""dishwasher_setdishwashermode""],""generated_services"":[""dishwasherMode_setDishwasherMode""],""gt_receivers"":[""#Dishwasher""],""generated_receivers"":[""#Dishwasher""],""gt_numeric_literals"":[],""generated_numeric_literals"":[],""gt_string_args"":[""dry""],""generated_string_args"":[""dry""]}","(#Dishwasher).dishwasher_setdishwashermode(""dry"")","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")","{""name"":"""",""cron"":"""",""period"":0,""script"":""(#Dishwasher).dishwasher_setdishwashermode(\""dry\"")""}","{""name"":""DishwasherDryMode"",""cron"":"""",""period"":-1,""code"":""(#Dishwasher).dishwasherMode_setDishwasherMode(\""dry\"")""}",0,worker_direct,NaN,0,53129,52,0,26.703795,24.1048,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_responses/row_1_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/prompts/row_1_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudl...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker,Switch the dishwasher to dry mode.,식기세척기를 건조 모드로 설정해줘.,"{""name"": """", \n""cron"": """", \n""period"": 0, \n""script"": ""(#Dishwasher).dishwasher_setdishwashermode(\""dry\"")""}"
1,2,1,suite_base,0,27.7439,False,False,0.341463,False,True,False,False,False,False,0.0,0.0,0.0,1.0,0.0,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","{""gt_services"":[""oven_addmoretime""],""generated_services"":[""dishwasherMode_setDishwasherMode""],""gt_receivers"":[""#Oven""],""generated_receivers"":[""#Dishwasher""],""gt_numeric_literals"":[""300""],""generated_numeric_literals"":[],""gt_string_args"":[],""generated_string_args"":[""dry""]}",(#Oven).oven_addmoretime(300),"(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")","{""name"":"""",""cron"":"""",""period"":0,""script"":""(#Oven).oven_addmoretime(300)""}","{""name"":""DishwasherDryMode"",""cron"":"""",""period"":-1,""code"":""(#Dishwasher).dishwasherMode_setDishwasherMode(\""dry\"")""}",0,worker_direct,NaN,0,53129,52,0,26.894260,24.1048,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_responses/row_2_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/prompts/row_2_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudl...",datasets/se

## 16. Failure taxonomy table and advisor routing policy

이 cell은 `failure_reason`, `generation_state`, cloud auxiliary signal을 advisor target family / block / mutation policy로 매핑하는 taxonomy를 정의합니다.

검증 포인트:

- `cron_mismatch`, `period_mismatch`는 fallback이 아니라 `Temporal_Rule`로 직접 매핑합니다.
- `missing_required_key:<keys>`는 `missing_required_key`로 normalize해서 `Output_Schema`로 매핑합니다.
- `gt_mismatch`, `semantic`, `gpt_semantic`은 umbrella signal이므로 concrete reason이 있으면 후순위로 처리합니다.
- generation/runtime failure는 semantic/service mutation이 아니라 health/runtime/budget route로 short-circuit합니다.

In [22]:
# ============================================================
# Cell 16. Failure taxonomy table and routing policy
# ============================================================

TAXONOMY_ROWS = [
    # Schedule
    {
        "category": "Schedule",
        "signal": "cron_mismatch",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["gt_json.cron", "generated_json.cron", "cron_match", "schedule_match"],
        "diagnostic_template": 'cron mismatch: gt cron={gt_cron!r}, generated cron={generated_cron!r}. Fixed wall-clock schedule was lost.',
        "target_family": "Temporal_Rule",
        "target_block_id": "06",
        "mutation_policy": "fixed wall-clock/day schedule uses cron first; do not replace with period + Clock guard",
        "difficulty": "easy",
        "priority": 30,
        "final_check": "Added explicit mapping; no DET_Helper fallback.",
        "micro_rule": "For explicit fixed times, weekdays, midnight, or scheduled one-shot commands, derive cron first and preserve it exactly. Do not replace a fixed schedule with a period loop or a Clock guard unless repeated monitoring is explicit.",
    },
    {
        "category": "Schedule",
        "signal": "period_mismatch",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["gt_json.period", "generated_json.period", "period_match", "schedule_match"],
        "diagnostic_template": "period mismatch: gt period={gt_period!r}, generated period={generated_period!r}.",
        "target_family": "Temporal_Rule",
        "target_block_id": "06",
        "mutation_policy": "classify one-shot / cron / repeated loop first, then apply period policy",
        "difficulty": "easy",
        "priority": 30,
        "final_check": "Added explicit mapping; no DET_Helper fallback.",
        "micro_rule": "For one-shot action or scheduled one-shot commands, use period=0 unless repeated monitoring is explicit. Use positive period only for repeated monitoring loops and never use -1 as a substitute for a valid one-shot period.",
    },
    # Overall / umbrella
    {
        "category": "Overall",
        "signal": "gt_mismatch",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["gt_code", "generated_code", "gt_similarity", "diff_summary"],
        "diagnostic_template": "gt mismatch: code is schema-valid but not target-equivalent; prioritize concrete mismatches.",
        "target_family": "DET_Helper",
        "target_block_id": "06",
        "mutation_policy": "umbrella only; do not make primary patch if concrete service/receiver/schedule/numeric/enum reason exists",
        "difficulty": "medium",
        "priority": 90,
        "final_check": "Keep as fallback or support reason only.",
        "micro_rule": "When code is schema-valid but not target-equivalent, compare schedule, receiver, service, numeric, enum, dataflow, and action order before final output.",
    },
    # Service
    {
        "category": "Service",
        "signal": "gt_service_coverage",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["diff_summary.gt_services", "diff_summary.generated_services"],
        "diagnostic_template": "missing GT service: gt services={gt_services}, generated services={generated_services}.",
        "target_family": "Service_Mapping",
        "target_block_id": "02",
        "mutation_policy": "select command target receiver first, then select schema-valid service under that receiver",
        "difficulty": "easy",
        "priority": 20,
        "final_check": "High automation value.",
        "micro_rule": "Include every service implied by the command. Select services only from the injected schema under the selected receiver and do not substitute adjacent service families.",
    },
    {
        "category": "Service",
        "signal": "unknown_service",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["diff_summary.generated_services", "diff_summary.gt_services", "generated_code"],
        "diagnostic_template": "unknown/canonical service error: generated services={generated_services}; expected schema/GT services={gt_services}.",
        "target_family": "Service_Mapping",
        "target_block_id": "02",
        "mutation_policy": "replace non-schema member with nearest valid canonical schema member before final output",
        "difficulty": "easy",
        "priority": 10,
        "final_check": "Already maps to Service_Mapping; expand with camelCase/class-style prohibition.",
        "micro_rule": "Never invent service/member names. Copy the canonical device-prefixed service member exactly from the injected service schema. Do not emit camelCase, class-style, capitalized, or paraphrased service names.",
    },
    # Receiver
    {
        "category": "Receiver",
        "signal": "gt_receiver_coverage",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["diff_summary.gt_receivers", "diff_summary.generated_receivers"],
        "diagnostic_template": "receiver mismatch: gt receivers={gt_receivers}, generated receivers={generated_receivers}.",
        "target_family": "Receiver_Tag_Preservation",
        "target_block_id": "02",
        "mutation_policy": "owner/location/group/sector tag preservation; condition receiver and action receiver may differ",
        "difficulty": "easy",
        "priority": 15,
        "final_check": "Mapping is appropriate.",
        "micro_rule": "Select receiver tags from the current command target before service selection. Preserve owner, location, group, and sector tags exactly, and do not reuse a receiver from another row.",
    },
    # Numeric / enum / args
    {
        "category": "Numeric",
        "signal": "numeric_grounding",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["diff_summary.gt_numeric_literals", "diff_summary.generated_numeric_literals", "service descriptor"],
        "diagnostic_template": "numeric mismatch: gt numeric literals={gt_numeric_literals}, generated numeric literals={generated_numeric_literals}.",
        "target_family": "Numeric_Unit_Grounding",
        "target_block_id": "06",
        "mutation_policy": "temporal numbers and service argument numbers both use descriptor-grounded conversion",
        "difficulty": "easy",
        "priority": 35,
        "final_check": "Separate from pure Temporal_Rule when numeric is a service argument.",
        "micro_rule": "Preserve required numeric arguments and thresholds from the command. Convert units using the selected service descriptor, such as minutes to seconds for seconds-based arguments.",
    },
    {
        "category": "Enum/String",
        "signal": "enum_grounding",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["diff_summary.gt_string_args", "diff_summary.generated_string_args"],
        "diagnostic_template": "enum mismatch: gt string args={gt_string_args}, generated string args={generated_string_args}.",
        "target_family": "Enum_Grounding",
        "target_block_id": "02",
        "mutation_policy": "copy allowed enum from selected service descriptor only",
        "difficulty": "easy",
        "priority": 40,
        "final_check": "Mapping is appropriate.",
        "micro_rule": "For enum-valued services, copy the allowed enum value exactly from the selected service descriptor. Do not translate, paraphrase, or borrow enum values from another device or service.",
    },
    {
        "category": "Argument",
        "signal": "arg_type",
        "signal_type": "failure_reason",
        "produced_by": "local_report",
        "evidence_fields": ["service descriptor", "gt_code", "generated_code"],
        "diagnostic_template": "argument type mismatch: preserve positional order, numeric/string/boolean type, and schema separator.",
        "target_family": "Argument_Grounding",
        "target_block_id": "02",
        "mutation_policy": "type/order/separator/format rule",
        "difficulty": "medium",
        "priority": 45,
        "final_check": "Argument shape must be descriptor-grounded.",
        "micro_rule": "Preserve positional argument order, argument type, separator, bounds, and format from the selected service descriptor.",
    },
    # Dataflow and structure
    {
        "category": "Dataflow",
        "signal": "dataflow",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["gt_code", "generated_code", "variable assignment/use pattern"],
        "diagnostic_template": "dataflow mismatch: generated does not preserve read-bind-use structure required by GT.",
        "target_family": "Dataflow",
        "target_block_id": "06",
        "mutation_policy": "sensor read → variable bind → downstream speak/action structure preservation",
        "difficulty": "medium",
        "priority": 50,
        "final_check": "JOILang ':=' variable pattern should be included in diagnostic logic.",
        "micro_rule": "When reading a value for reporting or control, bind it with JOILang ':=' and use that bound value downstream. Do not replace read-bind-use flow with an unrelated direct action.",
    },
    {
        "category": "Semantic",
        "signal": "semantic",
        "signal_type": "failure_reason",
        "produced_by": "local_or_cloud",
        "evidence_fields": ["command", "gt_code", "generated_code", "reasoning"],
        "diagnostic_template": "semantic intent mismatch: generated code follows a different high-level intent than the command.",
        "target_family": "Skeleton",
        "target_block_id": "06",
        "mutation_policy": "use only with concrete diagnostic; do not patch from semantic alone",
        "difficulty": "hard",
        "priority": 100,
        "final_check": "Dangerous as standalone patch reason.",
        "micro_rule": "Classify the current command as one-shot, condition-action, cron schedule, period loop, or trigger-then-repeat before service emission.",
    },
    {
        "category": "Condition",
        "signal": "conditions",
        "signal_type": "cloud_dimension",
        "produced_by": "cloud_auxiliary",
        "evidence_fields": ["condition diff", "cloud reasoning"],
        "diagnostic_template": "condition mismatch: generated condition does not preserve command precondition or trigger subject.",
        "target_family": "Skeleton",
        "target_block_id": "06",
        "mutation_policy": "condition subject/action target separation",
        "difficulty": "hard",
        "priority": 85,
        "final_check": "Use only as auxiliary with strict DET evidence.",
        "micro_rule": "Preserve explicit if-condition subjects and action targets separately. Do not convert a precondition into an action or omit the trigger subject.",
    },
    {
        "category": "Precondition",
        "signal": "precondition",
        "signal_type": "failure_reason",
        "produced_by": "local_report",
        "evidence_fields": ["command", "gt condition blocks", "generated condition blocks"],
        "diagnostic_template": "precondition mismatch: output ignores or changes required if-condition before action.",
        "target_family": "Skeleton",
        "target_block_id": "06",
        "mutation_policy": "precondition-first skeleton",
        "difficulty": "medium",
        "priority": 60,
        "final_check": "Mapping to Skeleton is appropriate.",
        "micro_rule": "Represent explicit state preconditions as guard conditions before actions. Do not replace a required state check with an unconditional action.",
    },
    # Output schema and generation state
    {
        "category": "JSON",
        "signal": "invalid_json",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["generation_error_type", "raw candidate", "extraction status"],
        "diagnostic_template": "invalid JSON: return exactly one JSON object with keys name, cron, period, code; no markdown/prose/code fences.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "JSON-only if raw candidate exists; Generation_Health if raw candidate is empty",
        "difficulty": "easy",
        "priority": 5,
        "final_check": "Use generation-state route to split empty/runtime vs malformed JSON.",
        "micro_rule": "Return exactly one JSON object with required keys name, cron, period, and code. Do not emit markdown fences, prose, comments, or multiple JSON objects.",
    },
    {
        "category": "JSON",
        "signal": "invalid_json.non_json_text",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["raw candidate excerpt", "json_error_type"],
        "diagnostic_template": "non-json output: raw candidate exists but starts as prose/text.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "JSON-only rule",
        "difficulty": "easy",
        "priority": 5,
        "final_check": "Route exists.",
        "micro_rule": "Return bare JSON only. Do not introduce the answer with prose or explanation.",
    },
    {
        "category": "JSON",
        "signal": "invalid_json.markdown_fence",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["raw candidate", "had_markdown_fence"],
        "diagnostic_template": "markdown fence error: remove JSON/code fences and return bare JSON only.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "fence prohibition",
        "difficulty": "easy",
        "priority": 5,
        "final_check": "Route exists.",
        "micro_rule": "Do not wrap the final JSON in markdown code fences. Output the JSON object directly.",
    },
    {
        "category": "JSON",
        "signal": "invalid_json.malformed_json",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["raw candidate excerpt", "parse error"],
        "diagnostic_template": "malformed JSON: raw output exists but cannot be parsed.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "parseable JSON contract",
        "difficulty": "easy",
        "priority": 5,
        "final_check": "Semantic mutation prohibited.",
        "micro_rule": "Emit parseable JSON with double-quoted keys and values where required. Do not output trailing text after the JSON object.",
    },
    {
        "category": "JSON",
        "signal": "truncated_json",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["raw candidate excerpt", "max tokens"],
        "diagnostic_template": "truncated JSON: output ended before a complete JSON object was produced.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "shorter parseable JSON; also inspect max_new_tokens",
        "difficulty": "easy",
        "priority": 5,
        "final_check": "May require runtime token budget.",
        "micro_rule": "Keep the final JSON concise and complete. Close every object/string and avoid verbose names or explanations that risk truncation.",
    },
    {
        "category": "Schema",
        "signal": "missing_required_key",
        "signal_type": "failure_reason_prefix",
        "produced_by": "strict_det",
        "evidence_fields": ["generated JSON keys", "missing suffix"],
        "diagnostic_template": "missing required JSON key: {reason}.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "required key rule",
        "difficulty": "easy",
        "priority": 7,
        "final_check": "Normalize missing_required_key:<keys>.",
        "micro_rule": "Always include required keys name, cron, period, and code. Do not rename code to script in final generated JSON unless the evaluator explicitly accepts it.",
    },
    {
        "category": "Schema",
        "signal": "schema_missing_required_keys",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["parsed JSON keys", "missing key list"],
        "diagnostic_template": "schema missing required keys: required final keys were omitted.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "required keys and aliases",
        "difficulty": "easy",
        "priority": 7,
        "final_check": "Route exists.",
        "micro_rule": "Validate final JSON keys before returning. The object must contain name, cron, period, and code.",
    },
    {
        "category": "Schema",
        "signal": "schema_invalid_field_type",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["parsed JSON field types"],
        "diagnostic_template": "schema invalid field type: name/cron/code must be strings and period must be scalar.",
        "target_family": "Output_Schema",
        "target_block_id": "03",
        "mutation_policy": "field type rule",
        "difficulty": "easy",
        "priority": 7,
        "final_check": "Route exists.",
        "micro_rule": "Use string values for name, cron, and code, and use an integer/scalar value for period. Do not use arrays or objects for these fields.",
    },
    {
        "category": "Empty",
        "signal": "missing_generated_code",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["generated_json.code", "raw_response_path", "generation_state"],
        "diagnostic_template": "empty generated code: candidate has no behavior although GT is non-empty.",
        "target_family": "Intent_Fulfillment",
        "target_block_id": "06",
        "mutation_policy": "valid JSON empty behavior routes to Intent/Skeleton unless raw generation failed",
        "difficulty": "easy",
        "priority": 12,
        "final_check": "Do not route blindly to Output_Schema.",
        "micro_rule": "For a non-empty user command, the code field must contain at least one required JOILang action, condition, or schedule body. Do not return empty code unless the GT behavior is explicitly empty.",
    },
    {
        "category": "Dataset",
        "signal": "missing_official_gt",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["dataset gt column"],
        "diagnostic_template": "dataset issue: official gt is missing; do not mutate prompt based on this row.",
        "target_family": "No_Mutation",
        "target_block_id": "00",
        "mutation_policy": "exclude from prompt mutation",
        "difficulty": "easy",
        "priority": 0,
        "final_check": "Add No_Mutation mapping.",
        "micro_rule": "Do not mutate prompts from rows whose official GT is missing.",
    },
    {
        "category": "Dataset",
        "signal": "missing_gt_code",
        "signal_type": "failure_reason",
        "produced_by": "strict_det",
        "evidence_fields": ["gt_json", "gt_code"],
        "diagnostic_template": "dataset issue: GT code is empty; suppress semantic prompt mutation unless empty behavior is intended.",
        "target_family": "No_Mutation",
        "target_block_id": "00",
        "mutation_policy": "exclude or validate empty behavior",
        "difficulty": "easy",
        "priority": 0,
        "final_check": "Connect with valid_json_empty_behavior_match.",
        "micro_rule": "Do not create semantic/service patches from rows with empty GT code unless the row explicitly tests empty behavior.",
    },
    {
        "category": "Generation",
        "signal": "generation_empty_output",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["raw candidate presence", "output fields", "generation error"],
        "diagnostic_template": "generation failure: no valid raw candidate was produced.",
        "target_family": "Generation_Health",
        "target_block_id": "00",
        "mutation_policy": "short-circuit semantic diagnostics; inspect worker/runtime/prompt length",
        "difficulty": "easy",
        "priority": 1,
        "final_check": "Route exists.",
        "micro_rule": "Fix generation health before semantic prompt mutation. Inspect raw response, worker logs, prompt length, model config, and timeout.",
    },
    {
        "category": "Generation",
        "signal": "generation_runtime_error",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["generation_error_type", "worker error"],
        "diagnostic_template": "runtime failure: generation raised runtime error.",
        "target_family": "Generation_Health",
        "target_block_id": "00",
        "mutation_policy": "runtime fix first",
        "difficulty": "easy",
        "priority": 1,
        "final_check": "Route exists.",
        "micro_rule": "Runtime failures are not JOILang semantic failures. Fix worker/runtime error before adding service or receiver rules.",
    },
    {
        "category": "Generation",
        "signal": "generation_cuda_oom",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["OOM flag", "error text", "prompt tokens", "VRAM"],
        "diagnostic_template": "CUDA OOM: reduce prompt payload/context or runtime memory before semantic rules.",
        "target_family": "Prompt_Budget",
        "target_block_id": "00",
        "mutation_policy": "prompt budget reduction, schema top-k, quantization/runtime config",
        "difficulty": "easy",
        "priority": 1,
        "final_check": "Route exists.",
        "micro_rule": "For CUDA OOM, reduce prompt payload or runtime memory first. Do not add semantic rules that increase prompt length.",
    },
    {
        "category": "Generation",
        "signal": "generation_timeout",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["timeout flag/error", "latency", "worker timeout"],
        "diagnostic_template": "generation timeout: tune timeout/retry/model loading before semantic prompt mutation.",
        "target_family": "Runtime_Health",
        "target_block_id": "00",
        "mutation_policy": "retry/time budget/runtime policy",
        "difficulty": "easy",
        "priority": 1,
        "final_check": "Route exists.",
        "micro_rule": "For timeouts, tune worker timeout, retry policy, or model loading. Do not infer service/receiver semantic failure from timeout rows.",
    },
    {
        "category": "Extraction",
        "signal": "candidate_extraction_failure",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["raw output exists", "extraction failure"],
        "diagnostic_template": "candidate extraction failure: inspect wrapping/fences/prose and extraction regex before semantic changes.",
        "target_family": "Parser_Extraction",
        "target_block_id": "03",
        "mutation_policy": "extractor improvement or JSON-only rule",
        "difficulty": "medium",
        "priority": 8,
        "final_check": "Route exists.",
        "micro_rule": "Ensure the model returns exactly one bare JSON object so the extractor can identify it unambiguously.",
    },
    {
        "category": "Empty valid",
        "signal": "valid_json_empty_behavior_match",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["parsed JSON empty", "GT empty", "DET pass"],
        "diagnostic_template": "no-op match: valid empty JSON was expected because GT behavior is empty.",
        "target_family": "No_Mutation",
        "target_block_id": "00",
        "mutation_policy": "no mutation",
        "difficulty": "easy",
        "priority": 0,
        "final_check": "Route exists.",
        "micro_rule": "Do not mutate prompts for rows where empty behavior is the correct GT behavior.",
    },
    {
        "category": "Empty valid",
        "signal": "valid_json_empty_behavior_failure",
        "signal_type": "generation_state",
        "produced_by": "generation_state",
        "evidence_fields": ["parsed JSON empty", "GT non-empty"],
        "diagnostic_template": "valid JSON but empty behavior: command requires non-empty code.",
        "target_family": "Intent_Fulfillment",
        "target_block_id": "06",
        "mutation_policy": "minimum behavior generation",
        "difficulty": "easy",
        "priority": 12,
        "final_check": "Route exists.",
        "micro_rule": "If the command requests any action, condition, schedule, or notification, generate non-empty code that implements it.",
    },
    # Cloud auxiliary
    {
        "category": "Cloud semantic",
        "signal": "semantic_intent",
        "signal_type": "cloud_dimension",
        "produced_by": "cloud_auxiliary",
        "evidence_fields": ["ls_semantic_intent", "Lang reasoning"],
        "diagnostic_template": "cloud semantic-intent low score: use only as auxiliary explanation; strict DET remains primary.",
        "target_family": "Skeleton",
        "target_block_id": "06",
        "mutation_policy": "priority boost only when aligned with strict DET",
        "difficulty": "medium",
        "priority": 95,
        "final_check": "Cloud is not official metric.",
        "micro_rule": "Use cloud semantic feedback only as auxiliary reasoning when it agrees with strict DET component failures.",
    },
    {
        "category": "Device/service cloud",
        "signal": "device_service",
        "signal_type": "cloud_dimension",
        "produced_by": "cloud_auxiliary",
        "evidence_fields": ["ls_device_service", "Lang reasoning"],
        "diagnostic_template": "cloud device-service low score: selected receiver/service does not align with command target.",
        "target_family": "Service_Mapping",
        "target_block_id": "02",
        "mutation_policy": "combine with strict service/receiver diagnostics only",
        "difficulty": "medium",
        "priority": 80,
        "final_check": "Cloud-only patch is overclaim.",
        "micro_rule": "Use device-service cloud feedback only when strict DET also shows service or receiver mismatch.",
    },
    {
        "category": "Cloud semantic",
        "signal": "gpt_semantic",
        "signal_type": "cloud_dimension",
        "produced_by": "cloud_auxiliary",
        "evidence_fields": ["overall_gpt", "GPT reasoning"],
        "diagnostic_template": "GPT semantic mismatch: auxiliary holistic judgment says output changes user intent.",
        "target_family": "Skeleton",
        "target_block_id": "06",
        "mutation_policy": "strict DET first",
        "difficulty": "medium",
        "priority": 95,
        "final_check": "Advisor prompt must say cloud auxiliary, not official.",
        "micro_rule": "Do not use GPT holistic score as the official metric. Use it only to explain strict DET-backed failures.",
    },
    # Minimality / collapse
    {
        "category": "Minimality",
        "signal": "extraneous",
        "signal_type": "failure_reason",
        "produced_by": "local_report",
        "evidence_fields": ["generated-only services/actions/receivers"],
        "diagnostic_template": "extraneous action: remove generated actions not implied by command or GT.",
        "target_family": "Minimality",
        "target_block_id": "06",
        "mutation_policy": "remove unnecessary read/action/wrapper/state",
        "difficulty": "medium",
        "priority": 70,
        "final_check": "Mapping to Minimality.",
        "micro_rule": "Do not add services, actions, reads, variables, or wrapper logic that are not implied by the current command.",
    },
    {
        "category": "Collapse",
        "signal": "output_collapse",
        "signal_type": "postprocess",
        "produced_by": "row_advisor_mapping",
        "evidence_fields": ["generated_code grouping", "row_no", "command diversity"],
        "diagnostic_template": "output collapse: same generated code reused across unrelated rows.",
        "target_family": "Skeleton",
        "target_block_id": "06",
        "mutation_policy": "row independence and command-specific receiver/service selection",
        "difficulty": "medium",
        "priority": 3,
        "final_check": "Not native DET reason; generated by postprocess.",
        "micro_rule": "Solve each row independently from the current command. Never reuse a previous row's JSON name, receiver, service, enum, or code skeleton.",
    },
]

taxonomy_df = pd.DataFrame(TAXONOMY_ROWS)
FAILURE_ADVISOR_MAP = {r["signal"]: r for r in TAXONOMY_ROWS}
DEFAULT_TAXONOMY_ROW = {
    "category": "Unknown",
    "signal": "unknown",
    "signal_type": "fallback",
    "produced_by": "row_advisor_mapping",
    "evidence_fields": ["gt_code", "generated_code"],
    "diagnostic_template": "unclassified failure: compare GT and generated output component-by-component.",
    "target_family": "DET_Helper",
    "target_block_id": "06",
    "mutation_policy": "fallback only",
    "difficulty": "unknown",
    "priority": 999,
    "final_check": "Needs manual review.",
    "micro_rule": "Compare GT and generated output component-by-component and repair the most concrete mismatch first.",
}

GENERATION_SHORT_CIRCUIT_STATES = {
    "generation_empty_output",
    "generation_runtime_error",
    "generation_cuda_oom",
    "generation_timeout",
    "candidate_extraction_failure",
}

NO_MUTATION_STATES = {
    "valid_json_empty_behavior_match",
    "missing_official_gt",
    "missing_gt_code",
}

UMBRELLA_SIGNALS = {
    "gt_mismatch",
    "semantic",
    "semantic_intent",
    "gpt_semantic",
}

print("taxonomy rows:", len(taxonomy_df))
display(taxonomy_df[[
    "category", "signal", "signal_type", "target_family", "target_block_id",
    "mutation_policy", "difficulty", "final_check"
]])

taxonomy rows: 36


,category,signal,signal_type,target_family,target_block_id,mutation_policy,difficulty,final_check
0,Schedule,cron_mismatch,failure_reason,Temporal_Rule,06,fixed wall-clock/day schedule uses cron first; do not replace with period + Clock guard,easy,Added explicit mapping; no DET_Helper fallback.
1,Schedule,period_mismatch,failure_reason,Temporal_Rule,06,"classify one-shot / cron / repeated loop first, then apply period policy",easy,Added explicit mapping; no DET_Helper fallback.
2,Overall,gt_mismatch,failure_reason,DET_Helper,06,umbrella only; do not make primary patch if concrete service/receiver/schedule/numeric/enum reason exists,medium,Keep as fallback or support reason only.
3,Service,gt_service_coverage,failure_reason,Service_Mapping,02,"select command target receiver first, then select schema-valid service under that receiver",easy,High automation value.
4,Service,unknown_service,failure_reason,Service_Mapping,02,replace non-schema member with nearest valid canonical schema member before final output,easy,Already maps to Service_Mapping; expand with camelCase/class-style prohibition.
5,Receiver,gt_receiver_coverage,failure_reason,Receiver_Tag_Preservation,02,owner/location/group/sector tag preservation; condition receiver and action receiver may differ,easy,Mapping is appropriate.
6,Numeric,numeric_grounding,failure_reason,Numeric_Unit_Grounding,06,temporal numbers and service argument numbers both use descriptor-grounded conversion,easy,Separate from pure Temporal_Rule when numeric is a service argument.
7,Enum/String,enum_grounding,failure_reason,Enum_Grounding,02,copy allowed enum from selected service descriptor only,easy,Mapping is appropriate.
8,Argument,arg_type,failure_reason,Argument_Grounding,02,type/order/separator/format rule,medium,Argument shape must be descriptor-grounded.
9,Dataflow,dataflow,failure_reason,Dataflow,06,sensor read → variable bind → downstream speak/action structure preservation,medium,JOILang ':=' variable pattern should be included in diagnostic logic.


## 17. Row-level diagnostic and mapping functions

이 cell은 `failure_reasons + diff_summary + gt_json + generated_json + generation_state + optional cloud auxiliary`를 row-level advisor evidence로 변환하는 함수를 정의합니다.

핵심 구현:

- `missing_required_key:<keys>` normalize
- `period_mismatch`, `cron_mismatch` 상세 값 비교
- service/receiver/numeric/enum mismatch 상세 비교
- camelCase/class-style service hallucination 감지
- repeated output collapse 감지
- generation-state branch short-circuit
- umbrella reason은 concrete reason 뒤로 priority 조정

In [67]:
# ============================================================
# Cell 17. Row-level diagnostic and mapping functions
# ============================================================

def safe_json_loads(value, default):
    """
    Robust JSON loader for row_evaluation/advisor cells.

    Important fix:
    pandas NaN must be treated as empty before it becomes the string "nan".
    Also, list/dict values must be returned before pd.isna(value), because
    pd.isna(list_like) can produce ambiguous truth-value warnings.
    """
    if value is None:
        return default

    if isinstance(value, (dict, list, tuple)):
        return value

    try:
        if pd.isna(value):
            return default
    except Exception:
        pass

    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return default

    try:
        return json.loads(text)
    except Exception:
        return default

def as_list(value):
    obj = safe_json_loads(value, [])
    if isinstance(obj, list):
        return [str(x) for x in obj if str(x).strip()]
    if isinstance(obj, str) and obj.strip():
        # tolerate comma-delimited fallback
        if obj.strip().startswith("["):
            return []
        return [x.strip() for x in obj.split(",") if x.strip()]
    return []

def as_dict(value):
    obj = safe_json_loads(value, {})
    return obj if isinstance(obj, dict) else {}

def normalize_code_text(value):
    """
    Normalize JOILang code text while treating NaN/None/null as empty.

    This prevents generated_code=NaN from being interpreted as a non-empty
    string and prevents generation_error_type=NaN from cascading into a
    false runtime failure classification.
    """
    if value is None:
        return ""

    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False)

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    text = str(value).replace("\\n", "\n").replace("\\t", "    ").strip()
    if text.lower() in {"nan", "none", "null"}:
        return ""
    return text

def row_gt_json(row):
    obj = as_dict(row.get("gt_json", ""))
    if obj:
        return obj
    return as_dict(row.get("gt", ""))

def row_generated_json(row):
    obj = as_dict(row.get("generated_json", ""))
    if obj:
        return obj
    candidates = safe_json_loads(row.get("candidates", ""), [])
    if isinstance(candidates, list) and candidates and isinstance(candidates[0], dict):
        return candidates[0]
    return {}

def json_code(obj):
    if not isinstance(obj, dict):
        return ""
    return normalize_code_text(obj.get("code", obj.get("script", "")))

def row_gt_code(row):
    code = normalize_code_text(row.get("gt_code", ""))
    return code or json_code(row_gt_json(row))

def row_generated_code(row):
    code = normalize_code_text(row.get("generated_code", ""))
    return code or json_code(row_generated_json(row))

def row_diff_summary(row):
    return as_dict(row.get("diff_summary", ""))

def get_period(obj):
    return obj.get("period", "") if isinstance(obj, dict) else ""

def get_cron(obj):
    return obj.get("cron", "") if isinstance(obj, dict) else ""

def compact_list(xs, max_items=10):
    if xs is None:
        return []
    xs = list(xs) if isinstance(xs, (list, tuple, set)) else [xs]
    xs = [str(x) for x in xs if str(x).strip()]
    if len(xs) <= max_items:
        return xs
    return xs[:max_items] + [f"...(+{len(xs) - max_items})"]

def unique_keep_order(items):
    out = []
    seen = set()
    for item in items:
        key = json.dumps(item, ensure_ascii=False, sort_keys=True) if isinstance(item, (dict, list)) else str(item)
        if key not in seen:
            out.append(item)
            seen.add(key)
    return out

def normalize_signal(reason):
    token = str(reason or "").strip()
    if not token:
        return ""
    if token.startswith("missing_required_key"):
        return "missing_required_key"
    if token in {"malformed_json", "invalid_json.malformed", "invalid_json.malformed_json"}:
        return "invalid_json.malformed_json"
    if token in {"non_json_text", "invalid_json.non_json"}:
        return "invalid_json.non_json_text"
    if token in {"markdown_fence", "invalid_json.fence"}:
        return "invalid_json.markdown_fence"
    return token.split(":", 1)[0]

def taxonomy_for_signal(signal):
    signal = normalize_signal(signal)
    return FAILURE_ADVISOR_MAP.get(signal, DEFAULT_TAXONOMY_ROW)

def is_probably_camel_or_class_style(member):
    text = str(member or "")
    return bool(re.search(r"[a-z][A-Z]|[A-Z][a-z]", text))

def row_failure_reasons(row):
    return as_list(row.get("failure_reasons", ""))


def _clean_error_type(value):
    """
    Return a meaningful generation error type, or an empty string.

    Critical policy:
    NaN, "nan", None, "none", and "null" are not generation errors.
    """
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    text = str(value).strip()
    if text.lower() in {"", "nan", "none", "null"}:
        return ""
    return text

def infer_generation_state(row):
    """
    Best-effort state inference for row_evaluation.csv-only workflow.

    Critical repair:
    - generated_code or generated_json.code/script being non-empty means the
      model produced a candidate, so the row must not be short-circuited as
      generation_runtime_error.
    - generation_error_type=NaN is not an error.
    - only a meaningful non-empty error_type with no usable code becomes a
      generation health state.
    """
    existing = as_dict(row.get("generation_state", ""))
    if existing.get("class"):
        return existing

    error_type = _clean_error_type(row.get("generation_error_type", ""))
    error_count = pd.to_numeric(row.get("generation_error_count", 0), errors="coerce")
    if pd.isna(error_count):
        error_count = 0

    gt = row_gt_json(row)
    gen = row_generated_json(row)
    gen_code = row_generated_code(row)
    gt_code = row_gt_code(row)
    raw_exists = bool(normalize_code_text(row.get("raw_response_path", "")))

    # Most important branch:
    # a non-empty candidate/code is a semantic/schema DET case, not a runtime failure.
    if gen_code:
        klass = "valid_json_nonempty"
    elif not gen and error_type:
        lowered = error_type.lower()
        if "oom" in lowered:
            klass = "generation_cuda_oom"
        elif "timeout" in lowered:
            klass = "generation_timeout"
        elif "extraction" in lowered:
            klass = "candidate_extraction_failure"
        elif "invalid_json" in lowered:
            klass = "invalid_json"
        elif "worker_crash" in lowered or "runtime" in lowered or "crash" in lowered:
            klass = "generation_runtime_error"
        else:
            klass = "generation_runtime_error"
    elif not gen:
        klass = "generation_empty_output"
    elif not gen_code and gt_code:
        klass = "valid_json_empty_behavior_failure"
    elif not gen_code and not gt_code:
        klass = "valid_json_empty_behavior_match"
    else:
        klass = "valid_json_nonempty"

    return {
        "class": klass,
        "generation_error_type": error_type,
        "generation_error_count": int(error_count),
        "raw_response_path": str(row.get("raw_response_path", "") or ""),
        "raw_response_recorded": raw_exists,
        "gt_code_empty": not bool(gt_code),
        "generated_code_empty": not bool(gen_code),
    }

def service_diagnostic(diff):
    gt_services = compact_list(diff.get("gt_services", []))
    gen_services = compact_list(diff.get("generated_services", []))
    msg = f"service mismatch: gt services={gt_services}, generated services={gen_services}"
    notes = []
    for svc in gen_services:
        if is_probably_camel_or_class_style(svc):
            notes.append(f"generated service {svc!r} looks camelCase/class-style; copy schema canonical lowercase_underscore member")
    if notes:
        msg += "; " + "; ".join(notes)
    return msg

def receiver_diagnostic(diff):
    return (
        f"receiver mismatch: gt receivers={compact_list(diff.get('gt_receivers', []))}, "
        f"generated receivers={compact_list(diff.get('generated_receivers', []))}"
    )

def numeric_diagnostic(diff):
    return (
        f"numeric mismatch: gt numeric literals={compact_list(diff.get('gt_numeric_literals', []))}, "
        f"generated numeric literals={compact_list(diff.get('generated_numeric_literals', []))}"
    )

def enum_diagnostic(diff):
    return (
        f"enum/string mismatch: gt string args={compact_list(diff.get('gt_string_args', []))}, "
        f"generated string args={compact_list(diff.get('generated_string_args', []))}"
    )

def build_diagnostic_for_signal(signal, row):
    signal = normalize_signal(signal)
    gt = row_gt_json(row)
    gen = row_generated_json(row)
    diff = row_diff_summary(row)

    if signal == "period_mismatch":
        return f"period mismatch: gt period={get_period(gt)!r}, generated period={get_period(gen)!r}"
    if signal == "cron_mismatch":
        return f"cron mismatch: gt cron={get_cron(gt)!r}, generated cron={get_cron(gen)!r}"
    if signal in {"gt_service_coverage", "unknown_service", "service_match", "device_service"}:
        return service_diagnostic(diff)
    if signal in {"gt_receiver_coverage", "receiver_invalid"}:
        return receiver_diagnostic(diff)
    if signal == "numeric_grounding":
        return numeric_diagnostic(diff)
    if signal == "enum_grounding":
        return enum_diagnostic(diff)
    if signal == "arg_type":
        return "argument type mismatch: compare selected service descriptor with generated argument order, type, separator, and bounds"
    if signal == "dataflow":
        return "dataflow mismatch: compare JOILang ':=' variable binding, sensor read, and downstream use"
    if signal in {"precondition", "conditions"}:
        return "condition/precondition mismatch: generated condition does not preserve command trigger or required state guard"
    if signal in {"semantic", "semantic_intent", "gpt_semantic"}:
        return "semantic intent mismatch: use only as auxiliary unless backed by strict DET component mismatch"
    if signal == "gt_mismatch":
        return (
            f"overall GT mismatch: gt_similarity={row.get('gt_similarity', '')}, "
            f"code_match={row.get('code_match', '')}; prioritize concrete component diagnostics"
        )
    if signal == "missing_generated_code":
        return (
            f"empty generated code: gt code length={len(row_gt_code(row))}, "
            f"generated code length={len(row_generated_code(row))}"
        )
    if signal == "missing_required_key":
        return f"missing required JSON key: {str(row.get('failure_reasons', ''))}"
    if signal == "invalid_json":
        return "invalid JSON: generated output could not be parsed as the required single JSON object"
    if signal.startswith("invalid_json."):
        return taxonomy_for_signal(signal)["diagnostic_template"]
    if signal.startswith("schema_"):
        return taxonomy_for_signal(signal)["diagnostic_template"]
    if signal in GENERATION_SHORT_CIRCUIT_STATES:
        return taxonomy_for_signal(signal)["diagnostic_template"]
    if signal in NO_MUTATION_STATES:
        return taxonomy_for_signal(signal)["diagnostic_template"]
    if signal == "extraneous":
        return "extraneous action: generated output contains actions/services not implied by command or GT"

    return f"{signal}: no specialized diagnostic rule; compare gt_code and generated_code"

def row_cloud_auxiliary(row):
    """
    Collect optional cloud judge fields if merged into row_evaluation.csv.
    Missing/skipped/error cloud evidence must not be interpreted as semantic failure.
    """
    out = {}
    for c in row.index:
        lc = str(c).lower()
        if lc.startswith("ls_") or lc.startswith("cloud_") or lc in {"overall_gpt", "gpt_reasoning", "lang_reasoning"}:
            val = row.get(c, "")
            try:
                if pd.isna(val):
                    continue
            except Exception:
                pass
            if str(val).strip():
                out[c] = val
    return out

def collapse_key_for_row(row):
    gen_code = row_generated_code(row)
    if gen_code:
        text = gen_code
    else:
        gen = row_generated_json(row)
        text = json.dumps(gen, ensure_ascii=False, sort_keys=True) if gen else ""
    return re.sub(r"\s+", "", text.strip().lower())

def build_collapse_counts(df):
    keys = [collapse_key_for_row(row) for _, row in df.iterrows()]
    return Counter([k for k in keys if k])

def branch_for_row(signals, generation_state):
    klass = generation_state.get("class", "")
    if klass in GENERATION_SHORT_CIRCUIT_STATES:
        return "generation_health_short_circuit"
    if klass in {"invalid_json", "invalid_json.non_json_text", "invalid_json.markdown_fence", "invalid_json.malformed_json", "truncated_json", "schema_missing_required_keys", "schema_invalid_field_type"}:
        return "output_schema"
    if klass in NO_MUTATION_STATES:
        return "no_mutation"
    if klass == "valid_json_empty_behavior_failure" or "missing_generated_code" in signals:
        return "intent_fulfillment"
    if any(s not in UMBRELLA_SIGNALS for s in signals):
        return "strict_det_concrete_mismatch"
    if any(s in UMBRELLA_SIGNALS for s in signals):
        return "umbrella_semantic_fallback"
    return "no_failure_signal"

def sort_signals_for_patch(signals):
    def key(sig):
        row = taxonomy_for_signal(sig)
        # umbrella goes last unless no other signal
        umb = 1 if sig in UMBRELLA_SIGNALS else 0
        return (umb, int(row.get("priority", 999)))
    return sorted(unique_keep_order(signals), key=key)

def record_recommended_mutation(signal, row):
    tax = taxonomy_for_signal(signal)
    return {
        "source_signal": signal,
        "target_block_id": tax["target_block_id"],
        "target_block_family": tax["target_family"],
        "suggested_mutation_type": tax["mutation_policy"],
        "micro_rule": tax["micro_rule"],
        "verification": tax["final_check"],
    }

## 18. Build row-level advisor feedback records

이 cell은 `row_evaluation.csv`의 각 실패 row를 advisor가 바로 쓸 수 있는 evidence row로 변환합니다.

산출 컬럼:

- `concrete_diagnostics`
- `recommended_prompt_mutations`
- `advisor_families`
- `target_block_ids`
- `branch`
- `severity_score`
- `generated_collapse_count`

In [68]:
# ============================================================
# Cell 18. Build row-level advisor feedback records
# ============================================================

def severity_score_for_row(row, signals, generation_state, collapse_count):
    det_score = pd.to_numeric(row.get("det_score", 0), errors="coerce")
    if pd.isna(det_score):
        det_score = 0.0

    score = max(0.0, 100.0 - float(det_score))
    score += min(len(signals), 10) * 2.0

    branch = branch_for_row(signals, generation_state)
    if branch == "generation_health_short_circuit":
        score += 30.0
    if "unknown_service" in signals:
        score += 10.0
    if "gt_receiver_coverage" in signals:
        score += 10.0
    if "missing_generated_code" in signals:
        score += 20.0
    if collapse_count >= 2:
        score += min(collapse_count, 20) * 3.0

    return round(score, 4)

def build_row_advisor_record(row, collapse_counts):
    raw_reasons = row_failure_reasons(row)
    signals = [normalize_signal(r) for r in raw_reasons if normalize_signal(r)]

    generation_state = infer_generation_state(row)
    gen_state_class = generation_state.get("class", "")
    has_generated_code = bool(row_generated_code(row))

    # Only add generation-health state as a signal when there is no usable generated code.
    # A row with generated_code/generated_json.code is a strict DET semantic/schema mismatch,
    # even if generation_error_type is NaN or a stale metadata field.
    if gen_state_class and gen_state_class not in {"valid_json_nonempty"} and not has_generated_code:
        signals.append(normalize_signal(gen_state_class))

    # Add output collapse postprocess signal.
    ckey = collapse_key_for_row(row)
    collapse_count = collapse_counts.get(ckey, 0) if ckey else 0
    if collapse_count >= 2:
        signals.append("output_collapse")

    signals = sort_signals_for_patch(signals)
    branch = branch_for_row(signals, generation_state)

    # Short-circuit: generation health rows should not get semantic/service patches
    # only when no generated code exists. If generated code exists, keep concrete
    # strict DET signals such as unknown_service, period_mismatch, receiver mismatch, etc.
    if branch == "generation_health_short_circuit" and not has_generated_code:
        allowed = set(GENERATION_SHORT_CIRCUIT_STATES) | {"output_collapse"}
        signals = [s for s in signals if s in allowed] or [gen_state_class]
    elif branch == "generation_health_short_circuit" and has_generated_code:
        branch = "strict_det_concrete_mismatch"

    # No mutation branch excludes all patchable rules.
    if branch == "no_mutation":
        signals = [s for s in signals if taxonomy_for_signal(s)["target_family"] == "No_Mutation"] or [gen_state_class]

    diagnostics = []
    for sig in signals:
        diagnostics.append(build_diagnostic_for_signal(sig, row))

    if collapse_count >= 2 and "output_collapse" not in signals:
        diagnostics.append(
            f"output collapse: same generated output appears in {collapse_count} rows; enforce row independence"
        )

    mutations = [record_recommended_mutation(sig, row) for sig in signals]
    mutations = [m for m in mutations if m["target_block_family"] != "No_Mutation"]

    if has_generated_code:
        # Defensive cleanup: stale generation-health signals must not dominate rows
        # that produced actual JOILang code.
        mutations = [
            m for m in mutations
            if m["target_block_family"] not in {"Generation_Health", "Runtime_Health", "Prompt_Budget", "Parser_Extraction"}
            and m["target_block_id"] != "00"
        ]

    families = unique_keep_order([m["target_block_family"] for m in mutations])
    target_block_ids = unique_keep_order([m["target_block_id"] for m in mutations])

    gt = row_gt_json(row)
    gen = row_generated_json(row)
    diff = row_diff_summary(row)
    cloud_aux = row_cloud_auxiliary(row)

    record = {
        "row_no": row.get("row_no", ""),
        "category": row.get("category", ""),
        "command_eng": row.get("command_eng", ""),
        "command_kor": row.get("command_kor", ""),
        "det_score": row.get("det_score", ""),
        "det_pass": row.get("det_pass", ""),
        "gt_exact": row.get("gt_exact", ""),
        "gt_similarity": row.get("gt_similarity", ""),
        "branch": branch,
        "generation_state_class": generation_state.get("class", ""),
        "generation_state": json.dumps(generation_state, ensure_ascii=False),

        "failure_reasons": json.dumps(raw_reasons, ensure_ascii=False),
        "normalized_signals": json.dumps(signals, ensure_ascii=False),
        "primary_advisor_family": families[0] if families else taxonomy_for_signal(signals[0])["target_family"] if signals else "No_Mutation",
        "advisor_families": json.dumps(families, ensure_ascii=False),
        "target_block_ids": json.dumps(target_block_ids, ensure_ascii=False),
        "concrete_diagnostics": json.dumps(unique_keep_order(diagnostics), ensure_ascii=False),
        "recommended_prompt_mutations": json.dumps(mutations, ensure_ascii=False),

        "severity_score": severity_score_for_row(row, signals, generation_state, collapse_count),
        "generated_collapse_count": collapse_count,

        "gt_period": get_period(gt),
        "generated_period": get_period(gen),
        "gt_cron": get_cron(gt),
        "generated_cron": get_cron(gen),

        "gt_services": json.dumps(diff.get("gt_services", []), ensure_ascii=False),
        "generated_services": json.dumps(diff.get("generated_services", []), ensure_ascii=False),
        "gt_receivers": json.dumps(diff.get("gt_receivers", []), ensure_ascii=False),
        "generated_receivers": json.dumps(diff.get("generated_receivers", []), ensure_ascii=False),
        "gt_numeric_literals": json.dumps(diff.get("gt_numeric_literals", []), ensure_ascii=False),
        "generated_numeric_literals": json.dumps(diff.get("generated_numeric_literals", []), ensure_ascii=False),
        "gt_string_args": json.dumps(diff.get("gt_string_args", []), ensure_ascii=False),
        "generated_string_args": json.dumps(diff.get("generated_string_args", []), ensure_ascii=False),

        "gt_code": row_gt_code(row),
        "generated_code": row_generated_code(row),
        "gt_json": json.dumps(gt, ensure_ascii=False),
        "generated_json": json.dumps(gen, ensure_ascii=False),
        "diff_summary": json.dumps(diff, ensure_ascii=False),
        "cloud_auxiliary": json.dumps(cloud_aux, ensure_ascii=False),

        "raw_response_path": row.get("raw_response_path", ""),
        "prompt_log_paths": row.get("prompt_log_paths", ""),
    }
    return record

In [69]:
row_eval_df[row_eval_df['row_no']==1]

,row_no,category,genome_id,candidate_index,det_score,det_pass,gt_exact,gt_similarity,schedule_match,cron_match,period_match,code_match,service_valid,receiver_valid,gt_service_coverage,gt_service_precision,gt_receiver_coverage,dataflow_score,numeric_grounding,enum_grounding,failure_reasons,diff_summary,gt_code,generated_code,gt_json,generated_json,generation,candidate_strategy,generation_error_type,generation_error_count,generation_prompt_tokens_total,generation_completion_tokens_total,generation_total_tokens_total,latency_sec,peak_vram_gb,raw_response_path,prompt_log_paths,service_schema_path,service_context_mode,service_context_source,service_list_retrieval_scores,repair_applied,repair_actions,backend,command_eng,command_kor,gt
0,1,1,suite_base,0,65.1471,False,False,0.921569,False,True,False,False,False,True,0.0,0.0,1.0,1.0,1.0,1.0,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service""]","{""gt_services"":[""dishwasher_setdishwashermode""],""generated_services"":[""dishwasherMode_setDishwasherMode""],""gt_receivers"":[""#Dishwasher""],""generated_receivers"":[""#Dishwasher""],""gt_numeric_literals"":[],""generated_numeric_literals"":[],""gt_string_args"":[""dry""],""generated_string_args"":[""dry""]}","(#Dishwasher).dishwasher_setdishwashermode(""dry"")","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")","{""name"":"""",""cron"":"""",""period"":0,""script"":""(#Dishwasher).dishwasher_setdishwashermode(\""dry\"")""}","{""name"":""DishwasherDryMode"",""cron"":"""",""period"":-1,""code"":""(#Dishwasher).dishwasherMode_setDishwasherMode(\""dry\"")""}",0,worker_direct,NaN,0,53129,52,0,26.703795,24.1048,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_responses/row_1_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/prompts/row_1_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudl...",datasets/service_list_ver2.0.1.json,schema_fallback,provided_schema,"{""status"": ""retrieval_disabled"", ""reason"": ""canonical ga_search has not enabled service retrieval yet""}",False,[],worker,Switch the dishwasher to dry mode.,식기세척기를 건조 모드로 설정해줘.,"{""name"": """", \n""cron"": """", \n""period"": 0, \n""script"": ""(#Dishwasher).dishwasher_setdishwashermode(\""dry\"")""}"


In [70]:
collapse_counts = build_collapse_counts(row_eval_df)

row_advisor_records = []
for _, row in row_eval_df.iterrows():
    reasons = row_failure_reasons(row)
    det_pass = str(row.get("det_pass", "")).strip().lower() == "true"
    gen_state = infer_generation_state(row)

    # Keep failed rows, generation issue rows, and rows with explicit reasons.
    if det_pass and not reasons and gen_state.get("class") in {"valid_json_nonempty", "valid_json_empty_behavior_match"}:
        continue

    row_advisor_records.append(build_row_advisor_record(row, collapse_counts))

row_advisor_df = pd.DataFrame(row_advisor_records)
if not row_advisor_df.empty:
    row_advisor_df = row_advisor_df.sort_values(
        by=["severity_score", "row_no"],
        ascending=[False, True],
    ).reset_index(drop=True)

print("row_advisor_records:", len(row_advisor_df))
display_cols = [c for c in [
    "row_no", "category", "det_score", "branch", "generation_state_class",
    "primary_advisor_family", "target_block_ids", "failure_reasons",
    "concrete_diagnostics", "severity_score", "generated_collapse_count"
] if c in row_advisor_df.columns]
display(row_advisor_df[display_cols].head(50))

# Coverage verification: every observed normalized signal should have taxonomy.
observed_signals = Counter()
for s in row_advisor_df.get("normalized_signals", []):
    for sig in as_list(s):
        observed_signals[sig] += 1
taxonomy_coverage_rows = []
for sig, count in observed_signals.most_common():
    tax = taxonomy_for_signal(sig)
    taxonomy_coverage_rows.append({
        "signal": sig,
        "count": count,
        "mapped": tax is not DEFAULT_TAXONOMY_ROW,
        "target_family": tax["target_family"],
        "target_block_id": tax["target_block_id"],
        "final_check": tax["final_check"],
    })
taxonomy_coverage_df = pd.DataFrame(taxonomy_coverage_rows)
print("taxonomy coverage")
display(taxonomy_coverage_df)

row_advisor_records: 280


,row_no,category,det_score,branch,generation_state_class,primary_advisor_family,target_block_ids,failure_reasons,concrete_diagnostics,severity_score,generated_collapse_count
0,218,7,10.0350,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding"", ""enum_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['door_doorstate', 'levelcontrol_movetolevel'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setDishwasherMode' looks camelCase/cla...",187.9650,280
1,222,7,11.9512,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding"", ""enum_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['armrobot_currentposition', 'speaker_speak', 'windowcovering_uporopen'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setDishwashe...",186.0488,280
2,195,7,12.4534,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding"", ""enum_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['menuprovider_getmenu', 'speaker_speak'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setDishwasherMode' looks camelCase/class-st...",185.5466,280
3,186,7,14.1463,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding"", ""enum_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['clock_hour', 'robotvacuumcleaner_setrobotvacuumcleanermodemode'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setDishwasherMode'...",183.8537,280
4,187,7,15.0000,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding"", ""enum_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['clock_hour', 'siren_setsirenmode'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setDishwasherMode' looks camelCase/class-style; ...",183.0000,280
5,251,8,13.6620,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['clock_hour', 'door_close', 'light_movetobrightness', 'lightsensor_brightness'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setD...",182.3380,280
6,211,7,13.8554,strict_det_concrete_mismatch,valid_json_nonempty,Skeleton,"[""06"", ""02""]","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated

taxonomy coverage


,signal,count,mapped,target_family,target_block_id,final_check
0,output_collapse,280,True,Skeleton,06,Not native DET reason; generated by postprocess.
1,unknown_service,280,True,Service_Mapping,02,Already maps to Service_Mapping; expand with camelCase/class-style prohibition.
2,gt_service_coverage,280,True,Service_Mapping,02,High automation value.
3,period_mismatch,280,True,Temporal_Rule,06,Added explicit mapping; no DET_Helper fallback.
4,gt_mismatch,280,True,DET_Helper,06,Keep as fallback or support reason only.
5,gt_receiver_coverage,279,True,Receiver_Tag_Preservation,02,Mapping is appropriate.
6,numeric_grounding,161,True,Numeric_Unit_Grounding,06,Separate from pure Temporal_Rule when numeric is a service argument.
7,enum_grounding,161,True,Enum_Grounding,02,Mapping is appropriate.
8,cron_mismatch,40,True,Temporal_Rule,06,Added explicit mapping; no DET_Helper fallback.


In [47]:
row_advisor_df[row_advisor_df['row_no']==251]

,row_no,category,command_eng,command_kor,det_score,det_pass,gt_exact,gt_similarity,branch,generation_state_class,generation_state,failure_reasons,normalized_signals,primary_advisor_family,advisor_families,target_block_ids,concrete_diagnostics,recommended_prompt_mutations,severity_score,generated_collapse_count,gt_period,generated_period,gt_cron,generated_cron,gt_services,generated_services,gt_receivers,generated_receivers,gt_numeric_literals,generated_numeric_literals,gt_string_args,generated_string_args,gt_code,generated_code,gt_json,generated_json,diff_summary,cloud_auxiliary,raw_response_path,prompt_log_paths
3,251,8,"At midnight, close the door and check the light every hour until 6 AM; if the brightness is greater than 30, lower it to 10.","자정이 되면 문을 닫고, 오전 6시까지 한 시간마다 조명을 체크해서 밝기가 30보다 크면 10으로 낮춰줘.",13.662,False,False,0.122066,generation_health_short_circuit,generation_runtime_error,"{""class"": ""generation_runtime_error"", ""generation_error_type"": ""nan"", ""raw_response_path"": ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_responses/row_251_cand...","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding""]","[""generation_runtime_error"", ""output_collapse""]",Generation_Health,"[""Generation_Health"", ""Skeleton""]","[""00"", ""06""]","[""runtime failure: generation raised runtime error."", ""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code""]","[{""source_signal"": ""generation_runtime_error"", ""target_block_id"": ""00"", ""target_block_family"": ""Generation_Health"", ""suggested_mutation_type"": ""runtime fix first"", ""micro_rule"": ""Runtime failures are not JOILang semantic failures. Fix worker/runtime error before adding service or receiver rules....",180.338,280,3600000,-1,0 0 * * *,,"[""clock_hour"", ""door_close"", ""light_movetobrightness"", ""lightsensor_brightness""]","[""dishwasherMode_setDishwasherMode""]","[""#Door"", ""#Light"", ""(#Clock"", ""(#Light""]","[""#Dishwasher""]","[""0"", ""0"", ""1"", ""6"", ""30"", ""10""]",[],[],"[""dry""]",active := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\n},"(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")","{""name"": """", ""cron"": ""0 0 * * *"", ""period"": 3600000, ""script"": ""\nactive := 0\n\nif (active == 0) {\n\n (#Door).door_close()\n\n active = 1\n\n}\n\nif ((#Clock).clock_hour == 6) {\n\n break\n\n}\n\nif ((#Light).lightsensor_brightness > 30) {\n\n (#Light).light_movetobrightness(10)\n\...","{""name"": ""DishwasherDryMode"", ""cron"": """", ""period"": -1, ""code"": ""(#Dishwasher).dishwasherMode_setDishwasherMode(\""dry\"")""}","{""gt_services"": [""clock_hour"", ""door_close"", ""light_movetobrightness"", ""lightsensor_brightness""], ""generated_services"": [""dishwasherMode_setDishwasherMode""], ""gt_receivers"": [""#Door"", ""#Light"", ""(#Clock"", ""(#Light""], ""generated_receivers"": [""#Dishwasher""], ""gt_numeric_literals"": [""0"", ""0"", ""1"", ...",{},/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_responses/row_251_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/prompts/row_251_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/clou..."


In [71]:
import json
import ast
import pandas as pd
from itertools import zip_longest
from IPython.display import HTML, display

def render_advisor_report(df, target_row_no):
    """
    DataFrame에서 특정 row_no를 찾아 가독성 높은 HTML 리포트 형태로 출력합니다.
    """
    # 1. 데이터 추출
    row_data = df[df['row_no'] == target_row_no]
    if row_data.empty:
        print(f"⚠️ Row {target_row_no}를 찾을 수 없습니다.")
        return
    row = row_data.iloc[0]

    # 2. 안전한 파싱 (문자열화된 리스트나 JSON 복원)
    def safe_parse(val, is_json=False):
        if pd.isna(val) or val == "":
            return {} if is_json else []
        if isinstance(val, (dict, list)):
            return val
        try:
            return json.loads(val)
        except:
            try:
                return ast.literal_eval(val)
            except:
                return val

    gt_json_obj = safe_parse(row.get('gt_json', '{}'), is_json=True)
    gen_json_obj = safe_parse(row.get('generated_json', '{}'), is_json=True)
    
    # 예쁘게 들여쓰기된 JSON 문자열 생성
    gt_json_str = json.dumps(gt_json_obj, indent=2, ensure_ascii=False) if isinstance(gt_json_obj, dict) else str(gt_json_obj)
    gen_json_str = json.dumps(gen_json_obj, indent=2, ensure_ascii=False) if isinstance(gen_json_obj, dict) else str(gen_json_obj)

    failures = safe_parse(row.get('failure_reasons', '[]'))
    diagnostics = safe_parse(row.get('concrete_diagnostics', '[]'))
    mutations = safe_parse(row.get('recommended_prompt_mutations', '[]'), is_json=True)

    # 문자열로 들어온 단일 요소가 리스트가 아닌 경우 리스트로 캐스팅
    if not isinstance(failures, list): failures = [failures]
    if not isinstance(diagnostics, list): diagnostics = [diagnostics]
    if not isinstance(mutations, list): mutations = [mutations]

    # 3. HTML Layout 생성
    html_content = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 100%; margin: auto; border: 1px solid #d1d5db; border-radius: 8px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); overflow: hidden;">
        
        <div style="background-color: #1f2937; color: white; padding: 16px 20px; display: flex; justify-content: space-between; align-items: center;">
            <h2 style="margin: 0; font-size: 1.25rem;">Row <span style="color: #60a5fa;">#{target_row_no}</span> Analysis Report</h2>
            <div style="font-size: 0.9rem; background: rgba(255,255,255,0.1); padding: 4px 10px; border-radius: 4px;">
                <strong>DET Score:</strong> {row.get('det_score', 0)} &nbsp;|&nbsp; <strong>Severity:</strong> {row.get('severity_score', 0)}
            </div>
        </div>

        <div style="padding: 16px 20px; background-color: #f3f4f6; border-bottom: 1px solid #d1d5db;">
            <p style="margin: 0 0 8px 0; font-size: 15px; color: black;"><strong>🇬🇧 ENG:</strong> {row.get('command_eng', 'N/A')}</p>
            <p style="margin: 0; font-size: 15px; color: black;"><strong>🇰🇷 KOR:</strong> {row.get('command_kor', 'N/A')}</p>
        </div>

        <div style="display: flex; flex-wrap: wrap; border-bottom: 1px solid #d1d5db;">
            <div style="flex: 1; min-width: 300px; padding: 16px; border-right: 1px solid #d1d5db; background: #f8fafc;">
                <h4 style="margin: 0 0 10px 0; color: #059669; font-size: 15px;">✅ Ground Truth JSON</h4>
                <pre style="background: #ecfdf5; color: black; padding: 12px; border-radius: 6px; overflow-x: auto; white-space: pre-wrap; font-size: 13px; line-height: 1.4; border: 1px solid #a7f3d0;">{gt_json_str}</pre>
            </div>
            <div style="flex: 1; min-width: 300px; padding: 16px; background: #fffcfc;">
                <h4 style="margin: 0 0 10px 0; color: #dc2626; font-size: 15px;">❌ Generated JSON</h4>
                <pre style="background: #fef2f2; color: black; padding: 12px; border-radius: 6px; overflow-x: auto; white-space: pre-wrap; font-size: 13px; line-height: 1.4; border: 1px solid #fecaca;">{gen_json_str}</pre>
            </div>
        </div>

        <div style="padding: 20px; background-color: #ffffff;">
            <h3 style="margin: 0 0 16px 0; color: #1e3a8a; font-size: 1.1rem; border-bottom: 2px solid #bfdbfe; padding-bottom: 6px; display: inline-block;">🔍 Diagnostic & Advisor Feedback Flow</h3>
            <div style="display: flex; flex-direction: column; gap: 16px;">
    """

    # 에러, 진단, 추천 내역을 묶어서 (Card 스타일로) 반복 출력
    for i, (fail, diag, mut) in enumerate(zip_longest(failures, diagnostics, mutations, fillvalue=None)):
        fail_str = str(fail) if fail else "Unknown Failure"
        diag_str = str(diag) if diag else "No detailed diagnostic available."
        
        # Mutation Dictionary 처리
        mut_type = "N/A"
        block_id = "N/A"
        micro_rule = "N/A"
        if isinstance(mut, dict):
            mut_type = mut.get('suggested_mutation_type', 'N/A')
            block_id = mut.get('target_block_id', 'N/A')
            micro_rule = mut.get('micro_rule', 'N/A')
        elif isinstance(mut, str):
            mut_type = mut

        html_content += f"""
                <div style="border: 1px solid #e5e7eb; border-radius: 8px; padding: 16px; background: #f9fafb; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
                    <div style="display: flex; align-items: center; gap: 8px; margin-bottom: 12px;">
                        <span style="background: #fee2e2; color: #b91c1c; padding: 4px 10px; border-radius: 9999px; font-size: 12px; font-weight: bold;">Error {i+1}</span>
                        <span style="font-weight: 600; font-size: 15px; color: #111827;">{fail_str}</span>
                    </div>
                    
                    <div style="margin-left: 12px; padding-left: 16px; border-left: 3px solid #93c5fd;">
                        <p style="margin: 0 0 8px 0; font-size: 14px; color: #374151;">
                            <strong>🩺 Diagnostic:</strong> {diag_str}
                        </p>
                        <p style="margin: 0 0 8px 0; font-size: 14px; color: #374151;">
                            <strong>🎯 Target / Action:</strong> [Block <span style="color: #2563eb; font-weight:bold;">{block_id}</span>] - {mut_type}
                        </p>
                        <p style="margin: 0; font-size: 14px; color: #047857; background: #d1fae5; padding: 8px; border-radius: 4px;">
                            <strong>💡 Advisor Rule:</strong> {micro_rule}
                        </p>
                    </div>
                </div>
        """

    html_content += """
            </div>
        </div>
    </div>
    """

    display(HTML(html_content))

# 실행 예시 (위에서 생성하신 row_advisor_df를 사용합니다)
render_advisor_report(row_advisor_df, target_row_no=251)

## 19. Save advisor-rich feedback artifacts

이 cell은 다음 파일을 저장합니다.

- `row_advisor_mapping.csv`: Excel 검토용
- `row_advisor_mapping.jsonl`: row별 machine-readable record
- `advisor_rich_feedback.json`: advisor prompt 입력 payload

In [85]:
# ============================================================
# Cell 19. Save advisor-rich feedback artifacts
# ============================================================

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def records_from_json_series(series):
    out = []
    for x in series:
        obj = safe_json_loads(x, [])
        if isinstance(obj, list):
            out.extend(obj)
    return out

def build_family_clusters(row_advisor_df, max_examples_per_family=10):
    clusters = []
    if row_advisor_df.empty:
        return clusters

    family_counts = Counter()
    for fams in row_advisor_df["advisor_families"].tolist():
        for fam in as_list(fams):
            family_counts[fam] += 1

    for family, count in family_counts.most_common():
        hit = row_advisor_df[
            row_advisor_df["advisor_families"].astype(str).str.contains(family, regex=False, na=False)
        ].sort_values("severity_score", ascending=False).head(max_examples_per_family)

        target_blocks = Counter()
        diag_examples = []
        mutation_examples = []
        row_ids = []

        for _, r in hit.iterrows():
            row_ids.append(str(r.get("row_no", "")))
            for bid in as_list(r.get("target_block_ids", "")):
                target_blocks[bid] += 1
            diags = as_list(r.get("concrete_diagnostics", ""))
            if diags:
                diag_examples.append(diags[0])
            for mut in safe_json_loads(r.get("recommended_prompt_mutations", ""), []):
                if mut.get("target_block_family") == family:
                    mutation_examples.append(mut)

        clusters.append({
            "target_block_family": family,
            "target_block_id": target_blocks.most_common(1)[0][0] if target_blocks else "",
            "count": int(count),
            "example_rows": row_ids,
            "top_concrete_diagnostics": unique_keep_order(diag_examples)[:8],
            "recommended_mutation_examples": unique_keep_order(mutation_examples)[:8],
        })
    return clusters

def build_advisor_rich_feedback_payload(row_advisor_df, max_rows=160):
    reason_counter = Counter()
    branch_counter = Counter()
    family_counter = Counter()
    state_counter = Counter()
    mutation_counter = Counter()

    for _, row in row_advisor_df.iterrows():
        for sig in as_list(row.get("normalized_signals", "")):
            reason_counter[sig] += 1
        branch_counter[str(row.get("branch", ""))] += 1
        state_counter[str(row.get("generation_state_class", ""))] += 1
        for fam in as_list(row.get("advisor_families", "")):
            family_counter[fam] += 1
        for mut in safe_json_loads(row.get("recommended_prompt_mutations", ""), []):
            mutation_counter[mut.get("suggested_mutation_type", "")] += 1

    rows = row_advisor_df.sort_values("severity_score", ascending=False).head(max_rows).to_dict("records") if not row_advisor_df.empty else []

    payload = {
        "schema_version": "row_advisor_rich_feedback_v2",
        "source_row_evaluation_csv": str(ROW_EVALUATION_CSV),
        "created_at": pd.Timestamp.now().isoformat(),
        "official_metric": "strict_det",
        "cloud_is_auxiliary": True,
        "purpose": (
            "Empirical row-level strict DET failure evidence for JOILang prompt mutation advisor. "
            "Use concrete_diagnostics and target_block_id/micro_rule rather than failure_reason labels alone."
        ),
        "directive": {
            "expand": "Expand each micro_rule into a context-aware constraint based on concrete diagnostics.",
            "implement": "Produce localized text patches for the assigned target_block_id only.",
            "verify": "Check schema/JOILang safety and avoid broad prompt rewrites or structural regressions.",
        },
        "summary": {
            "rows_total": int(len(row_eval_df)),
            "rows_with_feedback": int(len(row_advisor_df)),
            "top_failure_reasons": reason_counter.most_common(50),
            "branch_distribution": branch_counter.most_common(),
            "generation_state_distribution": state_counter.most_common(),
            "top_advisor_families": family_counter.most_common(30),
            "top_mutation_policies": mutation_counter.most_common(50),
        },
        "family_clusters": build_family_clusters(row_advisor_df, max_examples_per_family=12),
        "rows": rows,
        "taxonomy_table": TAXONOMY_ROWS,
    }
    return payload

row_advisor_csv = ROW_ADVISOR_OUT_DIR / "row_advisor_mapping.csv"
row_advisor_jsonl = ROW_ADVISOR_OUT_DIR / "row_advisor_mapping.jsonl"
advisor_rich_feedback_json = ROW_ADVISOR_OUT_DIR / "advisor_rich_feedback.json"
taxonomy_csv = ROW_ADVISOR_OUT_DIR / "failure_taxonomy_table.csv"

row_advisor_df.to_csv(row_advisor_csv, index=False, encoding="utf-8-sig")
write_jsonl(row_advisor_jsonl, row_advisor_records)
taxonomy_df.to_csv(taxonomy_csv, index=False, encoding="utf-8-sig")

advisor_payload = build_advisor_rich_feedback_payload(row_advisor_df, max_rows=160)
advisor_rich_feedback_json.write_text(
    json.dumps(advisor_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved:")
print(" -", row_advisor_csv)
print(" -", row_advisor_jsonl)
print(" -", advisor_rich_feedback_json)
print(" -", taxonomy_csv)

display(pd.DataFrame(advisor_payload["summary"]["top_failure_reasons"], columns=["signal", "count"]).head(30))
display(pd.DataFrame(advisor_payload["summary"]["top_advisor_families"], columns=["family", "count"]).head(20))
display(pd.DataFrame(advisor_payload["summary"]["branch_distribution"], columns=["branch", "count"]))

Saved:
 - /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/row_advisor_mapping.csv
 - /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/row_advisor_mapping.jsonl
 - /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_rich_feedback.json
 - /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/failure_taxonomy_table.csv


,signal,count
0,output_collapse,280
1,unknown_service,280
2,gt_service_coverage,280
3,period_mismatch,280
4,gt_mismatch,280
5,gt_receiver_coverage,279
6,numeric_grounding,161
7,enum_grounding,161
8,cron_mismatch,40


,family,count
0,Skeleton,280
1,Service_Mapping,280
2,Temporal_Rule,280
3,DET_Helper,280
4,Receiver_Tag_Preservation,279
5,Numeric_Unit_Grounding,161
6,Enum_Grounding,161


,branch,count
0,strict_det_concrete_mismatch,280


## 20. Expand / implement / verify localized prompt patches

이 cell은 `advisor_rich_feedback.json`의 `family_clusters`를 deterministic하게 patch proposal로 변환합니다.

목표:

- 각 cluster에 대해 `micro_rule`을 그대로 반복하지 않고, `concrete_diagnostics`를 반영해 확장합니다.
- `target_block_id`에만 localized patch를 제안합니다.
- JSON schema/JOILang 구조를 깨지 않는지 간단한 검증을 수행합니다.
- 산출물은 strict JSON 형태의 `expanded_verified_patches.json`입니다.

In [87]:
# ============================================================
# Cell 20. Expand / implement / verify localized patch JSON
# ============================================================
AUTHORITATIVE_BLOCK_BY_FAMILY = {
    "Service_Mapping": "02",
    "Receiver_Tag_Preservation": "02",
    "Enum_Grounding": "02",
    "Argument_Grounding": "02",

    "Output_Schema": "03",
    "Parser_Extraction": "03",

    "Temporal_Rule": "06",
    "Numeric_Unit_Grounding": "06",
    "Dataflow": "06",
    "Intent_Fulfillment": "06",
    "Skeleton": "06",
    "DET_Helper": "06",
    "Minimality": "06",

    "Generation_Health": "00",
    "Runtime_Health": "00",
    "Prompt_Budget": "00",
    "No_Mutation": "00",
}

FAMILY_DIAGNOSTIC_KEYWORDS = {
    "Service_Mapping": ["service mismatch", "unknown", "canonical", "camelCase", "class-style"],
    "Receiver_Tag_Preservation": ["receiver mismatch", "receiver"],
    "Temporal_Rule": ["period mismatch", "cron mismatch", "schedule"],
    "Numeric_Unit_Grounding": ["numeric mismatch", "numeric"],
    "Enum_Grounding": ["enum", "string args"],
    "Dataflow": ["dataflow", "read-bind-use"],
    "Skeleton": ["output collapse", "row independently", "skeleton"],
    "DET_Helper": ["overall GT mismatch", "gt mismatch", "component"],
    "Output_Schema": ["invalid JSON", "missing required", "schema"],
}

def truncate_sentence(text, max_chars=420):
    text = re.sub(r"\s+", " ", str(text).strip())
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "."

def representative_diagnostics_for_cluster(cluster, max_items=4):
    family = cluster.get("target_block_family", "")
    diags = [str(x) for x in (cluster.get("top_concrete_diagnostics", []) or []) if str(x).strip()]
    keywords = FAMILY_DIAGNOSTIC_KEYWORDS.get(family, [])

    if keywords:
        matched = [
            d for d in diags
            if any(k.lower() in d.lower() for k in keywords)
        ]
        if matched:
            return matched[:max_items]

    # fallback: family-specific default diagnostic
    if family == "Service_Mapping":
        return ["service mismatch: expected canonical schema service but generated service differs or is hallucinated."]
    if family == "Receiver_Tag_Preservation":
        return ["receiver mismatch: generated receiver differs from command/GT receiver."]
    if family == "Temporal_Rule":
        return ["schedule mismatch: cron/period policy differs from GT."]
    if family == "Numeric_Unit_Grounding":
        return ["numeric mismatch: required numeric literal or converted unit is missing or incorrect."]
    if family == "Enum_Grounding":
        return ["enum/string mismatch: generated enum argument differs from selected service descriptor."]
    if family == "Skeleton":
        return ["output collapse: same generated code reused across unrelated rows."]
    if family == "DET_Helper":
        return ["overall GT mismatch: perform component-wise final verification."]

    return diags[:max_items]

def merged_micro_rules_for_cluster(cluster):
    rules = []
    for m in cluster.get("recommended_mutation_examples", []) or []:
        rule = m.get("micro_rule", "")
        if rule:
            rules.append(rule)
    return unique_keep_order(rules)

def infer_patch_focus(cluster):
    family = cluster.get("target_block_family", "")
    diagnostics = " ".join(representative_diagnostics_for_cluster(cluster, max_items=8)).lower()

    if "camelcase" in diagnostics or "class-style" in diagnostics or family == "Service_Mapping":
        return "canonical_service"
    if "receiver mismatch" in diagnostics or family == "Receiver_Tag_Preservation":
        return "receiver_first"
    if "period mismatch" in diagnostics or "cron mismatch" in diagnostics or family == "Temporal_Rule":
        return "schedule_policy"
    if "numeric mismatch" in diagnostics or family == "Numeric_Unit_Grounding":
        return "numeric_descriptor"
    if "enum" in diagnostics or family == "Enum_Grounding":
        return "enum_descriptor"
    if "empty generated code" in diagnostics or family == "Intent_Fulfillment":
        return "non_empty_behavior"
    if family == "Output_Schema":
        return "json_schema"
    if "collapse" in diagnostics:
        return "row_independence"
    return "general_det"

def build_context_aware_patch(cluster):
    family = cluster.get("target_block_family", "")
    target_block_id = AUTHORITATIVE_BLOCK_BY_FAMILY.get(
        family,
        str(cluster.get("target_block_id", "") or "06")
    )
    count = cluster.get("count", 0)
    diags = representative_diagnostics_for_cluster(cluster, max_items=6)
    rules = merged_micro_rules_for_cluster(cluster)
    focus = infer_patch_focus(cluster)

    # Atomic 2-3 sentence patches.
    if focus == "canonical_service":
        patch = (
            "Never invent service/member names. Copy the exact canonical device-prefixed service member from the injected schema, preserving lowercase, underscores, and device prefix. "
            "If a generated member looks camelCase, class-style, capitalized, or paraphrased, replace it with the nearest schema-valid canonical member before final JSON."
        )
    elif focus == "receiver_first":
        patch = (
            "Select the receiver tag from the current command target before choosing any service. "
            "Preserve owner/location/group/sector tags exactly and choose only services attached to that receiver; never reuse a receiver from a previous row."
        )
    elif focus == "schedule_policy":
        patch = (
            "Classify schedule type before writing JSON: one-shot action, fixed cron trigger, repeated period loop, delay sequence, or trigger-then-repeat. "
            "Use period=0 for one-shot or scheduled one-shot commands, preserve explicit cron triggers, and use positive period only when repeated monitoring is explicit."
        )
    elif focus == "numeric_descriptor":
        patch = (
            "Preserve every numeric literal required by the current command and bind it to the selected service argument. "
            "Convert units using the service descriptor, such as minutes to seconds for seconds-based arguments, and never drop numeric thresholds or durations."
        )
    elif focus == "enum_descriptor":
        patch = (
            "For enum-valued services, copy the allowed enum string exactly from the selected service descriptor. "
            "Do not translate, paraphrase, or borrow enum values from another device or previous row."
        )
    elif focus == "json_schema":
        patch = (
            "Return exactly one bare JSON object with required keys name, cron, period, and code. "
            "Do not emit markdown fences, prose, comments, multiple JSON objects, missing keys, or invalid field types. "
            "If a device-list/schema snippet is emitted or transformed, use each device name as the object key, e.g. device_name: {info: ..., examples: ...}; never place the device name as a nested value."
        )
    elif focus == "non_empty_behavior":
        patch = (
            "For every non-empty user command, the code field must contain the required JOILang action, condition, schedule body, or notification. "
            "Do not return empty code unless the official GT behavior is explicitly empty."
        )
    elif focus == "row_independence":
        patch = (
            "Solve each dataset row independently from the current command only. "
            "Never reuse a previous row's JSON name, receiver, service, enum argument, numeric argument, or code skeleton."
        )
    else:
        base = rules[0] if rules else DEFAULT_TAXONOMY_ROW["micro_rule"]
        patch = (
            truncate_sentence(base, max_chars=260) + " "
            "Verify schedule, receiver, service, numeric, enum, dataflow, and output schema before final JSON."
        )

    evidence_text = "; ".join(diags[:3])
    rationale = (
        f"Expanded {family} rule from {count} empirical row(s). "
        f"Representative diagnostics: {evidence_text if evidence_text else 'fallback diagnostic generated from mapped failure signal; no missing advisor patch emitted'}"
    )
    return {
        "target_block_id": target_block_id,
        "target_block_family": family,
        "rationale": truncate_sentence(rationale, max_chars=500),
        "suggested_patch_text": truncate_sentence(patch, max_chars=520),
        "verification": {
            "localized_patch_only": target_block_id not in {"", "00"},
            "uses_concrete_diagnostics": bool(diags),
            "imperative_rule": any(patch.strip().startswith(v) for v in ["Never", "Select", "Classify", "Preserve", "For", "Return", "Solve", "Do"]),
            "json_schema_safe": "JSON" in patch or family not in {"Output_Schema"},
            "does_not_rewrite_entire_prompt": True,
            "no_markdown_fence": "```" not in patch,
        },
        "evidence_rows": cluster.get("example_rows", []),
        "evidence_diagnostics": diags,
    }

def merge_patches_by_block_and_family(patches):
    grouped = {}
    for p in patches:
        key = (p["target_block_id"], p["target_block_family"])
        if key not in grouped:
            grouped[key] = p
        else:
            old = grouped[key]
            old["evidence_rows"] = unique_keep_order(old.get("evidence_rows", []) + p.get("evidence_rows", []))
            old["evidence_diagnostics"] = unique_keep_order(old.get("evidence_diagnostics", []) + p.get("evidence_diagnostics", []))
            old["rationale"] = truncate_sentence(
                old["rationale"] + " Additional diagnostics: " + "; ".join(p.get("evidence_diagnostics", [])[:2]),
                max_chars=650,
            )
    return list(grouped.values())

def verify_patch_plan(plan):
    errors = []
    warnings = []
    for i, patch in enumerate(plan.get("mutated_blocks", [])):
        bid = str(patch.get("target_block_id", ""))
        txt = str(patch.get("suggested_patch_text", ""))
        family = str(patch.get("target_block_family", ""))

        if not bid or bid == "00":
            warnings.append(f"patch[{i}] is non-injectable/no-mutation target_block_id={bid}; excluded from injectable plan")
        if not txt.strip():
            errors.append(f"patch[{i}] empty suggested_patch_text")
        if "```" in txt:
            errors.append(f"patch[{i}] contains markdown fence")
        if len(txt) > 700:
            warnings.append(f"patch[{i}] is long ({len(txt)} chars); consider splitting")
        if family == "Output_Schema" and "JSON" not in txt:
            warnings.append(f"patch[{i}] Output_Schema patch does not explicitly mention JSON")
        if family == "Service_Mapping" and not any(k in txt for k in ["schema", "canonical", "service"]):
            warnings.append(f"patch[{i}] Service_Mapping patch may be too vague")
    return {"errors": errors, "warnings": warnings, "ok": not errors}

clusters = advisor_payload.get("family_clusters", [])
raw_patches = []
for cluster in clusters:
    # Do not inject Generation_Health/No_Mutation/Runtime-only as prompt text unless target block is real.
    family = cluster.get("target_block_family", "")
    bid = str(cluster.get("target_block_id", "") or "")
    if family in {"No_Mutation", "Generation_Health", "Runtime_Health", "Prompt_Budget"} or bid == "00":
        continue
    raw_patches.append(build_context_aware_patch(cluster))

mutated_blocks = merge_patches_by_block_and_family(raw_patches)
expanded_verified_patches = {
    "schema_version": "v1",
    "source_advisor_rich_feedback": str(advisor_rich_feedback_json),
    "official_metric": "strict_det",
    "cloud_is_auxiliary": True,
    "mutated_blocks": mutated_blocks,
}
expanded_verified_patches["verification_summary"] = verify_patch_plan(expanded_verified_patches)

if not expanded_verified_patches["mutated_blocks"] and advisor_payload.get("rows"):
    cluster_preview = pd.DataFrame([
        {
            "target_block_family": c.get("target_block_family"),
            "target_block_id": c.get("target_block_id"),
            "count": c.get("count"),
            "example_rows": c.get("example_rows"),
            "top_concrete_diagnostics": c.get("top_concrete_diagnostics"),
        }
        for c in advisor_payload.get("family_clusters", [])
    ])

    print("[WARNING] No actionable prompt patches were generated.")
    print("This usually means family_clusters contain only Generation_Health/Runtime_Health/No_Mutation or Block 00.")
    print("If row_evaluation.csv has non-empty generated_code with DET failures, rerun Cells 17-19 after the NaN generation_error fix.")
    display(cluster_preview)

    raise RuntimeError(
        "No actionable prompt patch generated. Rerun Cells 17, 18, and 19 with the fixed generation-state logic before Cell 20."
    )


expanded_verified_patches_json = ROW_ADVISOR_OUT_DIR / "expanded_verified_patches.json"
expanded_verified_patches_json.write_text(
    json.dumps(expanded_verified_patches, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("expanded_verified_patches_json:", expanded_verified_patches_json)
print(json.dumps(expanded_verified_patches["verification_summary"], ensure_ascii=False, indent=2))
display(pd.DataFrame(expanded_verified_patches["mutated_blocks"]))

print("\nStrict JSON preview:")
print(json.dumps({
    "schema_version": expanded_verified_patches["schema_version"],
    "mutated_blocks": [
        {
            "target_block_id": p["target_block_id"],
            "target_block_family": p["target_block_family"],
            "rationale": p["rationale"],
            "suggested_patch_text": p["suggested_patch_text"],
        }
        for p in expanded_verified_patches["mutated_blocks"]
    ]
}, ensure_ascii=False, indent=2))

expanded_verified_patches_json: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/expanded_verified_patches.json
{
  "errors": [],
  "warnings": [],
  "ok": true
}


,target_block_id,target_block_family,rationale,suggested_patch_text,verification,evidence_rows,evidence_diagnostics
0,06,Skeleton,Expanded Skeleton rule from 280 empirical row(s). Representative diagnostics: output collapse: same generated code reused across unrelated rows.,"Solve each dataset row independently from the current command only. Never reuse a previous row's JSON name, receiver, service, enum argument, numeric argument, or code skeleton.","{'localized_patch_only': True, 'uses_concrete_diagnostics': True, 'imperative_rule': True, 'json_schema_safe': True, 'does_not_rewrite_entire_prompt': True, 'no_markdown_fence': True}","[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output collapse: same generated code reused across unrelated rows.]
1,02,Service_Mapping,Expanded Service_Mapping rule from 280 empirical row(s). Representative diagnostics: service mismatch: expected canonical schema service but generated service differs or is hallucinated.,"Never invent service/member names. Copy the exact canonical device-prefixed service member from the injected schema, preserving lowercase, underscores, and device prefix. If a generated member looks camelCase, class-style, capitalized, or paraphrased, replace it with the nearest schema-valid can...","{'localized_patch_only': True, 'uses_concrete_diagnostics': True, 'imperative_rule': True, 'json_schema_safe': True, 'does_not_rewrite_entire_prompt': True, 'no_markdown_fence': True}","[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[service mismatch: expected canonical schema service but generated service differs or is hallucinated.]
2,06,Temporal_Rule,Expanded Temporal_Rule rule from 280 empirical row(s). Representative diagnostics: schedule mismatch: cron/period policy differs from GT.,"Classify schedule type before writing JSON: one-shot action, fixed cron trigger, repeated period loop, delay sequence, or trigger-then-repeat. Use period=0 for one-shot or scheduled one-shot commands, preserve explicit cron triggers, and use positive period only when repeated monitoring is expli...","{'localized_patch_only': True, 'uses_concrete_diagnostics': True, 'imperative_rule': True, 'json_schema_safe': True, 'does_not_rewrite_entire_prompt': True, 'no_markdown_fence': True}","[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[schedule mismatch: cron/period policy differs from GT.]
3,06,DET_Helper,Expanded DET_Helper rule from 280 empirical row(s). Representative diagnostics: overall GT mismatch: perform component-wise final verification.,"When code is schema-valid but not target-equivalent, compare schedule, receiver, service, numeric, enum, dataflow, and action order before final output. Verify schedule, receiver, service, numeric, enum, dataflow, and output schema before final JSON.","{'localized_patch_only': True, 'uses_concrete_diagnostics': True, 'imperative_rule': False, 'json_schema_safe': True, 'does_not_rewrite_entire_prompt': True, 'no_markdown_fence': True}","[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[overall GT mismatch: perform component-wise final verification.]
4,02,Receiver_Tag_Preservation,Expanded Receiver_Tag_Preservation rule from 279 empirical row(s). Representative diagnostics: receiver mismatch: generated receiver differs from command/GT receiver.,Select the receiver tag from the current command target before choosing any service. Preserve owner/location/group/sector tags exactly and choose only services attached to that receiver; never reuse a receiver from a previous row.,"{'localized_patch_only': True, 'uses_concrete_diagnostics': True, 'imperative_rule': True, 'json_schema_safe': True, 'does_not_rewrite_entire_prompt': True, 'no_markdown_fence': True}","[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[receiver mismatch: generated receiver differs from command/GT receiver.]
5,06,Numeric_Unit_Grounding,Expanded Numeric_Unit_Grounding rule from 161 empirical row(s). Representative diagnostics


Strict JSON preview:
{
  "schema_version": "v1",
  "mutated_blocks": [
    {
      "target_block_id": "06",
      "target_block_family": "Skeleton",
      "rationale": "Expanded Skeleton rule from 280 empirical row(s). Representative diagnostics: output collapse: same generated code reused across unrelated rows.",
      "suggested_patch_text": "Solve each dataset row independently from the current command only. Never reuse a previous row's JSON name, receiver, service, enum argument, numeric argument, or code skeleton."
    },
    {
      "target_block_id": "02",
      "target_block_family": "Service_Mapping",
      "rationale": "Expanded Service_Mapping rule from 280 empirical row(s). Representative diagnostics: service mismatch: expected canonical schema service but generated service differs or is hallucinated.",
      "suggested_patch_text": "Never invent service/member names. Copy the exact canonical device-prefixed service member from the injected schema, preserving lowercase, un

In [89]:
clusters = advisor_payload.get("family_clusters", [])

cluster_df = pd.DataFrame([
    {
        "target_block_family": c.get("target_block_family"),
        "target_block_id": c.get("target_block_id"),
        "count": c.get("count"),
        "example_rows": c.get("example_rows"),
        "top_concrete_diagnostics": c.get("top_concrete_diagnostics"),
    }
    for c in clusters
])

display(cluster_df)
display(cluster_df["target_block_family"].value_counts(dropna=False))

,target_block_family,target_block_id,count,example_rows,top_concrete_diagnostics
0,Skeleton,06,280,"[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]
1,Service_Mapping,06,280,"[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]
2,Temporal_Rule,06,280,"[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]
3,DET_Helper,06,280,"[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]
4,Receiver_Tag_Preservation,06,279,"[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]
5,Numeric_Unit_Grounding,06,161,"[218, 222, 195, 186, 187, 251, 211, 219, 230, 216, 217, 264]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]
6,Enum_Grounding,06,161,"[218, 222, 195, 186, 187, 264, 174, 280, 259, 150, 114, 170]",[output_collapse: no specialized diagnostic rule; compare gt_code and generated_code]


target_block_family
Skeleton                     1
Service_Mapping              1
Temporal_Rule                1
DET_Helper                   1
Receiver_Tag_Preservation    1
Numeric_Unit_Grounding       1
Enum_Grounding               1
Name: count, dtype: int64

## 21. Convert verified mutations to prompt_patches.json and validate mapping coverage

Cell 20의 `expanded_verified_patches.json`은 strict JSON preview 용도입니다.  
이 cell은 이를 실제 `utils.ga_search.prompt_patch_apply`가 읽을 수 있는 `prompt_patches.json` 스키마로 변환합니다.

추가 검증:

- row별 `failure_reasons` 중 taxonomy에 매핑되지 않은 signal을 찾음
- 매핑이 없는 signal도 최소 fallback `DET_Helper / Block 06` diagnostic과 advisor rule을 부여
- 빈 diagnostic, 빈 target/action, 빈 advisor rule이 나오지 않도록 검사
- `cron_mismatch`, `period_mismatch`, `missing_required_key:<keys>` 등은 반드시 구체 family로 routing되는지 확인

In [97]:
# ============================================================
# Cell 21. Convert verified mutations to prompt_patches.json and validate mapping coverage
# ============================================================
EXPECTED_BLOCK_BY_FAMILY = {
    "Service_Mapping": "02",
    "Receiver_Tag_Preservation": "02",
    "Enum_Grounding": "02",
    "Argument_Grounding": "02",
    "Output_Schema": "03",
    "Temporal_Rule": "06",
    "Numeric_Unit_Grounding": "06",
    "Dataflow": "06",
    "Intent_Fulfillment": "06",
    "Skeleton": "06",
    "DET_Helper": "06",
    "Minimality": "06",
}

def _ensure_loaded_row_advisor_artifacts():
    """
    Robustly load row-advisor artifacts if the kernel was restarted and variables are missing.
    """
    global ROW_ADVISOR_OUT_DIR, advisor_rich_feedback_json, advisor_payload, expanded_verified_patches_json, expanded_verified_patches

    if "ROW_ADVISOR_OUT_DIR" not in globals():
        if "ROW_EVALUATION_CSV" in globals():
            ROW_ADVISOR_OUT_DIR = Path(ROW_EVALUATION_CSV).parent.parent / "row_advisor_mapping"
        else:
            raise RuntimeError("ROW_ADVISOR_OUT_DIR is not defined. Run Cells 15-20 first.")

    ROW_ADVISOR_OUT_DIR = Path(ROW_ADVISOR_OUT_DIR)
    advisor_rich_feedback_json = globals().get(
        "advisor_rich_feedback_json",
        ROW_ADVISOR_OUT_DIR / "advisor_rich_feedback.json",
    )
    expanded_verified_patches_json = globals().get(
        "expanded_verified_patches_json",
        ROW_ADVISOR_OUT_DIR / "expanded_verified_patches.json",
    )

    if "advisor_payload" not in globals() or not isinstance(globals().get("advisor_payload"), dict):
        if Path(advisor_rich_feedback_json).exists():
            advisor_payload = json.loads(Path(advisor_rich_feedback_json).read_text(encoding="utf-8"))
        else:
            raise FileNotFoundError(advisor_rich_feedback_json)

    if "expanded_verified_patches" not in globals() or not isinstance(globals().get("expanded_verified_patches"), dict):
        if Path(expanded_verified_patches_json).exists():
            expanded_verified_patches = json.loads(Path(expanded_verified_patches_json).read_text(encoding="utf-8"))
        else:
            raise FileNotFoundError(expanded_verified_patches_json)

    return advisor_payload, expanded_verified_patches

advisor_payload, expanded_verified_patches = _ensure_loaded_row_advisor_artifacts()

def collect_all_signals_from_advisor_payload(payload):
    signals = []
    for row in payload.get("rows", []) or []:
        for key in ["normalized_signals", "failure_reason_bases", "failure_reasons"]:
            value = row.get(key, [])
            for item in as_list(value):
                sig = normalize_signal(item)
                if sig:
                    signals.append(sig)
    return unique_keep_order(signals)

def fallback_taxonomy_for_signal(signal):
    """
    Never return an empty mapping. If taxonomy is missing, route to DET_Helper / Block 06
    and force the exact signal name into the diagnostic and advisor rule.
    """
    sig = normalize_signal(signal) or str(signal or "unknown_failure")
    return {
        "category": "Fallback",
        "signal": sig,
        "signal_type": "failure_reason",
        "produced_by": "postprocess",
        "evidence_fields": ["failure_reasons", "gt_code", "generated_code", "diff_summary"],
        "diagnostic_template": f"{sig}: mapped fallback diagnostic. Compare gt_code, generated_code, and diff_summary for this exact failure signal.",
        "target_family": "DET_Helper",
        "target_block_id": "06",
        "mutation_policy": f"Do not ignore {sig}; add a localized DET repair hint if no more specific mapping exists.",
        "difficulty": "fallback",
        "priority": 95,
        "final_check": "Fallback mapping prevents empty advisor output.",
        "micro_rule": f"When {sig} appears, compare the row's GT and generated components and repair the concrete mismatch before final JSON.",
    }

def taxonomy_for_signal_strict(signal):
    sig = normalize_signal(signal)
    row = FAILURE_ADVISOR_MAP.get(sig)
    if row:
        return row
    return fallback_taxonomy_for_signal(sig)

def mapping_coverage_report(payload):
    signals = collect_all_signals_from_advisor_payload(payload)
    rows = []
    for sig in signals:
        mapped = sig in FAILURE_ADVISOR_MAP
        tax = taxonomy_for_signal_strict(sig)
        rows.append({
            "signal": sig,
            "mapped": bool(mapped),
            "target_family": tax.get("target_family", "DET_Helper"),
            "target_block_id": tax.get("target_block_id", "06"),
            "diagnostic_template": tax.get("diagnostic_template", ""),
            "micro_rule": tax.get("micro_rule", ""),
        })
    return pd.DataFrame(rows)

coverage_df = mapping_coverage_report(advisor_payload)
coverage_csv = ROW_ADVISOR_OUT_DIR / "advisor_mapping_coverage_report.csv"
coverage_df.to_csv(coverage_csv, index=False, encoding="utf-8-sig")

unmapped = coverage_df[~coverage_df["mapped"]] if not coverage_df.empty else pd.DataFrame()
print("mapping coverage report:", coverage_csv)
print("signals:", len(coverage_df), "unmapped:", len(unmapped))
display(coverage_df.sort_values(["mapped", "signal"]).head(200))

required_signals = ["cron_mismatch", "period_mismatch", "missing_required_key", "gt_mismatch"]
required_check = []
for sig in required_signals:
    tax = taxonomy_for_signal_strict(sig)
    required_check.append({
        "signal": sig,
        "target_family": tax.get("target_family"),
        "target_block_id": tax.get("target_block_id"),
        "has_non_empty_diagnostic": bool(str(tax.get("diagnostic_template", "")).strip()),
        "has_non_empty_rule": bool(str(tax.get("micro_rule", "")).strip()),
    })
required_check_df = pd.DataFrame(required_check)
display(required_check_df)

def convert_verified_patch_to_prompt_patch(patch, index):
    family = str(patch.get("target_block_family", "") or "DET_Helper")
    raw_bid = str(patch.get("target_block_id", "") or "06")
    bid = EXPECTED_BLOCK_BY_FAMILY.get(family, raw_bid)
    text = str(patch.get("suggested_patch_text", "") or "").strip()
    rationale = str(patch.get("rationale", "") or "").strip()
    evidence_rows = patch.get("evidence_rows", []) or []
    evidence_diagnostics = patch.get("evidence_diagnostics", []) or []

    if not text:
        text = "Compare GT and generated output component-by-component and repair the concrete mismatch before final JSON."
    if not rationale:
        rationale = f"Fallback rationale for {family}; advisor patch emitted with conservative DET guidance."

    return {
        "patch_id": f"row_advisor_{index:03d}_{family}_{bid}",
        "target_block_family": family,
        "target_block_id": bid,
        "operation": "append_micro_rule",
        "priority": max(1, 100 - index),
        "patch_text": text,
        "rationale": rationale,
        "evidence_rows": [str(x) for x in evidence_rows],
        "evidence_diagnostics": [str(x) for x in evidence_diagnostics],
        "mutation_intent": "strict_det_row_advisor_repair",
        "strict_det_basis": {
            "official_metric": "strict_det",
            "cloud_is_auxiliary": True,
            "source_advisor_rich_feedback": str(advisor_rich_feedback_json),
            "source_expanded_verified_patches": str(expanded_verified_patches_json),
        },
    }

verified_blocks = expanded_verified_patches.get("mutated_blocks", []) or []
prompt_patches = []
for idx, patch in enumerate(verified_blocks, start=1):
    bid = str(patch.get("target_block_id", ""))
    family = str(patch.get("target_block_family", ""))
    if family in {"No_Mutation", "Generation_Health", "Runtime_Health", "Prompt_Budget"} or bid in {"", "00"}:
        continue
    prompt_patches.append(convert_verified_patch_to_prompt_patch(patch, idx))

if not prompt_patches and advisor_payload.get("rows"):
    cluster_preview = pd.DataFrame([
        {
            "target_block_family": c.get("target_block_family"),
            "target_block_id": c.get("target_block_id"),
            "count": c.get("count"),
            "example_rows": c.get("example_rows"),
            "top_concrete_diagnostics": c.get("top_concrete_diagnostics"),
        }
        for c in advisor_payload.get("family_clusters", [])
    ])
    print("[ERROR] No injectable semantic/schema prompt patch was produced.")
    print("Do not create a Generation_Health/Block00 fallback patch.")
    print("Rerun Cells 17-20 after fixing/recomputing row_advisor_df and advisor_payload.")
    display(cluster_preview)
    raise RuntimeError(
        "advisor_prompt_patches.json would contain no actionable JOICode prompt repair patches."
    )

advisor_prompt_patch_payload = {
    "advisor_meta": {
        "schema_version": "row_advisor_prompt_patches_v1",
        "source": "row_advisor_mapping",
        "official_metric": "strict_det",
        "cloud_is_auxiliary": True,
        "created_at": pd.Timestamp.now().isoformat(),
        "coverage_report": str(coverage_csv),
        "note": "Generated from row-level concrete diagnostics. Patch application is optional and disabled by default in Cell 22.",
    },
    "prompt_patches": prompt_patches,
}

advisor_prompt_patch_path = ROW_ADVISOR_OUT_DIR / "advisor_prompt_patches.json"
advisor_prompt_patch_path.write_text(
    json.dumps(advisor_prompt_patch_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

def evidence_matches_family(family, diagnostics):
    text = " ".join(str(x) for x in diagnostics).lower()
    expected = {
        "Service_Mapping": ["service", "canonical", "camelcase", "class-style"],
        "Receiver_Tag_Preservation": ["receiver"],
        "Temporal_Rule": ["period", "cron", "schedule"],
        "Numeric_Unit_Grounding": ["numeric"],
        "Enum_Grounding": ["enum", "string"],
        "Skeleton": ["collapse", "independently", "skeleton"],
        "DET_Helper": ["gt mismatch", "component", "compare"],
        "Output_Schema": ["json", "required", "schema"],
        "Dataflow": ["dataflow", "read-bind-use", "bind"],
        "Intent_Fulfillment": ["empty", "non-empty", "behavior", "action"],
    }
    keys = expected.get(family, [])
    return not keys or any(k in text for k in keys)

def validate_prompt_patch_payload(payload):
    errors = []
    warnings = []
    patches = payload.get("prompt_patches", [])

    injectable = [
        p for p in patches
        if str(p.get("target_block_id", "")) not in {"", "00"}
        and str(p.get("target_block_family", "")) not in {
            "Generation_Health",
            "Runtime_Health",
            "Prompt_Budget",
            "No_Mutation",
            "Parser_Extraction",
        }
    ]

    if advisor_payload.get("rows") and not injectable:
        errors.append(
            "No injectable semantic/schema prompt patch was generated. "
            "Do not apply this payload; rerun Cells 17-20 with corrected generation-state logic."
        )

    for i, p in enumerate(patches):
        bid = str(p.get("target_block_id", ""))
        fam = str(p.get("target_block_family", ""))
        text = str(p.get("patch_text", ""))
        diags = p.get("evidence_diagnostics", []) or []

        if not bid:
            errors.append(f"patch[{i}] has empty target_block_id")
        if not fam:
            errors.append(f"patch[{i}] has empty target_block_family")
        if bid == "00":
            errors.append(f"patch[{i}] targets non-injectable block 00")
        if fam in {"Generation_Health", "Runtime_Health", "Prompt_Budget", "No_Mutation", "Parser_Extraction"}:
            errors.append(f"patch[{i}] is not a JOICode semantic/schema prompt repair family: {fam}")
        if not text.strip():
            errors.append(f"patch[{i}] has empty patch_text")
        if "```" in text:
            errors.append(f"patch[{i}] contains markdown fence")
        if len(text) > 900:
            warnings.append(f"patch[{i}] is long: {len(text)} chars")

        expected_bid = EXPECTED_BLOCK_BY_FAMILY.get(fam)
        if expected_bid and bid != expected_bid:
            errors.append(
                f"patch[{i}] family/block mismatch: {fam} expected Block {expected_bid}, got Block {bid}"
            )

        if not evidence_matches_family(fam, diags):
            warnings.append(
                f"patch[{i}] evidence_diagnostics may not match family={fam}: {diags[:2]}"
            )

    return {"ok": not errors, "errors": errors, "warnings": warnings}
    

mapping coverage report: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_mapping_coverage_report.csv
signals: 9 unmapped: 0


,signal,mapped,target_family,target_block_id,diagnostic_template,micro_rule
4,cron_mismatch,True,Temporal_Rule,06,"cron mismatch: gt cron={gt_cron!r}, generated cron={generated_cron!r}. Fixed wall-clock schedule was lost.","For explicit fixed times, weekdays, midnight, or scheduled one-shot commands, derive cron first and preserve it exactly. Do not replace a fixed schedule with a period loop or a Clock guard unless repeated monitoring is explicit."
7,enum_grounding,True,Enum_Grounding,02,"enum mismatch: gt string args={gt_string_args}, generated string args={generated_string_args}.","For enum-valued services, copy the allowed enum value exactly from the selected service descriptor. Do not translate, paraphrase, or borrow enum values from another device or service."
8,gt_mismatch,True,DET_Helper,06,gt mismatch: code is schema-valid but not target-equivalent; prioritize concrete mismatches.,"When code is schema-valid but not target-equivalent, compare schedule, receiver, service, numeric, enum, dataflow, and action order before final output."
2,gt_receiver_coverage,True,Receiver_Tag_Preservation,02,"receiver mismatch: gt receivers={gt_receivers}, generated receivers={generated_receivers}.","Select receiver tags from the current command target before service selection. Preserve owner, location, group, and sector tags exactly, and do not reuse a receiver from another row."
3,gt_service_coverage,True,Service_Mapping,02,"missing GT service: gt services={gt_services}, generated services={generated_services}.",Include every service implied by the command. Select services only from the injected schema under the selected receiver and do not substitute adjacent service families.
6,numeric_grounding,True,Numeric_Unit_Grounding,06,"numeric mismatch: gt numeric literals={gt_numeric_literals}, generated numeric literals={generated_numeric_literals}.","Preserve required numeric arguments and thresholds from the command. Convert units using the selected service descriptor, such as minutes to seconds for seconds-based arguments."
0,output_collapse,True,Skeleton,06,output collapse: same generated code reused across unrelated rows.,"Solve each row independently from the current command. Never reuse a previous row's JSON name, receiver, service, enum, or code skeleton."
5,period_mismatch,True,Temporal_Rule,06,"period mismatch: gt period={gt_period!r}, generated period={generated_period!r}.","For one-shot action or scheduled one-shot commands, use period=0 unless repeated monitoring is explicit. Use positive period only for repeated monitoring loops and never use -1 as a substitute for a valid one-shot period."
1,unknown_service,True,Service_Mapping,02,unknown/canonical service error: generated services={generated_services}; expected schema/GT services={gt_services}.,"Never invent service/member names. Copy the canonical device-prefixed service member exactly from the injected service schema. Do not emit camelCase, class-style, capitalized, or paraphrased service names."


,signal,target_family,target_block_id,has_non_empty_diagnostic,has_non_empty_rule
0,cron_mismatch,Temporal_Rule,06,True,True
1,period_mismatch,Temporal_Rule,06,True,True
2,missing_required_key,Output_Schema,03,True,True
3,gt_mismatch,DET_Helper,06,True,True


## 22. Optional: apply row-advisor prompt patches

이 cell은 Cell 21이 만든 `advisor_prompt_patches.json`을 실제 prompt genome에 적용합니다.

기본값은 안전하게 `false`입니다.  
Excel로 `row_advisor_mapping.csv`, `advisor_mapping_coverage_report.csv`, `advisor_prompt_patches.json`을 확인한 뒤 아래처럼 켜세요.

```python
RUN_ROW_ADVISOR_PATCH_APPLY = True
```

In [99]:
from pathlib import Path
import json
import pandas as pd

BASE = Path("/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping")

paths = {
    "row_advisor_mapping": BASE / "row_advisor_mapping.csv",
    "advisor_rich_feedback": BASE / "advisor_rich_feedback.json",
    "expanded_verified_patches": BASE / "expanded_verified_patches.json",
    "advisor_prompt_patches": BASE / "advisor_prompt_patches.json",
    "coverage_report": BASE / "advisor_mapping_coverage_report.csv",
}

for name, path in paths.items():
    print(name, path, path.exists(), path.stat().st_size if path.exists() else 0)

row_df = pd.read_csv(paths["row_advisor_mapping"])
print("\nrows:", len(row_df))
display(row_df.head(10))

print("\nprimary_advisor_family counts")
display(row_df["primary_advisor_family"].value_counts(dropna=False).head(30))

print("\nfailure_reason_bases counts")
from collections import Counter
import json
import pandas as pd

reason_counter = Counter()

for x in row_df.get("normalized_signals", []):
    try:
        vals = json.loads(x) if isinstance(x, str) else x
    except Exception:
        vals = []
    for v in vals or []:
        reason_counter[str(v)] += 1

display(pd.DataFrame(reason_counter.most_common(50), columns=["signal", "count"]))


print("\nexpanded_verified_patches")
expanded = json.loads(paths["expanded_verified_patches"].read_text(encoding="utf-8"))
print(json.dumps(expanded, ensure_ascii=False, indent=2)[:8000])

print("\nadvisor_prompt_patches")
patches = json.loads(paths["advisor_prompt_patches"].read_text(encoding="utf-8"))
print(json.dumps(patches, ensure_ascii=False, indent=2)[:8000])

patch_df = pd.DataFrame(advisor_prompt_patch_payload["prompt_patches"])

display(patch_df[[
    "patch_id",
    "target_block_family",
    "target_block_id",
    "priority",
    "evidence_diagnostics",
    "patch_text",
]])

display(
    patch_df[["target_block_family", "target_block_id"]]
    .drop_duplicates()
    .sort_values(["target_block_family", "target_block_id"])
)

print(json.dumps(advisor_prompt_patch_validation, ensure_ascii=False, indent=2))

row_advisor_mapping /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/row_advisor_mapping.csv True 2075162
advisor_rich_feedback /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_rich_feedback.json True 1467453
expanded_verified_patches /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/expanded_verified_patches.json True 8583
advisor_prompt_patches /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping

,row_no,category,command_eng,command_kor,det_score,det_pass,gt_exact,gt_similarity,branch,generation_state_class,generation_state,failure_reasons,normalized_signals,primary_advisor_family,advisor_families,target_block_ids,concrete_diagnostics,recommended_prompt_mutations,severity_score,generated_collapse_count,gt_period,generated_period,gt_cron,generated_cron,gt_services,generated_services,gt_receivers,generated_receivers,gt_numeric_literals,generated_numeric_literals,gt_string_args,generated_string_args,gt_code,generated_code,gt_json,generated_json,diff_summary,cloud_auxiliary,raw_response_path,prompt_log_paths
0,218,7,"Every hour from midnight to 5 AM, if at least one door is open, turn all hallway lights to 50%.","자정부터 오전 5시까지 1시간마다 체크해서 문이 하나라도 열려있으면, 복도의 조명을 모두 50%로 켜줘.",10.0350,False,False,0.167832,strict_det_concrete_mismatch,valid_json_nonempty,"{""class"": ""valid_json_nonempty"", ""generation_error_type"": """", ""generation_error_count"": 0, ""raw_response_path"": ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_r...","[""cron_mismatch"", ""period_mismatch"", ""gt_mismatch"", ""gt_service_coverage"", ""unknown_service"", ""gt_receiver_coverage"", ""numeric_grounding"", ""enum_grounding""]","[""output_collapse"", ""unknown_service"", ""gt_receiver_coverage"", ""gt_service_coverage"", ""cron_mismatch"", ""period_mismatch"", ""numeric_grounding"", ""enum_grounding"", ""gt_mismatch""]",Skeleton,"[""Skeleton"", ""Service_Mapping"", ""Receiver_Tag_Preservation"", ""Temporal_Rule"", ""Numeric_Unit_Grounding"", ""Enum_Grounding"", ""DET_Helper""]","[""06"", ""02""]","[""output_collapse: no specialized diagnostic rule; compare gt_code and generated_code"", ""service mismatch: gt services=['door_doorstate', 'levelcontrol_movetolevel'], generated services=['dishwasherMode_setDishwasherMode']; generated service 'dishwasherMode_setDishwasherMode' looks camelCase/cla...","[{""source_signal"": ""output_collapse"", ""target_block_id"": ""06"", ""target_block_family"": ""Skeleton"", ""suggested_mutation_type"": ""row independence and command-specific receiver/service selection"", ""micro_rule"": ""Solve each row independently from the current command. Never reuse a previous row's JSON...",187.9650,280,0,-1,0 0-5 * * *,NaN,"[""door_doorstate"", ""levelcontrol_movetolevel""]","[""dishwasherMode_setDishwasherMode""]","[""#Hallway#Light"", ""all(#Door""]","[""#Dishwasher""]","[""50"", ""0""]",[],"[""open""]","[""dry""]","if (all(#Door).door_doorstate ==| ""open"") {\n\n all(#Hallway #Light).levelcontrol_movetolevel(50, 0)\n\n}","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")","{""name"": """", ""cron"": ""0 0-5 * * *"", ""period"": 0, ""script"": ""\nif (all(#Door).door_doorstate ==| \""open\"") {\n\n all(#Hallway #Light).levelcontrol_movetolevel(50, 0)\n\n}""}","{""name"": ""DishwasherDryMode"", ""cron"": """", ""period"": -1, ""code"": ""(#Dishwasher).dishwasherMode_setDishwasherMode(\""dry\"")""}","{""gt_services"": [""door_doorstate"", ""levelcontrol_movetolevel""], ""generated_services"": [""dishwasherMode_setDishwasherMode""], ""gt_receivers"": [""#Hallway#Light"", ""all(#Door""], ""generated_receivers"": [""#Dishwasher""], ""gt_numeric_literals"": [""50"", ""0""], ""generated_numeric_literals"": [], ""gt_string_ar...",{},/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/raw_responses/row_218_cand_0_gen_000.json,"[""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/prompts/row_218_cand_0_gen_000.md"", ""/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/clou..."
1,222,7,"Every morning at 8 AM, make the speaker spea


primary_advisor_family counts


primary_advisor_family
Skeleton    280
Name: count, dtype: int64


failure_reason_bases counts


,signal,count
0,output_collapse,280
1,unknown_service,280
2,gt_service_coverage,280
3,period_mismatch,280
4,gt_mismatch,280
5,gt_receiver_coverage,279
6,numeric_grounding,161
7,enum_grounding,161
8,cron_mismatch,40



expanded_verified_patches
{
  "schema_version": "v1",
  "source_advisor_rich_feedback": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_rich_feedback.json",
  "official_metric": "strict_det",
  "cloud_is_auxiliary": true,
  "mutated_blocks": [
    {
      "target_block_id": "06",
      "target_block_family": "Skeleton",
      "rationale": "Expanded Skeleton rule from 280 empirical row(s). Representative diagnostics: output collapse: same generated code reused across unrelated rows.",
      "suggested_patch_text": "Solve each dataset row independently from the current command only. Never reuse a previous row's JSON name, receiver, service, enum argument, numeric argument, or code skeleton.",
      "verification": {
        "localized_patch_only": true,
        "uses_concrete_diagnostics": true,
        "imperative_rule": true,
  

,patch_id,target_block_family,target_block_id,priority,evidence_diagnostics,patch_text
0,row_advisor_001_Skeleton_06,Skeleton,06,99,[output collapse: same generated code reused across unrelated rows.],"Solve each dataset row independently from the current command only. Never reuse a previous row's JSON name, receiver, service, enum argument, numeric argument, or code skeleton."
1,row_advisor_002_Service_Mapping_02,Service_Mapping,02,98,[service mismatch: expected canonical schema service but generated service differs or is hallucinated.],"Never invent service/member names. Copy the exact canonical device-prefixed service member from the injected schema, preserving lowercase, underscores, and device prefix. If a generated member looks camelCase, class-style, capitalized, or paraphrased, replace it with the nearest schema-valid can..."
2,row_advisor_003_Temporal_Rule_06,Temporal_Rule,06,97,[schedule mismatch: cron/period policy differs from GT.],"Classify schedule type before writing JSON: one-shot action, fixed cron trigger, repeated period loop, delay sequence, or trigger-then-repeat. Use period=0 for one-shot or scheduled one-shot commands, preserve explicit cron triggers, and use positive period only when repeated monitoring is expli..."
3,row_advisor_004_DET_Helper_06,DET_Helper,06,96,[overall GT mismatch: perform component-wise final verification.],"When code is schema-valid but not target-equivalent, compare schedule, receiver, service, numeric, enum, dataflow, and action order before final output. Verify schedule, receiver, service, numeric, enum, dataflow, and output schema before final JSON."
4,row_advisor_005_Receiver_Tag_Preservation_02,Receiver_Tag_Preservation,02,95,[receiver mismatch: generated receiver differs from command/GT receiver.],Select the receiver tag from the current command target before choosing any service. Preserve owner/location/group/sector tags exactly and choose only services attached to that receiver; never reuse a receiver from a previous row.
5,row_advisor_006_Numeric_Unit_Grounding_06,Numeric_Unit_Grounding,06,94,[numeric mismatch: required numeric literal or converted unit is missing or incorrect.],"Preserve every numeric literal required by the current command and bind it to the selected service argument. Convert units using the service descriptor, such as minutes to seconds for seconds-based arguments, and never drop numeric thresholds or durations."
6,row_advisor_007_Enum_Grounding_02,Enum_Grounding,02,93,[enum/string mismatch: generated enum argument differs from selected service descriptor.],"For enum-valued services, copy the allowed enum string exactly from the selected service descriptor. Do not translate, paraphrase, or borrow enum values from another device or previous row."


,target_block_family,target_block_id
3,DET_Helper,06
6,Enum_Grounding,02
5,Numeric_Unit_Grounding,06
4,Receiver_Tag_Preservation,02
1,Service_Mapping,02
0,Skeleton,06
2,Temporal_Rule,06


{
  "ok": true,
  "errors": [],
  "warnings": []
}


In [100]:
# ============================================================
# Cell 22. Optional patch apply for row-advisor prompt patches
# ============================================================

#RUN_ROW_ADVISOR_PATCH_APPLY = os.environ.get("RUN_ROW_ADVISOR_PATCH_APPLY", "false").lower() == "true"
RUN_ROW_ADVISOR_PATCH_APPLY = "true"
row_advisor_patch_apply_dir = None

if "advisor_prompt_patch_path" not in globals():
    advisor_prompt_patch_path = ROW_ADVISOR_OUT_DIR / "advisor_prompt_patches.json"

if not Path(advisor_prompt_patch_path).exists():
    print("Patch apply skipped: advisor_prompt_patches.json not found.")
    print("Run Cell 21 first.")
elif not RUN_ROW_ADVISOR_PATCH_APPLY:
    print("Patch apply skipped by default.")
    print("Set RUN_ROW_ADVISOR_PATCH_APPLY=true after reviewing:")
    print(" -", advisor_prompt_patch_path)
else:
    validation = globals().get("advisor_prompt_patch_validation", {})
    if validation and not validation.get("ok", False):
        print("Patch apply skipped because validation has errors:")
        print(json.dumps(validation, ensure_ascii=False, indent=2))
    else:
        row_advisor_patch_apply_dir = run_patch_apply(
            f"row_advisor_patch_apply_{ts()}",
            advisor_prompt_patch_path,
        )
        print("row_advisor_patch_apply_dir:", row_advisor_patch_apply_dir)
        show_artifact_table([
            row_advisor_patch_apply_dir / "patched_genome.json",
            row_advisor_patch_apply_dir / "patch_application_report.json",
            row_advisor_patch_apply_dir / "patch_diff.md",
            row_advisor_patch_apply_dir / "patched_prompt_preview.md",
        ])
        show_file(row_advisor_patch_apply_dir / "patch_application_report.json", max_chars=6000)
        show_file(row_advisor_patch_apply_dir / "patch_diff.md", max_chars=6000)


[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.prompt_patch_apply --prompt-patches /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_prompt_patches.json --out-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/row_advisor_patch_apply_20260625_115411.log
{
  "created_at": "2026-06-25T02:54:11.112890+00:00",
  "prompt_patches_path": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_prompt_patc

,path,exists,size
0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patched_genome.json,True,2428
1,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patch_application_report.json,True,4536
2,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patch_diff.md,True,659
3,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patched_prompt_preview.md,True,1835


/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patch_application_report.json exists= True size= 4536
{
  "created_at": "2026-06-25T02:54:11.112890+00:00",
  "prompt_patches_path": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_prompt_patches.json",
  "base_genome_path": "fallback",
  "patched_genome_path": "/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patched_genome.json",
  "accepted_proposal_count": 7,
  "advisor_child_scheduled_count": 1,
  "advisor_backed_diff_count": 7,
  "patch_count": 7,
  "visible_patch_ids": [
    "row_advisor_001_Skeleton_06",
    "row_advisor_002_Service_Mapping_02",
    "row_advisor_003_Tem

In [101]:
show_file(row_advisor_patch_apply_dir / "patch_diff.md", max_chars=30000)
show_file(row_advisor_patch_apply_dir / "patched_prompt_preview.md", max_chars=60000)

/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_115411/patch_diff.md exists= True size= 659
# Prompt Patch Diff

## Applied Patches
- `row_advisor_001_Skeleton_06` op=append_micro_rule block=06 changed=True
- `row_advisor_002_Service_Mapping_02` op=append_micro_rule block=02 changed=True
- `row_advisor_003_Temporal_Rule_06` op=append_micro_rule block=06 changed=True
- `row_advisor_004_DET_Helper_06` op=append_micro_rule block=06 changed=True
- `row_advisor_005_Receiver_Tag_Preservation_02` op=append_micro_rule block=02 changed=True
- `row_advisor_006_Numeric_Unit_Grounding_06` op=append_micro_rule block=06 changed=True
- `row_advisor_007_Enum_Grounding_02` op=append_micro_rule block=02 changed=True

## Visibility
- visible_patch_count: `7`


/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/row_advisor_patch_apply_20260625_1154

## 23. Final advisor mapping self-check and expected output

이 cell은 최종적으로 다음을 확인합니다.

- `cron_mismatch` 같은 에러가 fallback 없이 advisor prompt에 붙을 수 있는지
- `gt_mismatch`가 최소 `DET_Helper / Block 06`로 들어가는지
- 각 error가 diagnostic, target/action, advisor rule을 갖는지
- Excel / JSON / patch artifacts가 모두 생성되었는지

In [102]:
# ============================================================
# Cell 23. Final self-check and expected output preview
# ============================================================

def advisor_rule_preview_for_signal(signal):
    sig = normalize_signal(signal)
    tax = taxonomy_for_signal_strict(sig) if "taxonomy_for_signal_strict" in globals() else taxonomy_for_signal(sig)
    return {
        "error": sig,
        "diagnostic": tax.get("diagnostic_template") or f"{sig}: fallback diagnostic generated from taxonomy.",
        "target_action": f"[Block {tax.get('target_block_id', '06')}] - {tax.get('target_family', 'DET_Helper')}",
        "advisor_rule": tax.get("micro_rule") or DEFAULT_TAXONOMY_ROW["micro_rule"],
        "mutation_policy": tax.get("mutation_policy", ""),
        "mapped": sig in FAILURE_ADVISOR_MAP,
    }

preview_signals = [
    "cron_mismatch",
    "period_mismatch",
    "gt_mismatch",
    "gt_service_coverage",
    "unknown_service",
    "gt_receiver_coverage",
    "numeric_grounding",
    "enum_grounding",
    "dataflow",
    "missing_required_key:code",
    "missing_generated_code",
    "generation_empty_output",
    "generation_cuda_oom",
    "candidate_extraction_failure",
    "output_collapse",
]

preview_rows = [advisor_rule_preview_for_signal(s) for s in preview_signals]
preview_df = pd.DataFrame(preview_rows)
display(preview_df)

problems = []
for row in preview_rows:
    if not str(row.get("diagnostic", "")).strip():
        problems.append(f"{row['error']}: empty diagnostic")
    if not str(row.get("target_action", "")).strip():
        problems.append(f"{row['error']}: empty target action")
    if not str(row.get("advisor_rule", "")).strip():
        problems.append(f"{row['error']}: empty advisor rule")

expected_files = [
    globals().get("row_advisor_csv", ROW_ADVISOR_OUT_DIR / "row_advisor_mapping.csv"),
    globals().get("row_advisor_jsonl", ROW_ADVISOR_OUT_DIR / "row_advisor_mapping.jsonl"),
    globals().get("advisor_rich_feedback_json", ROW_ADVISOR_OUT_DIR / "advisor_rich_feedback.json"),
    globals().get("taxonomy_csv", ROW_ADVISOR_OUT_DIR / "failure_taxonomy_table.csv"),
    globals().get("expanded_verified_patches_json", ROW_ADVISOR_OUT_DIR / "expanded_verified_patches.json"),
    globals().get("advisor_prompt_patch_path", ROW_ADVISOR_OUT_DIR / "advisor_prompt_patches.json"),
    globals().get("coverage_csv", ROW_ADVISOR_OUT_DIR / "advisor_mapping_coverage_report.csv"),
]

print("Expected artifacts:")
show_artifact_table(expected_files)

if problems:
    print("SELF-CHECK: FAIL")
    for p in problems:
        print(" -", p)
    raise AssertionError("Advisor mapping self-check failed")
else:
    print("SELF-CHECK: PASS")
    print("Every previewed error has diagnostic, target/action, and advisor rule.")

print("\nNext-day workflow:")
print("1. Open row_advisor_mapping.csv in Excel.")
print("2. Review concrete_diagnostics and recommended_prompt_mutations.")
print("3. Review advisor_prompt_patches.json.")
print("4. If accepted, set RUN_ROW_ADVISOR_PATCH_APPLY=true and run Cell 22.")
print("5. Rerun selected failed rows using Cell 14 with PATCHED_GENOME_JSON.")

,error,diagnostic,target_action,advisor_rule,mutation_policy,mapped
0,cron_mismatch,"cron mismatch: gt cron={gt_cron!r}, generated cron={generated_cron!r}. Fixed wall-clock schedule was lost.",[Block 06] - Temporal_Rule,"For explicit fixed times, weekdays, midnight, or scheduled one-shot commands, derive cron first and preserve it exactly. Do not replace a fixed schedule with a period loop or a Clock guard unless repeated monitoring is explicit.",fixed wall-clock/day schedule uses cron first; do not replace with period + Clock guard,True
1,period_mismatch,"period mismatch: gt period={gt_period!r}, generated period={generated_period!r}.",[Block 06] - Temporal_Rule,"For one-shot action or scheduled one-shot commands, use period=0 unless repeated monitoring is explicit. Use positive period only for repeated monitoring loops and never use -1 as a substitute for a valid one-shot period.","classify one-shot / cron / repeated loop first, then apply period policy",True
2,gt_mismatch,gt mismatch: code is schema-valid but not target-equivalent; prioritize concrete mismatches.,[Block 06] - DET_Helper,"When code is schema-valid but not target-equivalent, compare schedule, receiver, service, numeric, enum, dataflow, and action order before final output.",umbrella only; do not make primary patch if concrete service/receiver/schedule/numeric/enum reason exists,True
3,gt_service_coverage,"missing GT service: gt services={gt_services}, generated services={generated_services}.",[Block 02] - Service_Mapping,Include every service implied by the command. Select services only from the injected schema under the selected receiver and do not substitute adjacent service families.,"select command target receiver first, then select schema-valid service under that receiver",True
4,unknown_service,unknown/canonical service error: generated services={generated_services}; expected schema/GT services={gt_services}.,[Block 02] - Service_Mapping,"Never invent service/member names. Copy the canonical device-prefixed service member exactly from the injected service schema. Do not emit camelCase, class-style, capitalized, or paraphrased service names.",replace non-schema member with nearest valid canonical schema member before final output,True
5,gt_receiver_coverage,"receiver mismatch: gt receivers={gt_receivers}, generated receivers={generated_receivers}.",[Block 02] - Receiver_Tag_Preservation,"Select receiver tags from the current command target before service selection. Preserve owner, location, group, and sector tags exactly, and do not reuse a receiver from another row.",owner/location/group/sector tag preservation; condition receiver and action receiver may differ,True
6,numeric_grounding,"numeric mismatch: gt numeric literals={gt_numeric_literals}, generated numeric literals={generated_numeric_literals}.",[Block 06] - Numeric_Unit_Grounding,"Preserve required numeric arguments and thresholds from the command. Convert units using the selected service descriptor, such as minutes to seconds for seconds-based arguments.",temporal numbers and service argument numbers both use descriptor-grounded conversion,True
7,enum_grounding,"enum mismatch: gt string args={gt_string_args}, generated string args={generated_string_args}.",[Block 02] - Enum_Grounding,"For enum-valued services, copy the allowed enum value exactly from the selected service descriptor. Do not translate, paraphrase, or borrow enum values from another device or service.",copy allowed enum from selected service descriptor only,True
8,dataflow,dataflow mismatch: generated does not preserve read-bind-use structure required by GT.,[Block 06] - Dataflow,"When reading a value for reporting or control, bind it with JOILang ':=' and use that bound value downstream. Do not replace read-bind-use flow with an unrelated direct action.",sensor read → variable bind → downstream speak/action structure preservation,True
9,missing_required_key,missing required JSON key: {reason}.,[Block 03] - Output_Schema,"Always 

Expected artifacts:


,path,exists,size
0,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/row_advisor_mapping.csv,True,2075162
1,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/row_advisor_mapping.jsonl,True,2302408
2,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_rich_feedback.json,True,1467453
3,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/failure_taxonomy_table.csv,True,16188
4,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/expanded_verified_patches.json,True,8583
5,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_prompt_patches.json,True,12686
6,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_mapping_coverage_report.csv,True,2989


SELF-CHECK: PASS
Every previewed error has diagnostic, target/action, and advisor rule.

Next-day workflow:
1. Open row_advisor_mapping.csv in Excel.
2. Review concrete_diagnostics and recommended_prompt_mutations.
3. Review advisor_prompt_patches.json.
4. If accepted, set RUN_ROW_ADVISOR_PATCH_APPLY=true and run Cell 22.
5. Rerun selected failed rows using Cell 14 with PATCHED_GENOME_JSON.


In [103]:
# ============================================================
# Cell 24. Patch apply utility + merged JOILang prompt generation naming
# ============================================================

import os
import re
import json
import shutil
from pathlib import Path
from datetime import datetime
import pandas as pd

def _now_tag():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def _safe_name(x):
    x = str(x or "").strip()
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x.strip("_") or "unnamed"

def _read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8"))

def _write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def _notebook_root():
    if "NB_ROOT" in globals():
        return Path(NB_ROOT)
    if "ROW_ADVISOR_OUT_DIR" in globals():
        return Path(ROW_ADVISOR_OUT_DIR).parent
    return Path.cwd() / "artifacts" / "ga_search_tutorial_runs" / f"manual_{_now_tag()}"

MERGED_PROMPT_ROOT = _notebook_root() / "merged_joilang_code_prompts"
MERGED_PROMPT_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_BLOCK_BY_FAMILY = {
    "Service_Mapping": "02",
    "Receiver_Tag_Preservation": "02",
    "Enum_Grounding": "02",
    "Argument_Grounding": "02",
    "Output_Schema": "03",
    "Parser_Extraction": "03",
    "Temporal_Rule": "06",
    "Numeric_Unit_Grounding": "06",
    "Dataflow": "06",
    "Intent_Fulfillment": "06",
    "Skeleton": "06",
    "DET_Helper": "06",
    "Minimality": "06",
}

NON_INJECTABLE_FAMILIES = {
    "Generation_Health",
    "Runtime_Health",
    "Prompt_Budget",
    "No_Mutation",
    "Parser_Extraction",
}

def evidence_matches_family(family, diagnostics):
    text = " ".join(str(x) for x in diagnostics).lower()
    expected = {
        "Service_Mapping": ["service", "canonical", "camelcase", "class-style"],
        "Receiver_Tag_Preservation": ["receiver"],
        "Temporal_Rule": ["period", "cron", "schedule"],
        "Numeric_Unit_Grounding": ["numeric"],
        "Enum_Grounding": ["enum", "string"],
        "Skeleton": ["collapse", "independently", "skeleton"],
        "DET_Helper": ["gt mismatch", "component", "compare"],
        "Output_Schema": ["json", "required", "schema"],
        "Dataflow": ["dataflow", "read-bind-use", "bind"],
        "Intent_Fulfillment": ["empty", "non-empty", "behavior", "action"],
    }
    keys = expected.get(family, [])
    return not keys or any(k in text for k in keys)

def validate_patch_payload_strict(payload, *, require_injectable=True):
    errors = []
    warnings = []
    patches = payload.get("prompt_patches", []) or []

    injectable = [
        p for p in patches
        if str(p.get("target_block_id", "")) not in {"", "00"}
        and str(p.get("target_block_family", "")) not in NON_INJECTABLE_FAMILIES
    ]

    if require_injectable and not injectable:
        errors.append("No injectable semantic/schema prompt patch was generated.")

    for i, p in enumerate(patches):
        bid = str(p.get("target_block_id", ""))
        fam = str(p.get("target_block_family", ""))
        text = str(p.get("patch_text", ""))
        diags = p.get("evidence_diagnostics", []) or []

        if not bid:
            errors.append(f"patch[{i}] has empty target_block_id")
        if not fam:
            errors.append(f"patch[{i}] has empty target_block_family")
        if bid == "00":
            errors.append(f"patch[{i}] targets non-injectable block 00")
        if fam in NON_INJECTABLE_FAMILIES:
            errors.append(f"patch[{i}] is not a JOICode semantic/schema prompt repair family: {fam}")
        if not text.strip():
            errors.append(f"patch[{i}] has empty patch_text")
        if "```" in text:
            errors.append(f"patch[{i}] contains markdown fence")

        expected_bid = EXPECTED_BLOCK_BY_FAMILY.get(fam)
        if expected_bid and bid != expected_bid:
            errors.append(
                f"patch[{i}] family/block mismatch: {fam} expected Block {expected_bid}, got Block {bid}"
            )

        if not evidence_matches_family(fam, diags):
            warnings.append(
                f"patch[{i}] evidence_diagnostics may not match family={fam}: {diags[:2]}"
            )

        if len(text) > 900:
            warnings.append(f"patch[{i}] is long: {len(text)} chars")

    return {"ok": not errors, "errors": errors, "warnings": warnings, "patch_count": len(patches)}

def next_prompt_generation_id(variant):
    variant = _safe_name(variant)
    existing = sorted(MERGED_PROMPT_ROOT.glob(f"merged_joilang_code_prompt_{variant}_gen*.md"))
    gens = []
    for p in existing:
        m = re.search(r"_gen(\d+)\.md$", p.name)
        if m:
            gens.append(int(m.group(1)))
    return (max(gens) + 1) if gens else 1

def copy_merged_prompt_artifacts(apply_dir, variant, gen_id, patch_path):
    """
    Copy patch apply outputs to stable merged_joilang_code_prompt_{variant}_genXX.* names.
    """
    apply_dir = Path(apply_dir)
    variant = _safe_name(variant)
    gen_label = f"gen{gen_id:02d}"

    merged_prompt = MERGED_PROMPT_ROOT / f"merged_joilang_code_prompt_{variant}_{gen_label}.md"
    merged_diff = MERGED_PROMPT_ROOT / f"merged_joilang_code_prompt_{variant}_{gen_label}_diff.md"
    merged_genome = MERGED_PROMPT_ROOT / f"merged_joilang_code_prompt_{variant}_{gen_label}_patched_genome.json"
    merged_report = MERGED_PROMPT_ROOT / f"merged_joilang_code_prompt_{variant}_{gen_label}_patch_report.json"
    merged_manifest = MERGED_PROMPT_ROOT / f"merged_joilang_code_prompt_{variant}_{gen_label}_manifest.json"

    src_prompt = apply_dir / "patched_prompt_preview.md"
    src_diff = apply_dir / "patch_diff.md"
    src_genome = apply_dir / "patched_genome.json"
    src_report = apply_dir / "patch_application_report.json"

    if src_prompt.exists():
        shutil.copy2(src_prompt, merged_prompt)
    if src_diff.exists():
        shutil.copy2(src_diff, merged_diff)
    if src_genome.exists():
        shutil.copy2(src_genome, merged_genome)
    if src_report.exists():
        shutil.copy2(src_report, merged_report)

    manifest = {
        "variant": variant,
        "generation": gen_id,
        "gen_label": gen_label,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "patch_path": str(patch_path),
        "apply_dir": str(apply_dir),
        "merged_prompt": str(merged_prompt),
        "merged_diff": str(merged_diff),
        "merged_genome": str(merged_genome),
        "merged_report": str(merged_report),
    }
    _write_json(merged_manifest, manifest)

    return {
        "manifest": merged_manifest,
        "merged_prompt": merged_prompt,
        "merged_diff": merged_diff,
        "merged_genome": merged_genome,
        "merged_report": merged_report,
    }

def apply_patch_payload_with_generation(patch_path, variant, genome_json=None, label=None):
    """
    Validate advisor_prompt_patches JSON, apply it, and save merged_..._genXX prompt artifacts.
    Requires existing run_patch_apply() from earlier notebook cells.
    """
    patch_path = Path(patch_path)
    payload = _read_json(patch_path)
    validation = validate_patch_payload_strict(payload)

    print("patch_path:", patch_path)
    print("validation:")
    print(json.dumps(validation, ensure_ascii=False, indent=2))

    if not validation.get("ok", False):
        raise RuntimeError("Patch payload validation failed. Do not apply.")

    if "run_patch_apply" not in globals():
        raise RuntimeError("run_patch_apply() is not defined. Run setup/helper cells first.")

    variant = _safe_name(variant)
    gen_id = next_prompt_generation_id(variant)
    label = label or f"patch_apply_{variant}_gen{gen_id:02d}_{_now_tag()}"

    apply_dir = run_patch_apply(
        label,
        patch_path,
        genome_json=genome_json,
    )

    copied = copy_merged_prompt_artifacts(apply_dir, variant, gen_id, patch_path)

    print("apply_dir:", apply_dir)
    print("merged prompt:", copied["merged_prompt"])
    print("merged genome:", copied["merged_genome"])
    print("merged diff:", copied["merged_diff"])
    print("manifest:", copied["manifest"])

    return {
        "variant": variant,
        "generation": gen_id,
        "apply_dir": Path(apply_dir),
        "patch_path": patch_path,
        **copied,
    }

print("MERGED_PROMPT_ROOT:", MERGED_PROMPT_ROOT)

MERGED_PROMPT_ROOT: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/merged_joilang_code_prompts


In [104]:
# ============================================================
# Cell 25. Apply DET-rulebased patch and inspect merged JOILang prompt
# ============================================================

DET_RULEBASED_PATCH_PATH = Path(ROW_ADVISOR_OUT_DIR) / "advisor_prompt_patches.json"
assert DET_RULEBASED_PATCH_PATH.exists(), DET_RULEBASED_PATCH_PATH

det_rulebased_apply = apply_patch_payload_with_generation(
    DET_RULEBASED_PATCH_PATH,
    variant="det_rulebased",
)

DET_RULEBASED_PATCHED_GENOME = det_rulebased_apply["merged_genome"]
DET_RULEBASED_MERGED_PROMPT = det_rulebased_apply["merged_prompt"]

print("DET_RULEBASED_PATCHED_GENOME:", DET_RULEBASED_PATCHED_GENOME)
print("DET_RULEBASED_MERGED_PROMPT:", DET_RULEBASED_MERGED_PROMPT)

print("\n=== patch_diff.md ===")
show_file(det_rulebased_apply["merged_diff"], max_chars=30000)

print("\n=== patched_prompt_preview.md / merged prompt ===")
show_file(det_rulebased_apply["merged_prompt"], max_chars=60000)

patch_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_prompt_patches.json
validation:
{
  "ok": true,
  "errors": [],
  "warnings": [],
  "patch_count": 7
}

[CMD]
/home/mgjeong/miniconda3/envs/joi/bin/python3.10 -m utils.ga_search.prompt_patch_apply --prompt-patches /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/full_det_qwen25_coder_7b_allrows_20260624_021859/qwen25_coder_7b/row_advisor_mapping/advisor_prompt_patches.json --out-dir /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_det_rulebased_gen01_20260625_120203
[LOG] /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patch_apply_det_rulebased_gen01_20260625_120203/patc

In [105]:
# ============================================================
# Cell 26. Rerun selected failed rows with patched genome
# ============================================================

def parse_row_list(value, default_rows=None):
    if value is None or str(value).strip() == "":
        return list(default_rows or [])
    if isinstance(value, (list, tuple)):
        return [int(x) for x in value]
    out = []
    for part in str(value).replace(";", ",").split(","):
        part = part.strip()
        if not part:
            continue
        out.append(int(part))
    return out

def default_failed_rows_from_mapping(n=10):
    csv_path = Path(ROW_ADVISOR_OUT_DIR) / "row_advisor_mapping.csv"
    if not csv_path.exists():
        return [218, 222, 195, 251, 211]
    df = pd.read_csv(csv_path)
    if "severity_score" in df.columns:
        df = df.sort_values("severity_score", ascending=False)
    return df["row_no"].astype(int).drop_duplicates().head(n).tolist()

FAILED_ROWS_TO_RERUN = parse_row_list(
    os.environ.get("FAILED_ROWS_TO_RERUN", ""),
    default_rows=default_failed_rows_from_mapping(n=8),
)

print("FAILED_ROWS_TO_RERUN:", FAILED_ROWS_TO_RERUN)

def _cli_eval_one_row_with_genome(row_no, genome_json, variant, model_key=None, llm_extra_json_path=None, timeout_sec=1800):
    """
    Direct canonical CLI eval fallback.
    Use JSON string for --llm-extra-json because this CLI expects JSON payload, not a file path.
    """
    model_key = model_key or globals().get("MODEL_KEY", globals().get("MODEL_KEY_7B", "qwen25_coder_7b"))
    out_dir = _notebook_root() / "patched_eval_runs" / f"{_safe_name(variant)}_row{int(row_no):03d}_{_now_tag()}"
    out_dir.mkdir(parents=True, exist_ok=True)

    py = str(globals().get("PY", globals().get("JOI_PY", "python")))
    model = str(globals().get("MODEL", "gpt_mg.version0_13"))
    dataset = str(globals().get("DATASET"))
    service_schema = str(globals().get("SERVICE_SCHEMA"))

    cmd = [
        py, "-m", "utils.ga_search.cli", "eval",
        "--model", model,
        "--dataset", dataset,
        "--service-schema", service_schema,
        "--llm-mode", "worker",
        "--engine-mode", "real",
        "--model-key", model_key,
        "--det-profile", "strict",
        "--det-threshold", "70",
        "--out-dir", str(out_dir),
        "--print-mode", "summary",
        "--row-no", str(int(row_no)),
        "--genome-json", str(genome_json),
    ]

    if llm_extra_json_path is not None:
        extra_text = Path(llm_extra_json_path).read_text(encoding="utf-8")
        cmd += ["--llm-extra-json", extra_text]

    rc, out = run_cmd(
        cmd,
        log_path=out_dir / f"eval_row{int(row_no):03d}_{_safe_name(variant)}.log",
        check=False,
        timeout_sec=timeout_sec,
    )
    return out_dir, rc

def get_worker_extra_for_rerun(row_no, model_key=None):
    """
    Reuse notebook helper worker_extra_json_path() if available.
    """
    model_key = model_key or globals().get("MODEL_KEY", globals().get("MODEL_KEY_7B", "qwen25_coder_7b"))

    max_new_tokens = globals().get("MAX_NEW_TOKENS", globals().get("MAX_NEW_TOKENS_7B", 1024))

    if "worker_extra_json_path" in globals():
        return worker_extra_json_path(
            label=f"worker_extra_patched_row{int(row_no):03d}_{model_key}_{_now_tag()}",
            model_key=model_key,
            max_new_tokens=max_new_tokens,
        )

    print("[WARN] worker_extra_json_path() not found. Running without --llm-extra-json.")
    return None

def rerun_rows_with_patched_genome(genome_json, variant, row_nos, model_key=None, timeout_sec=1800):
    results = []
    for row_no in row_nos:
        print("\n" + "=" * 100)
        print(f"RERUN row={row_no}, variant={variant}, genome={genome_json}")
        extra = get_worker_extra_for_rerun(row_no, model_key=model_key)
        out_dir, rc = _cli_eval_one_row_with_genome(
            row_no=row_no,
            genome_json=genome_json,
            variant=variant,
            model_key=model_key,
            llm_extra_json_path=extra,
            timeout_sec=timeout_sec,
        )
        results.append({
            "row_no": int(row_no),
            "variant": variant,
            "out_dir": str(out_dir),
            "rc": rc,
        })
        print("rc:", rc)
        print("out_dir:", out_dir)
        try:
            row_summary(out_dir)
            show_gt_vs_generated(out_dir, row_no=int(row_no), max_rows=5)
        except Exception as e:
            print("[WARN] display failed:", repr(e))

    result_df = pd.DataFrame(results)
    result_path = _notebook_root() / "patched_eval_runs" / f"rerun_index_{_safe_name(variant)}_{_now_tag()}.csv"
    result_path.parent.mkdir(parents=True, exist_ok=True)
    result_df.to_csv(result_path, index=False, encoding="utf-8-sig")
    print("rerun index:", result_path)
    display(result_df)
    return result_df

RUN_DET_RULEBASED_RERUN = os.environ.get("RUN_DET_RULEBASED_RERUN", "false").lower() == "true"

det_rulebased_rerun_df = None
if RUN_DET_RULEBASED_RERUN:
    det_rulebased_rerun_df = rerun_rows_with_patched_genome(
        DET_RULEBASED_PATCHED_GENOME,
        variant="det_rulebased",
        row_nos=FAILED_ROWS_TO_RERUN,
        model_key=globals().get("MODEL_KEY", globals().get("MODEL_KEY_7B", "qwen25_coder_7b")),
        timeout_sec=1800,
    )
else:
    print("Rerun skipped by default.")
    print('Set os.environ["RUN_DET_RULEBASED_RERUN"]="true" and rerun this cell.')

FAILED_ROWS_TO_RERUN: [218, 222, 195, 186, 187, 251, 211, 219]
Rerun skipped by default.
Set os.environ["RUN_DET_RULEBASED_RERUN"]="true" and rerun this cell.


In [106]:
# ============================================================
# Cell 27. Before/after DET comparison
# ============================================================

def find_row_eval_csv(run_dir):
    run_dir = Path(run_dir)
    candidates = [
        run_dir / "eval" / "row_evaluation.csv",
        run_dir / "row_evaluation.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    hits = list(run_dir.rglob("row_evaluation.csv"))
    return hits[0] if hits else None

def load_eval_rows_from_rerun_index(rerun_df):
    dfs = []
    if rerun_df is None or rerun_df.empty:
        return pd.DataFrame()
    for _, row in rerun_df.iterrows():
        p = find_row_eval_csv(row["out_dir"])
        if p and Path(p).exists():
            df = pd.read_csv(p)
            df["rerun_out_dir"] = row["out_dir"]
            df["variant"] = row["variant"]
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def baseline_eval_df():
    if "ROW_EVALUATION_CSV" in globals() and Path(ROW_EVALUATION_CSV).exists():
        return pd.read_csv(ROW_EVALUATION_CSV)
    p = Path(ROW_ADVISOR_OUT_DIR).parent / "eval" / "row_evaluation.csv"
    if p.exists():
        return pd.read_csv(p)
    raise FileNotFoundError("Cannot locate baseline row_evaluation.csv")

def compare_baseline_vs_variant(rerun_df, label="variant"):
    base = baseline_eval_df()
    after = load_eval_rows_from_rerun_index(rerun_df)

    if after.empty:
        print("No rerun eval rows loaded.")
        return pd.DataFrame()

    keep_base = [
        c for c in [
            "row_no", "category", "det_score", "det_pass", "gt_similarity",
            "failure_reasons", "generated_code", "generated_json"
        ]
        if c in base.columns
    ]
    keep_after = [
        c for c in [
            "row_no", "category", "det_score", "det_pass", "gt_similarity",
            "failure_reasons", "generated_code", "generated_json", "variant", "rerun_out_dir"
        ]
        if c in after.columns
    ]

    merged = base[keep_base].merge(
        after[keep_after],
        on="row_no",
        suffixes=("_before", "_after"),
        how="inner",
    )

    if "det_score_before" in merged.columns and "det_score_after" in merged.columns:
        merged["det_score_delta"] = merged["det_score_after"] - merged["det_score_before"]

    if "det_pass_before" in merged.columns and "det_pass_after" in merged.columns:
        merged["pass_changed"] = merged["det_pass_before"].astype(str) + " -> " + merged["det_pass_after"].astype(str)

    cols = [
        c for c in [
            "row_no", "category_before", "det_score_before", "det_score_after",
            "det_score_delta", "det_pass_before", "det_pass_after", "pass_changed",
            "failure_reasons_before", "failure_reasons_after",
            "generated_code_before", "generated_code_after",
            "rerun_out_dir"
        ]
        if c in merged.columns
    ]

    print(label, "comparison rows:", len(merged))
    display(merged[cols].sort_values("det_score_delta", ascending=False if "det_score_delta" in merged.columns else True))
    return merged

det_rulebased_compare_df = None
if det_rulebased_rerun_df is not None:
    det_rulebased_compare_df = compare_baseline_vs_variant(det_rulebased_rerun_df, label="det_rulebased")
else:
    print("No det_rulebased_rerun_df yet. Run Cell 26 with RUN_DET_RULEBASED_RERUN=true.")

No det_rulebased_rerun_df yet. Run Cell 26 with RUN_DET_RULEBASED_RERUN=true.


In [107]:
# ============================================================
# Cell 28. Build GPT-4.1-mini advisor input package
# ============================================================

GPT_ADVISOR_ROOT = _notebook_root() / "gpt41mini_advisor"
GPT_ADVISOR_ROOT.mkdir(parents=True, exist_ok=True)

def compact_row_advisor_sample(max_rows=40):
    csv_path = Path(ROW_ADVISOR_OUT_DIR) / "row_advisor_mapping.csv"
    df = pd.read_csv(csv_path)

    if "severity_score" in df.columns:
        df = df.sort_values("severity_score", ascending=False)

    cols = [
        c for c in [
            "row_no", "category", "command_eng", "command_kor",
            "det_score", "det_pass", "failure_reasons", "normalized_signals",
            "advisor_families", "concrete_diagnostics",
            "gt_code", "generated_code", "gt_json", "generated_json",
            "diff_summary",
        ]
        if c in df.columns
    ]

    sample = df[cols].head(max_rows).to_dict("records")
    return sample

def build_gpt41mini_advisor_package(round_name="round01", max_rows=40):
    round_name = _safe_name(round_name)

    advisor_payload_path = Path(ROW_ADVISOR_OUT_DIR) / "advisor_rich_feedback.json"
    current_patch_path = Path(ROW_ADVISOR_OUT_DIR) / "advisor_prompt_patches.json"
    taxonomy_path = Path(ROW_ADVISOR_OUT_DIR) / "failure_taxonomy_table.csv"

    package = {
        "task": "Generate improved localized JOILang prompt patches from strict DET row diagnostics.",
        "official_metric": "strict_det",
        "cloud_is_auxiliary": True,
        "target_generator": "local Qwen JOILang code generator",
        "advisor_model": "gpt-4.1-mini",
        "input_files": {
            "advisor_rich_feedback": str(advisor_payload_path),
            "current_det_rulebased_patch": str(current_patch_path),
            "failure_taxonomy_table": str(taxonomy_path),
        },
        "expected_block_by_family": EXPECTED_BLOCK_BY_FAMILY,
        "non_injectable_families": sorted(NON_INJECTABLE_FAMILIES),
        "current_det_rulebased_patch": _read_json(current_patch_path),
        "row_samples": compact_row_advisor_sample(max_rows=max_rows),
        "strict_output_schema": {
            "advisor_meta": {
                "schema_version": "row_advisor_prompt_patches_v1",
                "source": f"gpt-4.1-mini_{round_name}",
                "official_metric": "strict_det",
                "cloud_is_auxiliary": True,
            },
            "prompt_patches": [
                {
                    "patch_id": f"gpt41mini_{round_name}_service_mapping_02",
                    "target_block_family": "Service_Mapping",
                    "target_block_id": "02",
                    "operation": "append_micro_rule",
                    "priority": 100,
                    "patch_text": "localized imperative prompt rule",
                    "rationale": "why this patch is needed based on concrete diagnostics",
                    "evidence_rows": ["row ids"],
                    "evidence_diagnostics": ["diagnostic strings"],
                    "mutation_intent": "gpt41mini_strict_det_prompt_repair",
                }
            ],
        },
    }

    system_prompt = """
You are an expert JOILang prompt advisor.

Your job is to propose localized prompt patches that improve a local JOILang code generator.
Strict DET is the official metric. Cloud reasoning is auxiliary only.
Use concrete row-level diagnostics, not broad semantic guesses.

Rules:
- Output exactly one valid JSON object.
- Do not include markdown fences or prose outside JSON.
- Do not propose Generation_Health, Runtime_Health, Prompt_Budget, No_Mutation, or Block 00 patches.
- Use target_block_id according to expected_block_by_family.
- Prefer concise, imperative, atomic patch_text.
- Preserve evidence_rows and evidence_diagnostics.
- Improve the current deterministic patch only when you can make it more precise.
"""

    user_prompt = f"""
Generate advisor_prompt_patches JSON for JOILang prompt repair.

Use this input package:
{json.dumps(package, ensure_ascii=False, indent=2)}

Return exactly one JSON object with:
- advisor_meta
- prompt_patches

No markdown fences. No commentary.
"""

    input_json = GPT_ADVISOR_ROOT / f"gpt41mini_{round_name}_advisor_input.json"
    prompt_md = GPT_ADVISOR_ROOT / f"gpt41mini_{round_name}_advisor_prompt.md"

    _write_json(input_json, package)
    prompt_md.write_text(
        "# System\n\n" + system_prompt.strip() + "\n\n# User\n\n" + user_prompt.strip(),
        encoding="utf-8",
    )

    print("input_json:", input_json)
    print("prompt_md:", prompt_md)
    show_file(prompt_md, max_chars=30000)

    return {
        "round_name": round_name,
        "input_json": input_json,
        "prompt_md": prompt_md,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }

gpt41mini_round01 = build_gpt41mini_advisor_package(round_name="round01", max_rows=40)

input_json: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/gpt41mini_round01_advisor_input.json
prompt_md: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/gpt41mini_round01_advisor_prompt.md
/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/gpt41mini_round01_advisor_prompt.md exists= True size= 141607
# System

You are an expert JOILang prompt advisor.

Your job is to propose localized prompt patches that improve a local JOILang code generator.
Strict DET is the official metric. Cloud reasoning is auxiliary only.
Use concrete row-level diagnostics, not broad semantic guesses.

Rules:
- Output exactly one valid JSON object.
- Do not include markdown fences or prose outside JSON.
- Do not propose Generation_Health, Runtime_Health, Prompt_Budget

In [111]:
os.environ["RUN_GPT41MINI_API"] = "true"
os.environ["GPT41MINI_MODEL"] = "gpt-4.1-mini"
import os
print(bool(os.environ.get("OPENAI_API_KEY")))

True


In [116]:
gpt41mini_output_obj = extract_json_object(GPT41MINI_PATCH_JSON_TEXT)

GPT41MINI_PATCH_PATH, _ = save_and_validate_advisor_output(
    gpt41mini_output_obj,
    variant="gpt41mini_round01",
)

gpt41mini_apply = apply_patch_payload_with_generation(
    GPT41MINI_PATCH_PATH,
    variant="gpt41mini_round01",
)

GPT41MINI_PATCHED_GENOME = gpt41mini_apply["merged_genome"]
GPT41MINI_MERGED_PROMPT = gpt41mini_apply["merged_prompt"]

show_file(gpt41mini_apply["merged_diff"], max_chars=30000)
show_file(gpt41mini_apply["merged_prompt"], max_chars=60000)

out_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/advisor_prompt_patches_gpt41mini_round01.json
validation_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/advisor_prompt_patches_gpt41mini_round01_validation.json
{
  "ok": false,
  "errors": [
    "No injectable semantic/schema prompt patch was generated."
  ],
  "warnings": [],
  "patch_count": 0
}


RuntimeError: GPT advisor output validation failed. Do not apply.

In [117]:
# ============================================================
# Cell 29. GPT-4.1-mini advisor output save / validate / apply
# ============================================================

RUN_GPT41MINI_API = os.environ.get("RUN_GPT41MINI_API", "false").lower() == "true"
GPT41MINI_MODEL = os.environ.get("GPT41MINI_MODEL", "gpt-4.1-mini")

def extract_json_object(text):
    text = str(text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end+1]
    return json.loads(text)

def save_and_validate_advisor_output(obj, variant):
    variant = _safe_name(variant)
    out_path = GPT_ADVISOR_ROOT / f"advisor_prompt_patches_{variant}.json"
    _write_json(out_path, obj)

    validation = validate_patch_payload_strict(obj)
    validation_path = GPT_ADVISOR_ROOT / f"advisor_prompt_patches_{variant}_validation.json"
    _write_json(validation_path, validation)

    print("out_path:", out_path)
    print("validation_path:", validation_path)
    print(json.dumps(validation, ensure_ascii=False, indent=2))

    patch_df = pd.DataFrame(obj.get("prompt_patches", []))
    if not patch_df.empty:
        display(patch_df[[
            c for c in [
                "patch_id", "target_block_family", "target_block_id",
                "priority", "evidence_diagnostics", "patch_text"
            ]
            if c in patch_df.columns
        ]])
        display(
            patch_df[["target_block_family", "target_block_id"]]
            .drop_duplicates()
            .sort_values(["target_block_family", "target_block_id"])
        )

    if not validation.get("ok", False):
        raise RuntimeError("GPT advisor output validation failed. Do not apply.")

    return out_path, validation_path

gpt41mini_output_obj = None

if RUN_GPT41MINI_API:
    try:
        from openai import OpenAI
    except Exception as e:
        raise RuntimeError("openai package is not available. Install it or paste GPT output manually.") from e

    client = OpenAI()
    resp = client.chat.completions.create(
        model=GPT41MINI_MODEL,
        messages=[
            {"role": "system", "content": gpt41mini_round01["system_prompt"]},
            {"role": "user", "content": gpt41mini_round01["user_prompt"]},
        ],
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    gpt41mini_output_text = resp.choices[0].message.content
    gpt41mini_output_obj = extract_json_object(gpt41mini_output_text)
else:
    print("GPT-4.1-mini API call skipped.")
    print("Paste GPT output JSON into GPT41MINI_PATCH_JSON_TEXT and rerun the lower block.")
    GPT41MINI_PATCH_JSON_TEXT = r'''
{
  "advisor_meta": {
    "schema_version": "row_advisor_prompt_patches_v1",
    "source": "gpt-4.1-mini_round01",
    "official_metric": "strict_det",
    "cloud_is_auxiliary": true
  },
  "prompt_patches": []
}
'''.strip()

    # Leave empty placeholder untouched. Replace it with real GPT output before parsing.
    placeholder = extract_json_object(GPT41MINI_PATCH_JSON_TEXT)
    if placeholder.get("prompt_patches"):
        gpt41mini_output_obj = placeholder
    else:
        print("[INFO] Placeholder has no patches. Replace GPT41MINI_PATCH_JSON_TEXT with GPT response first.")

GPT41MINI_PATCH_PATH = None
if gpt41mini_output_obj is not None:
    GPT41MINI_PATCH_PATH, _ = save_and_validate_advisor_output(
        gpt41mini_output_obj,
        variant="gpt41mini_round01",
    )

    gpt41mini_apply = apply_patch_payload_with_generation(
        GPT41MINI_PATCH_PATH,
        variant="gpt41mini_round01",
    )

    GPT41MINI_PATCHED_GENOME = gpt41mini_apply["merged_genome"]
    GPT41MINI_MERGED_PROMPT = gpt41mini_apply["merged_prompt"]

    show_file(gpt41mini_apply["merged_diff"], max_chars=30000)
    show_file(gpt41mini_apply["merged_prompt"], max_chars=60000)
else:
    print("No GPT-4.1-mini patch applied yet.")

RuntimeError: openai package is not available. Install it or paste GPT output manually.

In [ ]:
# ============================================================
# Cell 30. Build local self-advisor input package
# ============================================================

LOCAL_SELF_ADVISOR_ROOT = _notebook_root() / "local_self_advisor"
LOCAL_SELF_ADVISOR_ROOT.mkdir(parents=True, exist_ok=True)

def build_local_self_advisor_package(round_name="round01", max_rows=30):
    round_name = _safe_name(round_name)

    current_patch_path = Path(ROW_ADVISOR_OUT_DIR) / "advisor_prompt_patches.json"

    package = {
        "task": "Local self-advisor: propose prompt patches that this same local Qwen generator can follow better.",
        "official_metric": "strict_det",
        "cloud_is_auxiliary": False,
        "advisor_type": "local_self_advisor",
        "target_generator": "same local Qwen JOILang code generator",
        "expected_block_by_family": EXPECTED_BLOCK_BY_FAMILY,
        "forbidden_families": sorted(NON_INJECTABLE_FAMILIES),
        "current_det_rulebased_patch": _read_json(current_patch_path),
        "row_samples": compact_row_advisor_sample(max_rows=max_rows),
        "instruction": (
            "Find prompt rules that are especially easy for this local model to obey. "
            "Focus on row independence, exact schema copy, no camelCase service invention, receiver-first selection, "
            "cron/period policy, numeric preservation, and enum descriptor copying."
        ),
    }

    system_prompt = """
You are the same local model acting as a strict JOILang prompt self-advisor.
You must output only one JSON object.

You are not generating JOILang code now.
You are generating prompt patches that will help a local JOILang generator avoid repeated failures.

Rules:
- JSON only.
- No markdown fences.
- Do not include Generation_Health, Runtime_Health, Prompt_Budget, No_Mutation, or Block 00.
- Use target_block_id exactly from expected_block_by_family.
- Prefer short rules that this local model can follow.
- Do not propose broad vague advice.
"""

    user_prompt = f"""
Using the following strict DET feedback package, generate advisor_prompt_patches JSON.

{json.dumps(package, ensure_ascii=False, indent=2)}

Return exactly:
{{
  "advisor_meta": {{
    "schema_version": "row_advisor_prompt_patches_v1",
    "source": "local_self_advisor_{round_name}",
    "official_metric": "strict_det",
    "cloud_is_auxiliary": false
  }},
  "prompt_patches": [...]
}}
"""

    input_json = LOCAL_SELF_ADVISOR_ROOT / f"local_self_{round_name}_advisor_input.json"
    prompt_md = LOCAL_SELF_ADVISOR_ROOT / f"local_self_{round_name}_advisor_prompt.md"

    _write_json(input_json, package)
    prompt_md.write_text(
        "# System\n\n" + system_prompt.strip() + "\n\n# User\n\n" + user_prompt.strip(),
        encoding="utf-8",
    )

    print("input_json:", input_json)
    print("prompt_md:", prompt_md)
    show_file(prompt_md, max_chars=30000)

    return {
        "round_name": round_name,
        "input_json": input_json,
        "prompt_md": prompt_md,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }

local_self_round01 = build_local_self_advisor_package(round_name="round01", max_rows=30)

In [ ]:
# ============================================================
# Cell 30. Build local self-advisor input package
# ============================================================

LOCAL_SELF_ADVISOR_ROOT = _notebook_root() / "local_self_advisor"
LOCAL_SELF_ADVISOR_ROOT.mkdir(parents=True, exist_ok=True)

def build_local_self_advisor_package(round_name="round01", max_rows=30):
    round_name = _safe_name(round_name)

    current_patch_path = Path(ROW_ADVISOR_OUT_DIR) / "advisor_prompt_patches.json"

    package = {
        "task": "Local self-advisor: propose prompt patches that this same local Qwen generator can follow better.",
        "official_metric": "strict_det",
        "cloud_is_auxiliary": False,
        "advisor_type": "local_self_advisor",
        "target_generator": "same local Qwen JOILang code generator",
        "expected_block_by_family": EXPECTED_BLOCK_BY_FAMILY,
        "forbidden_families": sorted(NON_INJECTABLE_FAMILIES),
        "current_det_rulebased_patch": _read_json(current_patch_path),
        "row_samples": compact_row_advisor_sample(max_rows=max_rows),
        "instruction": (
            "Find prompt rules that are especially easy for this local model to obey. "
            "Focus on row independence, exact schema copy, no camelCase service invention, receiver-first selection, "
            "cron/period policy, numeric preservation, and enum descriptor copying."
        ),
    }

    system_prompt = """
You are the same local model acting as a strict JOILang prompt self-advisor.
You must output only one JSON object.

You are not generating JOILang code now.
You are generating prompt patches that will help a local JOILang generator avoid repeated failures.

Rules:
- JSON only.
- No markdown fences.
- Do not include Generation_Health, Runtime_Health, Prompt_Budget, No_Mutation, or Block 00.
- Use target_block_id exactly from expected_block_by_family.
- Prefer short rules that this local model can follow.
- Do not propose broad vague advice.
"""

    user_prompt = f"""
Using the following strict DET feedback package, generate advisor_prompt_patches JSON.

{json.dumps(package, ensure_ascii=False, indent=2)}

Return exactly:
{{
  "advisor_meta": {{
    "schema_version": "row_advisor_prompt_patches_v1",
    "source": "local_self_advisor_{round_name}",
    "official_metric": "strict_det",
    "cloud_is_auxiliary": false
  }},
  "prompt_patches": [...]
}}
"""

    input_json = LOCAL_SELF_ADVISOR_ROOT / f"local_self_{round_name}_advisor_input.json"
    prompt_md = LOCAL_SELF_ADVISOR_ROOT / f"local_self_{round_name}_advisor_prompt.md"

    _write_json(input_json, package)
    prompt_md.write_text(
        "# System\n\n" + system_prompt.strip() + "\n\n# User\n\n" + user_prompt.strip(),
        encoding="utf-8",
    )

    print("input_json:", input_json)
    print("prompt_md:", prompt_md)
    show_file(prompt_md, max_chars=30000)

    return {
        "round_name": round_name,
        "input_json": input_json,
        "prompt_md": prompt_md,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }

local_self_round01 = build_local_self_advisor_package(round_name="round01", max_rows=30)

In [ ]:
# ============================================================
# Cell 32. Advisor variant comparison index
# ============================================================

def collect_merged_prompt_index():
    rows = []
    for manifest_path in sorted(MERGED_PROMPT_ROOT.glob("merged_joilang_code_prompt_*_gen*_manifest.json")):
        try:
            m = _read_json(manifest_path)
        except Exception:
            continue
        rows.append({
            "variant": m.get("variant"),
            "generation": m.get("generation"),
            "merged_prompt": m.get("merged_prompt"),
            "merged_genome": m.get("merged_genome"),
            "merged_diff": m.get("merged_diff"),
            "patch_path": m.get("patch_path"),
            "apply_dir": m.get("apply_dir"),
            "manifest": str(manifest_path),
        })
    return pd.DataFrame(rows)

merged_prompt_index_df = collect_merged_prompt_index()
display(merged_prompt_index_df)

index_path = MERGED_PROMPT_ROOT / f"merged_prompt_index_{_now_tag()}.csv"
merged_prompt_index_df.to_csv(index_path, index=False, encoding="utf-8-sig")
print("index_path:", index_path)

print("\nNext:")
print("1. Inspect merged prompt files.")
print("2. Rerun selected failed rows with each patched_genome.")
print("3. Compare det_score_delta, det_pass changes, and remaining failure_reasons.")
print("4. Use GPT-4.1-mini round02 or local_self round02 if needed.")

In [115]:
GPT41MINI_RERUN_ALL = True

In [119]:
# ============================================================
# Cell 33. One-shot GPT-4.1-mini advisor → patch apply → local rerun
# No openai package required. Uses urllib standard library.
# ============================================================

import os
import re
import json
import time
import shutil
import urllib.request
import urllib.error
from pathlib import Path
from datetime import datetime

import pandas as pd

# ------------------------------------------------------------
# User controls
# ------------------------------------------------------------
GPT41MINI_MODEL = os.environ.get("GPT41MINI_MODEL", "gpt-4.1-mini")
GPT41MINI_VARIANT = os.environ.get("GPT41MINI_VARIANT", "gpt41mini_round01")

# 밥 먹고 올 동안 더 많이 돌리고 싶으면 True.
GPT41MINI_RERUN_ALL = os.environ.get("GPT41MINI_RERUN_ALL", "false").lower() == "true"

# 기본은 20 rows. 전체 280개는 GPT41MINI_RERUN_ALL=True.
GPT41MINI_RERUN_LIMIT = int(os.environ.get("GPT41MINI_RERUN_LIMIT", "20"))

# 직접 지정하려면 예: os.environ["GPT41MINI_RERUN_ROWS"]="1,2,3,121,170,251"
GPT41MINI_RERUN_ROWS_TEXT = os.environ.get("GPT41MINI_RERUN_ROWS", "").strip()

GPT41MINI_TEMPERATURE = float(os.environ.get("GPT41MINI_TEMPERATURE", "0.1"))
GPT41MINI_MAX_REPAIR_ATTEMPTS = int(os.environ.get("GPT41MINI_MAX_REPAIR_ATTEMPTS", "2"))
GPT41MINI_HTTP_TIMEOUT_SEC = int(os.environ.get("GPT41MINI_HTTP_TIMEOUT_SEC", "300"))

LOCAL_RERUN_TIMEOUT_SEC = int(os.environ.get("LOCAL_RERUN_TIMEOUT_SEC", "2400"))

# API key는 출력하지 않음.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not set in this Jupyter environment. "
        "Set os.environ['OPENAI_API_KEY']='...' or export it before launching Jupyter."
    )

# ------------------------------------------------------------
# Required variables / helpers check
# ------------------------------------------------------------
required_globals = [
    "ROW_ADVISOR_OUT_DIR",
    "validate_patch_payload_strict",
    "apply_patch_payload_with_generation",
    "run_cmd",
]
missing = [x for x in required_globals if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required notebook helpers/variables: {missing}. Run Cells 15-24 first.")

ROW_ADVISOR_OUT_DIR = Path(ROW_ADVISOR_OUT_DIR)

def _now_tag():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def _safe_name(x):
    x = str(x or "").strip()
    x = re.sub(r"[^A-Za-z0-9_.-]+", "_", x)
    return x.strip("_") or "unnamed"

def _read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8"))

def _write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def _notebook_root():
    if "NB_ROOT" in globals():
        return Path(NB_ROOT)
    return Path(ROW_ADVISOR_OUT_DIR).parent.parent

GPT_ADVISOR_ROOT = _notebook_root() / "gpt41mini_advisor"
GPT_ADVISOR_ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# JSON extraction / OpenAI API via urllib
# ------------------------------------------------------------
def extract_json_object_strict(text):
    text = str(text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", text)
        text = re.sub(r"\s*```$", "", text).strip()

    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]

    return json.loads(text)

def call_openai_chat_json(system_prompt, user_prompt, *, model, temperature=0.1, timeout_sec=300):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "response_format": {"type": "json_object"},
    }

    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urllib.request.urlopen(req, timeout=timeout_sec) as resp:
            raw = resp.read().decode("utf-8")
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"OpenAI HTTPError {e.code}: {body[:4000]}") from e
    except urllib.error.URLError as e:
        raise RuntimeError(f"OpenAI URLError: {e}") from e

    data = json.loads(raw)
    content = data["choices"][0]["message"]["content"]
    return extract_json_object_strict(content), data

# ------------------------------------------------------------
# Build compact GPT advisor input
# ------------------------------------------------------------
def compact_row_advisor_sample(max_rows=60):
    csv_path = ROW_ADVISOR_OUT_DIR / "row_advisor_mapping.csv"
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    df = pd.read_csv(csv_path)

    # severity_score가 있으면 높은 실패부터. 없으면 collapse/DET 실패가 이미 상단일 가능성이 큼.
    if "severity_score" in df.columns:
        df = df.sort_values("severity_score", ascending=False)

    cols = [
        c for c in [
            "row_no", "category", "command_eng", "command_kor",
            "det_score", "det_pass", "failure_reasons", "normalized_signals",
            "primary_advisor_family", "advisor_families", "concrete_diagnostics",
            "gt_code", "generated_code", "gt_json", "generated_json",
            "diff_summary", "generated_collapse_count",
        ]
        if c in df.columns
    ]
    return df[cols].head(max_rows).to_dict("records")

def build_gpt41mini_one_shot_prompts(max_rows=60):
    current_patch_path = ROW_ADVISOR_OUT_DIR / "advisor_prompt_patches.json"
    advisor_rich_feedback_path = ROW_ADVISOR_OUT_DIR / "advisor_rich_feedback.json"
    taxonomy_path = ROW_ADVISOR_OUT_DIR / "failure_taxonomy_table.csv"

    if not current_patch_path.exists():
        raise FileNotFoundError(current_patch_path)

    current_det_patch = _read_json(current_patch_path)
    row_samples = compact_row_advisor_sample(max_rows=max_rows)

    expected_block_by_family = {
        "Service_Mapping": "02",
        "Receiver_Tag_Preservation": "02",
        "Enum_Grounding": "02",
        "Argument_Grounding": "02",
        "Output_Schema": "03",
        "Temporal_Rule": "06",
        "Numeric_Unit_Grounding": "06",
        "Dataflow": "06",
        "Intent_Fulfillment": "06",
        "Skeleton": "06",
        "DET_Helper": "06",
        "Minimality": "06",
    }

    system_prompt = """
You are an expert JOILang prompt advisor.

Your job is to generate localized prompt patches that improve a local JOILang code generator.
Strict DET is the official metric. Cloud reasoning is auxiliary only.

Return exactly one valid JSON object. Do not include markdown fences or prose outside JSON.

Hard constraints:
- prompt_patches must contain at least 6 injectable semantic/schema patches.
- Do not output Generation_Health, Runtime_Health, Prompt_Budget, No_Mutation, Parser_Extraction, or Block 00 patches.
- Use target_block_id exactly from expected_block_by_family.
- Each patch_text must be concise, imperative, and directly actionable.
- Use row-level concrete diagnostics as evidence.
- Prefer targeted constraints over broad generic advice.
- Do not return an empty patch list.
""".strip()

    package = {
        "task": "Generate improved localized JOILang prompt patches from strict DET row diagnostics.",
        "official_metric": "strict_det",
        "cloud_is_auxiliary": True,
        "target_generator": "local Qwen JOILang code generator",
        "advisor_model": GPT41MINI_MODEL,
        "input_files": {
            "advisor_rich_feedback": str(advisor_rich_feedback_path),
            "current_det_rulebased_patch": str(current_patch_path),
            "failure_taxonomy_table": str(taxonomy_path),
        },
        "expected_block_by_family": expected_block_by_family,
        "forbidden_families": [
            "Generation_Health",
            "Runtime_Health",
            "Prompt_Budget",
            "No_Mutation",
            "Parser_Extraction",
        ],
        "dominant_failure_pattern": (
            "The local Qwen generator reused very similar DishwasherDryMode-style output across unrelated rows. "
            "It also produced camelCase/class-style service names, receiver mismatches, cron/period mismatches, "
            "dropped numeric literals, and enum/string argument mismatches."
        ),
        "current_det_rulebased_patch": current_det_patch,
        "row_samples": row_samples,
        "required_output_shape": {
            "advisor_meta": {
                "schema_version": "row_advisor_prompt_patches_v1",
                "source": GPT41MINI_VARIANT,
                "official_metric": "strict_det",
                "cloud_is_auxiliary": True,
            },
            "prompt_patches": [
                {
                    "patch_id": f"{GPT41MINI_VARIANT}_service_mapping_02",
                    "target_block_family": "Service_Mapping",
                    "target_block_id": "02",
                    "operation": "append_micro_rule",
                    "priority": 100,
                    "patch_text": "localized imperative prompt rule",
                    "rationale": "why this patch is needed based on concrete diagnostics",
                    "evidence_rows": ["row ids"],
                    "evidence_diagnostics": ["diagnostic strings"],
                    "mutation_intent": "gpt41mini_strict_det_prompt_repair",
                }
            ],
        },
    }

    user_prompt = f"""
Generate a non-empty advisor_prompt_patches JSON object for JOILang prompt repair.

Required patch families:
- Service_Mapping -> target_block_id "02"
- Receiver_Tag_Preservation -> target_block_id "02"
- Enum_Grounding -> target_block_id "02"
- Temporal_Rule -> target_block_id "06"
- Numeric_Unit_Grounding -> target_block_id "06"
- Skeleton -> target_block_id "06"
- DET_Helper -> target_block_id "06"

Input package:
{json.dumps(package, ensure_ascii=False, indent=2)}

Return exactly one valid JSON object with advisor_meta and prompt_patches.
No markdown fences. No commentary.
""".strip()

    prompt_path = GPT_ADVISOR_ROOT / f"{GPT41MINI_VARIANT}_one_shot_prompt_{_now_tag()}.md"
    prompt_path.write_text(
        "# System\n\n" + system_prompt + "\n\n# User\n\n" + user_prompt,
        encoding="utf-8",
    )
    print("GPT advisor prompt saved:", prompt_path)

    return system_prompt, user_prompt, prompt_path

# ------------------------------------------------------------
# Save, validate, and repair GPT advisor output if needed
# ------------------------------------------------------------
def save_gpt_patch_payload(obj, variant, suffix=""):
    variant = _safe_name(variant)
    suffix = f"_{suffix}" if suffix else ""
    out_path = GPT_ADVISOR_ROOT / f"advisor_prompt_patches_{variant}{suffix}.json"
    _write_json(out_path, obj)
    return out_path

def validate_or_raise_gpt_payload(obj, variant):
    validation = validate_patch_payload_strict(obj)
    validation_path = GPT_ADVISOR_ROOT / f"advisor_prompt_patches_{_safe_name(variant)}_validation.json"
    _write_json(validation_path, validation)

    print("validation_path:", validation_path)
    print(json.dumps(validation, ensure_ascii=False, indent=2))

    patch_df = pd.DataFrame(obj.get("prompt_patches", []))
    if not patch_df.empty:
        display(patch_df[[
            c for c in [
                "patch_id", "target_block_family", "target_block_id",
                "priority", "evidence_diagnostics", "patch_text"
            ]
            if c in patch_df.columns
        ]])
        display(
            patch_df[["target_block_family", "target_block_id"]]
            .drop_duplicates()
            .sort_values(["target_block_family", "target_block_id"])
        )

    if not validation.get("ok", False):
        raise RuntimeError("GPT advisor output validation failed. Do not apply.")

    return validation, validation_path

def repair_gpt_patch_with_errors(previous_obj, validation, system_prompt, original_user_prompt, attempt):
    repair_prompt = f"""
The previous JSON output failed validation.

Validation errors:
{json.dumps(validation.get("errors", []), ensure_ascii=False, indent=2)}

Validation warnings:
{json.dumps(validation.get("warnings", []), ensure_ascii=False, indent=2)}

Previous invalid output:
{json.dumps(previous_obj, ensure_ascii=False, indent=2)}

Repair requirements:
- Return exactly one valid JSON object.
- prompt_patches must be non-empty.
- Include at least these 7 patch families:
  Service_Mapping, Receiver_Tag_Preservation, Enum_Grounding, Temporal_Rule, Numeric_Unit_Grounding, Skeleton, DET_Helper.
- Use exact block mapping:
  Service_Mapping=02, Receiver_Tag_Preservation=02, Enum_Grounding=02,
  Temporal_Rule=06, Numeric_Unit_Grounding=06, Skeleton=06, DET_Helper=06.
- Do not include Block 00 or runtime/generation health patches.
- No markdown fences. No prose outside JSON.

Original task:
{original_user_prompt}
""".strip()

    print(f"\n[GPT repair attempt {attempt}]")
    return call_openai_chat_json(
        system_prompt,
        repair_prompt,
        model=GPT41MINI_MODEL,
        temperature=0.0,
        timeout_sec=GPT41MINI_HTTP_TIMEOUT_SEC,
    )

def get_valid_gpt41mini_patch_payload():
    system_prompt, user_prompt, prompt_path = build_gpt41mini_one_shot_prompts(max_rows=60)

    print(f"\n[GPT advisor call] model={GPT41MINI_MODEL}")
    obj, raw = call_openai_chat_json(
        system_prompt,
        user_prompt,
        model=GPT41MINI_MODEL,
        temperature=GPT41MINI_TEMPERATURE,
        timeout_sec=GPT41MINI_HTTP_TIMEOUT_SEC,
    )

    raw_path = GPT_ADVISOR_ROOT / f"{GPT41MINI_VARIANT}_raw_response_attempt0_{_now_tag()}.json"
    _write_json(raw_path, raw)
    save_gpt_patch_payload(obj, GPT41MINI_VARIANT, suffix="attempt0")

    for attempt in range(1, GPT41MINI_MAX_REPAIR_ATTEMPTS + 2):
        validation = validate_patch_payload_strict(obj)
        print(f"\n[validation attempt {attempt - 1}]")
        print(json.dumps(validation, ensure_ascii=False, indent=2))

        if validation.get("ok", False):
            final_path = save_gpt_patch_payload(obj, GPT41MINI_VARIANT)
            final_validation_path = GPT_ADVISOR_ROOT / f"advisor_prompt_patches_{_safe_name(GPT41MINI_VARIANT)}_validation.json"
            _write_json(final_validation_path, validation)
            print("GPT patch final_path:", final_path)
            print("GPT validation path:", final_validation_path)
            return obj, final_path, final_validation_path

        if attempt > GPT41MINI_MAX_REPAIR_ATTEMPTS:
            invalid_path = save_gpt_patch_payload(obj, GPT41MINI_VARIANT, suffix="invalid_final")
            raise RuntimeError(
                f"GPT advisor output still invalid after repairs. Saved invalid output: {invalid_path}"
            )

        obj, raw_repair = repair_gpt_patch_with_errors(
            obj,
            validation,
            system_prompt,
            user_prompt,
            attempt,
        )
        raw_repair_path = GPT_ADVISOR_ROOT / f"{GPT41MINI_VARIANT}_raw_response_repair{attempt}_{_now_tag()}.json"
        _write_json(raw_repair_path, raw_repair)
        save_gpt_patch_payload(obj, GPT41MINI_VARIANT, suffix=f"repair{attempt}")

# ------------------------------------------------------------
# Rerun helpers
# ------------------------------------------------------------
def parse_row_list(value):
    out = []
    for part in str(value or "").replace(";", ",").split(","):
        part = part.strip()
        if part:
            out.append(int(part))
    return out

def select_rerun_rows():
    if GPT41MINI_RERUN_ROWS_TEXT:
        rows = parse_row_list(GPT41MINI_RERUN_ROWS_TEXT)
        return rows

    csv_path = ROW_ADVISOR_OUT_DIR / "row_advisor_mapping.csv"
    df = pd.read_csv(csv_path)

    if "severity_score" in df.columns:
        df = df.sort_values("severity_score", ascending=False)

    rows = df["row_no"].astype(int).drop_duplicates().tolist()
    if GPT41MINI_RERUN_ALL:
        return rows
    return rows[:GPT41MINI_RERUN_LIMIT]

def make_worker_extra_for_row(row_no, model_key):
    max_new_tokens = globals().get("MAX_NEW_TOKENS", globals().get("MAX_NEW_TOKENS_7B", 1024))

    if "worker_extra_json_path" in globals():
        return worker_extra_json_path(
            label=f"worker_extra_gpt41mini_row{int(row_no):03d}_{model_key}_{_now_tag()}",
            model_key=model_key,
            max_new_tokens=max_new_tokens,
        )

    print("[WARN] worker_extra_json_path() not found. Running without llm-extra-json.")
    return None

def eval_one_row_with_patched_genome(row_no, genome_json, variant, model_key=None, timeout_sec=2400):
    model_key = model_key or globals().get("MODEL_KEY", globals().get("MODEL_KEY_7B", "qwen25_coder_7b"))

    out_root = _notebook_root() / "patched_eval_runs" / _safe_name(variant)
    out_dir = out_root / f"row{int(row_no):03d}_{_now_tag()}"
    out_dir.mkdir(parents=True, exist_ok=True)

    py = str(globals().get("PY", globals().get("JOI_PY", "python")))
    model = str(globals().get("MODEL", "gpt_mg.version0_13"))
    dataset = str(globals().get("DATASET"))
    service_schema = str(globals().get("SERVICE_SCHEMA"))

    cmd = [
        py, "-m", "utils.ga_search.cli", "eval",
        "--model", model,
        "--dataset", dataset,
        "--service-schema", service_schema,
        "--llm-mode", "worker",
        "--engine-mode", "real",
        "--model-key", model_key,
        "--det-profile", "strict",
        "--det-threshold", "70",
        "--out-dir", str(out_dir),
        "--print-mode", "summary",
        "--row-no", str(int(row_no)),
        "--genome-json", str(genome_json),
    ]

    extra_path = make_worker_extra_for_row(row_no, model_key)
    if extra_path is not None:
        extra_text = Path(extra_path).read_text(encoding="utf-8")
        cmd += ["--llm-extra-json", extra_text]

    rc, output = run_cmd(
        cmd,
        log_path=out_dir / f"eval_row{int(row_no):03d}_{_safe_name(variant)}.log",
        check=False,
        timeout_sec=timeout_sec,
    )

    return {
        "row_no": int(row_no),
        "variant": variant,
        "out_dir": str(out_dir),
        "rc": rc,
    }

def rerun_rows_with_gpt_patch(genome_json, rows, variant):
    results = []
    model_key = globals().get("MODEL_KEY", globals().get("MODEL_KEY_7B", "qwen25_coder_7b"))

    for idx, row_no in enumerate(rows, start=1):
        print("\n" + "=" * 100)
        print(f"[{idx}/{len(rows)}] local rerun row={row_no}, model_key={model_key}, variant={variant}")
        try:
            result = eval_one_row_with_patched_genome(
                row_no=row_no,
                genome_json=genome_json,
                variant=variant,
                model_key=model_key,
                timeout_sec=LOCAL_RERUN_TIMEOUT_SEC,
            )
        except Exception as e:
            result = {
                "row_no": int(row_no),
                "variant": variant,
                "out_dir": "",
                "rc": -999,
                "error": repr(e),
            }
            print("[ERROR]", repr(e))

        results.append(result)

    rerun_df = pd.DataFrame(results)
    index_path = _notebook_root() / "patched_eval_runs" / f"rerun_index_{_safe_name(variant)}_{_now_tag()}.csv"
    index_path.parent.mkdir(parents=True, exist_ok=True)
    rerun_df.to_csv(index_path, index=False, encoding="utf-8-sig")
    print("\nrerun index:", index_path)
    display(rerun_df)
    return rerun_df, index_path

def find_row_eval_csv(run_dir):
    run_dir = Path(run_dir)
    hits = list(run_dir.rglob("row_evaluation.csv"))
    return hits[0] if hits else None

def load_rerun_eval_rows(rerun_df):
    dfs = []
    for _, r in rerun_df.iterrows():
        if not str(r.get("out_dir", "")).strip():
            continue
        p = find_row_eval_csv(r["out_dir"])
        if p and Path(p).exists():
            df = pd.read_csv(p)
            df["rerun_out_dir"] = r["out_dir"]
            df["variant"] = r["variant"]
            dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

def baseline_eval_df():
    if "ROW_EVALUATION_CSV" in globals() and Path(ROW_EVALUATION_CSV).exists():
        return pd.read_csv(ROW_EVALUATION_CSV)

    p = Path(ROW_ADVISOR_OUT_DIR).parent / "eval" / "row_evaluation.csv"
    if p.exists():
        return pd.read_csv(p)

    raise FileNotFoundError("Cannot locate baseline row_evaluation.csv")

def compare_gpt_patch_results(rerun_df, variant):
    base = baseline_eval_df()
    after = load_rerun_eval_rows(rerun_df)

    if after.empty:
        print("No rerun eval rows loaded. Comparison skipped.")
        return pd.DataFrame()

    keep_base = [
        c for c in [
            "row_no", "category", "det_score", "det_pass", "gt_similarity",
            "failure_reasons", "generated_code", "generated_json"
        ]
        if c in base.columns
    ]
    keep_after = [
        c for c in [
            "row_no", "category", "det_score", "det_pass", "gt_similarity",
            "failure_reasons", "generated_code", "generated_json", "variant", "rerun_out_dir"
        ]
        if c in after.columns
    ]

    merged = base[keep_base].merge(
        after[keep_after],
        on="row_no",
        how="inner",
        suffixes=("_before", "_after"),
    )

    if "det_score_before" in merged.columns and "det_score_after" in merged.columns:
        merged["det_score_delta"] = merged["det_score_after"] - merged["det_score_before"]

    if "det_pass_before" in merged.columns and "det_pass_after" in merged.columns:
        merged["pass_changed"] = (
            merged["det_pass_before"].astype(str)
            + " -> "
            + merged["det_pass_after"].astype(str)
        )

    out_path = _notebook_root() / "patched_eval_runs" / f"compare_{_safe_name(variant)}_{_now_tag()}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")

    print("comparison_csv:", out_path)
    display(merged[[
        c for c in [
            "row_no", "category_before",
            "det_score_before", "det_score_after", "det_score_delta",
            "det_pass_before", "det_pass_after", "pass_changed",
            "failure_reasons_before", "failure_reasons_after",
            "generated_code_before", "generated_code_after",
            "rerun_out_dir",
        ]
        if c in merged.columns
    ]].sort_values("det_score_delta", ascending=False if "det_score_delta" in merged.columns else True))

    return merged

# ------------------------------------------------------------
# One-shot execution
# ------------------------------------------------------------
print("=" * 100)
print("STEP 1. GPT-4.1-mini advisor patch generation")
print("=" * 100)

gpt_patch_obj, GPT41MINI_PATCH_PATH, GPT41MINI_VALIDATION_PATH = get_valid_gpt41mini_patch_payload()

print("\n" + "=" * 100)
print("STEP 2. Apply GPT patch and create merged JOILang prompt")
print("=" * 100)

gpt41mini_apply = apply_patch_payload_with_generation(
    GPT41MINI_PATCH_PATH,
    variant=GPT41MINI_VARIANT,
)

GPT41MINI_PATCHED_GENOME = gpt41mini_apply["merged_genome"]
GPT41MINI_MERGED_PROMPT = gpt41mini_apply["merged_prompt"]

print("GPT41MINI_PATCH_PATH:", GPT41MINI_PATCH_PATH)
print("GPT41MINI_PATCHED_GENOME:", GPT41MINI_PATCHED_GENOME)
print("GPT41MINI_MERGED_PROMPT:", GPT41MINI_MERGED_PROMPT)

print("\n=== GPT patch diff ===")
show_file(gpt41mini_apply["merged_diff"], max_chars=30000)

print("\n=== GPT patched merged prompt preview ===")
show_file(gpt41mini_apply["merged_prompt"], max_chars=60000)

print("\n" + "=" * 100)
print("STEP 3. Local rerun with GPT-patched prompt")
print("=" * 100)

GPT41MINI_RERUN_ROWS = select_rerun_rows()
print("GPT41MINI_RERUN_ROWS:", GPT41MINI_RERUN_ROWS)
print("row_count:", len(GPT41MINI_RERUN_ROWS))

gpt41mini_rerun_df, GPT41MINI_RERUN_INDEX = rerun_rows_with_gpt_patch(
    genome_json=GPT41MINI_PATCHED_GENOME,
    rows=GPT41MINI_RERUN_ROWS,
    variant=GPT41MINI_VARIANT,
)

print("\n" + "=" * 100)
print("STEP 4. Before/after comparison")
print("=" * 100)

gpt41mini_compare_df = compare_gpt_patch_results(
    gpt41mini_rerun_df,
    variant=GPT41MINI_VARIANT,
)

print("\nDONE")
print("GPT41MINI_PATCH_PATH:", GPT41MINI_PATCH_PATH)
print("GPT41MINI_PATCHED_GENOME:", GPT41MINI_PATCHED_GENOME)
print("GPT41MINI_MERGED_PROMPT:", GPT41MINI_MERGED_PROMPT)
print("GPT41MINI_RERUN_INDEX:", GPT41MINI_RERUN_INDEX)

STEP 1. GPT-4.1-mini advisor patch generation
GPT advisor prompt saved: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/gpt41mini_round01_one_shot_prompt_20260625_120850.md

[GPT advisor call] model=gpt-4.1-mini

[validation attempt 0]
{
  "ok": true,
  "errors": [],
  "warnings": [],
  "patch_count": 7
}
GPT patch final_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/advisor_prompt_patches_gpt41mini_round01.json
GPT validation path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/advisor_prompt_patches_gpt41mini_round01_validation.json

STEP 2. Apply GPT patch and create merged JOILang prompt
patch_path: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/a

,row_no,variant,out_dir,rc
0,218,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row218_20260625_120929,2
1,222,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row222_20260625_120939,2
2,195,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row195_20260625_120949,2
3,186,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row186_20260625_121000,2
4,187,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row187_20260625_121010,2
5,251,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row251_20260625_121020,2
6,211,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row211_20260625_121031,2
7,219,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row219_20260625_121041,2
8,230,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row230_20260625_121052,2
9,216,gpt41mini_round01,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row216_20260625_121102,2



STEP 4. Before/after comparison
comparison_csv: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/compare_gpt41mini_round01_20260625_121256.csv


,row_no,category_before,det_score_before,det_score_after,det_score_delta,det_pass_before,det_pass_after,pass_changed,failure_reasons_before,failure_reasons_after,generated_code_before,generated_code_after,rerun_out_dir
6,211,7,13.8554,17.5,3.6446,False,False,False -> False,"[""cron_mismatch"",""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","[""missing_generated_code"",""cron_mismatch"",""gt_mismatch"",""gt_service_coverage"",""gt_receiver_coverage"",""numeric_grounding""]","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")",NaN,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row211_20260625_121031
2,174,6,16.4689,20.0,3.5311,False,False,False -> False,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","[""missing_generated_code"",""gt_mismatch"",""gt_service_coverage"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")",NaN,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row174_20260625_121143
1,150,5,16.7282,20.0,3.2718,False,False,False -> False,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","[""missing_generated_code"",""gt_mismatch"",""gt_service_coverage"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")",NaN,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row150_20260625_121225
0,114,4,16.8114,20.0,3.1886,False,False,False -> False,"[""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","[""missing_generated_code"",""gt_mismatch"",""gt_service_coverage"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")",NaN,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row114_20260625_121235
10,218,7,10.0350,12.5,2.4650,False,False,False -> False,"[""cron_mismatch"",""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","[""missing_generated_code"",""cron_mismatch"",""gt_mismatch"",""gt_service_coverage"",""gt_receiver_coverage"",""numeric_grounding"",""enum_grounding""]","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")",NaN,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row218_20260625_120929
8,216,7,15.6962,17.5,1.8038,False,False,False -> False,"[""cron_mismatch"",""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","[""missing_generated_code"",""cron_mismatch"",""gt_mismatch"",""gt_service_coverage"",""gt_receiver_coverage"",""numeric_grounding""]","(#Dishwasher).dishwasherMode_setDishwasherMode(""dry"")",NaN,/home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/gpt41mini_round01/row216_20260625_121102
9,217,7,16.1146,17.5,1.3854,False,False,False -> False,"[""cron_mismatch"",""period_mismatch"",""gt_mismatch"",""gt_service_coverage"",""unknown_service"",""gt_receiver_coverage"",""numeric_grounding""]","[""missing_generated_code"",""cron_mismatch"",""gt_mismatch"",""gt_service_coverage"","


DONE
GPT41MINI_PATCH_PATH: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/gpt41mini_advisor/advisor_prompt_patches_gpt41mini_round01.json
GPT41MINI_PATCHED_GENOME: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/merged_joilang_code_prompts/merged_joilang_code_prompt_gpt41mini_round01_gen01_patched_genome.json
GPT41MINI_MERGED_PROMPT: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/merged_joilang_code_prompts/merged_joilang_code_prompt_gpt41mini_round01_gen01.md
GPT41MINI_RERUN_INDEX: /home/mgjeong/Desktop/llm/JOILang-Server/artifacts/ga_search_tutorial_runs/cloudless_model_suite_20260624_021806/patched_eval_runs/rerun_index_gpt41mini_round01_20260625_121255.csv
